# Capstone Viggy — Sleep-EDF

**To resume in Colab:** leave the training and diagnostic switches in **1.1** set to **False**, then choose **Runtime → Run all**. Wait for **SESSION READY**. Setup restores F2, folds A–E, epoch times and the 33 TC1 features from the existing project data.

**To review results only:** open section **2**. The score tables and recorded confusion matrices are visible without running any cell or connecting to Drive. Section **5** contains the historical recipes as text.

| Section | Contents |
|---|---|
| **1 — Daily setup** | Options, Drive access, package versions, features and folds |
| **2 — Results** | Eight- and 68-participant scores, fixed splits and confusion matrices |
| **3 — Optional training** | TC1, TC1 + Wake crop, TC1 + LP40 and SEL40 |
| **4 — Optional diagnostics** | Signal checks, slow drift and selective filtering |
| **5 — Experiment archive** | Historical decisions and original code for reference |

**Development reference:** TC1 with Random Forest, mean kappa **0.751 ± 0.020**. F2 is the reference for changes to the 16-feature representation. Adding the Wake crop to TC1 improves macro-F1 and N1 recall with similar mean kappa. Adding LP40 did not show a clear benefit. The team has not yet selected a final model; the ten reserved participants remain untouched.

Data, caches and results are kept in the existing project folder under MyDrive. Daily setup reads the saved data and fills any missing epoch-time cache entries. It does not download the EDF files.


**Team hand-off: reproduce TC1 and TC1 + Wake crop**

- **Compare on the same split.** Section **2.3** lists the test participants in folds A–E. Section **3.1** is the TC1 recipe and **3.2** is TC1 + C1, where C1 is the training-only Wake crop.
- **Connect to the original run files.** Set **DATA** in 1.1 to the accessible CM2013_data folder. Setup needs the development EDFs and hypnograms in **train**, the 133 original B0 feature caches in **cache/viggy**, and the five **F2_68_fixed_AE_fold_A.json** through **F2_68_fixed_AE_fold_E.json** reports in **experiments**. This notebook does not contain the data or those caches.
- **Run the references in order.** With setup working, enable **RUN_TC1_68** for TC1. TC1 + C1 also needs the five **C1_68_crop30_fixed_AE_fold_*.json** crop reports and the TC1 fold reports. Enable **RUN_TC1_C1_68** after these are available. Matching saved results are reused.
- **Test an artifact-rejection variant under a new experiment ID.** Keep A–E and the complete test epochs when comparing against either baseline. If a method drops test epochs, report its coverage and a matched comparison separately. Keep the ten final hold-out subjects out of these experiments.


## 1 — Daily setup

### 1.1 — Session options

Keep all training and diagnostic switches set to **False** for normal setup. To resume an experiment, change only its switch to **True** and run the notebook in order. Saved folds are reused if the configuration matches. Switch it back to **False** afterwards.

Use **MOSTRAR_MATRIZES_68** in the settings cell to choose an optional plot from section **2.4**, or set it to **None**. The plot uses saved predictions. The static result tables are always available.


In [ ]:
# Session paths and options.
from pathlib import Path
import errno
import hashlib
import importlib.metadata
import importlib.util
import json
import os
import pickle
import platform
import shutil
import subprocess
import sys
import time

COURSE_REVISION = "5949b723bbd34d172f072446dfcb802ef54b49f7"
COURSE_URL = "https://github.com/farhad-abtahi/CM2013.git"
DATA = "/content/drive/MyDrive/CM2013_data"
TRAIN_DIR = Path(DATA) / "train"
CACHE_VIGGY = str(Path(DATA) / "cache" / "viggy")
EXPERIMENTS_DIR = Path(DATA) / "experiments"
LOCAL_SESSION_DIR = Path("/content/CM2013_viggy_runtime")
COURSE_DIR = Path("/content/_bsp_repo")

# Optional training: False skips preparation and fitting for that variant.
RUN_TC1_68 = False
RUN_TC1_C1_68 = False
RUN_TC1_LP40_68 = False
RUN_SEL40_68 = False

# Optional diagnostics may read recordings; they do not fit models.
RUN_DIAGNOSTICO_SINAL_68 = False
RUN_DIAGNOSTICO_SEL40_68 = False

# Plot saved predictions; None skips the plots.
MOSTRAR_MATRIZES_68 = "F2 vs SEL40"
USE_REAL = True
BASELINE_CONFIG = {}
SESSAO_PRONTA = False
_SETUP_STAGES_68 = set()
_IO_RETRIES = 3
_DRIVE_REMOUNT_USED = False
_DRIVER_RECOVERIES = []

EXPECTED_FOLDS_68 = {
    "A": "SC400 SC405 SC411 SC417 SC422 SC428 SC434 SC441 SC446 SC452 SC458 SC463 SC471 SC477".split(),
    "B": "SC401 SC406 SC412 SC418 SC423 SC429 SC435 SC442 SC447 SC453 SC459 SC465 SC473 SC481".split(),
    "C": "SC402 SC407 SC413 SC419 SC425 SC430 SC436 SC443 SC449 SC454 SC460 SC466 SC474 SC482".split(),
    "D": "SC403 SC409 SC414 SC420 SC426 SC431 SC437 SC444 SC450 SC455 SC461 SC467 SC475".split(),
    "E": "SC404 SC410 SC415 SC421 SC427 SC433 SC438 SC445 SC451 SC457 SC462 SC470 SC476".split(),
}
HOLDOUT_GROUPS = set(
    "SC408 SC416 SC424 SC432 SC440 SC448 SC456 SC464 SC472 SC480".split()
)
LOCAL_SESSION_DIR.mkdir(parents=True, exist_ok=True)

def _require_stage_68(nome):
    if nome not in _SETUP_STAGES_68:
        raise RuntimeError(
            f"Setup incomplete ({nome}). Run section 1 in order."
        )

print("Data folder:", DATA)
ativos = [
    nome for nome, ligado in {
        "TC1": RUN_TC1_68, "TC1 + corte": RUN_TC1_C1_68,
        "TC1 + LP40": RUN_TC1_LP40_68, "SEL40": RUN_SEL40_68,
    }.items() if ligado
]
print("Enabled training runs:", ", ".join(ativos) or "none")


### 1.2 — Connect to Drive and prepare reliable reads

Feature caches, reports and annotations are read through temporary copies on the Colab disk. Transient I/O errors are retried up to three times per operation, with at most one remount of Drive per setup. The source files are preserved.


In [ ]:
from google.colab import drive
import stat as _stat

drive.mount("/content/drive")

def _drive_io(operacao, descricao):
    """Retry transient I/O errors only, with at most one remount."""
    global _DRIVE_REMOUNT_USED
    transitorios = {
        errno.EIO, errno.ECONNABORTED, errno.ECONNRESET,
        errno.ENOTCONN, errno.ETIMEDOUT, getattr(errno, "ESTALE", 116),
    }
    for tentativa in range(_IO_RETRIES):
        try:
            return operacao()
        except OSError as exc:
            if exc.errno not in transitorios or tentativa == _IO_RETRIES - 1:
                raise OSError(
                    exc.errno,
                    f"{descricao}: {exc.strerror or str(exc)}. "
                    "Setup stopped. Existing data and results have been preserved.",
                    getattr(exc, "filename", None),
                ) from exc
            print(
                f"Temporary Drive error: {descricao}. "
                f"Retry {tentativa + 2}/{_IO_RETRIES}.",
                flush=True,
            )
            if exc.errno in {errno.ECONNABORTED, errno.ENOTCONN, errno.ECONNRESET}:
                if not _DRIVE_REMOUNT_USED:
                    _DRIVE_REMOUNT_USED = True
                    _DRIVER_RECOVERIES.append(descricao)
                    drive.mount("/content/drive", force_remount=True)
            time.sleep(1)

def _file_signature(caminho):
    caminho = Path(caminho)
    st = _drive_io(caminho.stat, f"check {caminho.name}")
    return {"name": caminho.name, "bytes": st.st_size, "mtime_ns": st.st_mtime_ns}

def _drive_is_file(caminho):
    caminho = Path(caminho)
    try:
        st = _drive_io(caminho.stat, f"check {caminho.name}")
    except FileNotFoundError:
        return False
    return _stat.S_ISREG(st.st_mode)

def _local_copy(caminho, grupo="files"):
    """Temporary Colab copy. The source on Drive is read-only."""
    caminho = Path(caminho)
    assinatura = _file_signature(caminho)
    chave = hashlib.sha256(
        json.dumps([str(caminho), assinatura], sort_keys=True).encode()
    ).hexdigest()[:20]
    destino = LOCAL_SESSION_DIR / grupo / f"{chave}_{caminho.name}"
    destino.parent.mkdir(parents=True, exist_ok=True)

    if destino.is_file() and destino.stat().st_size == assinatura["bytes"]:
        return destino

    def copiar():
        temporario = destino.with_suffix(destino.suffix + ".tmp")
        shutil.copyfile(caminho, temporario)
        assert temporario.stat().st_size == assinatura["bytes"], (
            f"Incomplete copy: {caminho.name}"
        )
        assert _file_signature(caminho) == assinatura, (
            f"The file changed while it was being read: {caminho.name}"
        )
        temporario.replace(destino)
        return destino

    return _drive_io(copiar, f"copy {caminho.name} to local disk")

def _read_json(caminho):
    local = _local_copy(caminho, "reports")
    return json.loads(local.read_text(encoding="utf-8"))

def _atomic_bytes(caminho, conteudo):
    caminho = Path(caminho)
    def escrever():
        caminho.parent.mkdir(parents=True, exist_ok=True)
        temporario = caminho.with_suffix(caminho.suffix + ".tmp")
        temporario.write_bytes(conteudo)
        temporario.replace(caminho)
    _drive_io(escrever, f"save {caminho.name}")

def converter_numpy(obj):
    if isinstance(obj, np.ndarray):
        return obj.tolist()
    if isinstance(obj, np.generic):
        return obj.item()
    raise TypeError(f"Unsupported type: {type(obj).__name__}")

def _atomic_json(caminho, objeto):
    conteudo = json.dumps(
        objeto, default=converter_numpy, ensure_ascii=False, indent=2
    ).encode("utf-8")
    _atomic_bytes(caminho, conteudo)

assert _stat.S_ISDIR(_drive_io(Path(DATA).stat, "open the data folder").st_mode), DATA
assert _stat.S_ISDIR(_drive_io(TRAIN_DIR.stat, "open the training folder").st_mode), TRAIN_DIR
print("Drive is mounted and the data folder is accessible.")
_SETUP_STAGES_68.add("drive")


### 1.3 — Restore the pinned course code and package versions

The notebook reads the saved F2 configuration, checks the course revision and aligns package versions with the reference run. If a package is already loaded at a different version, restart the runtime when prompted and run the notebook again.


In [ ]:
# Read the saved configuration before importing scientific packages.
_require_stage_68("drive")
ref_a_path = EXPERIMENTS_DIR / "F2_68_fixed_AE_fold_A.json"
ref_a_guardada_68 = _read_json(ref_a_path)
config_f2_guardada_68 = ref_a_guardada_68.get("metadata", ref_a_guardada_68)
assert config_f2_guardada_68["course_revision"] == COURSE_REVISION
assert config_f2_guardada_68["fold_assignment"] == EXPECTED_FOLDS_68

# Reuse or fetch the course code at the pinned revision.
if not (COURSE_DIR / ".git").is_dir():
    if COURSE_DIR.exists() and any(COURSE_DIR.iterdir()):
        raise RuntimeError(
            f"{COURSE_DIR} already contains files. Use a different folder for the course code."
        )
    COURSE_DIR.mkdir(parents=True, exist_ok=True)
    subprocess.run(["git", "init", "--quiet", str(COURSE_DIR)], check=True)
    subprocess.run(
        ["git", "-C", str(COURSE_DIR), "remote", "add", "origin", COURSE_URL],
        check=True,
    )
    subprocess.run(
        ["git", "-C", str(COURSE_DIR), "fetch", "--quiet", "--depth", "1",
         "origin", COURSE_REVISION], check=True,
    )
    subprocess.run(
        ["git", "-C", str(COURSE_DIR), "checkout", "--quiet",
         "--detach", "FETCH_HEAD"], check=True,
    )

course_revision_loaded = subprocess.check_output(
    ["git", "-C", str(COURSE_DIR), "rev-parse", "HEAD"], text=True
).strip()
assert course_revision_loaded == COURSE_REVISION, (
    "The existing checkout is at a different revision. Keep your changes "
    "and use a fresh runtime or a separate folder for the course code."
)

# Match the package versions recorded for F2.
PACKAGE_MAP = {
    "numpy": "numpy", "scipy": "scipy", "sklearn": "scikit-learn",
    "mne": "mne", "matplotlib": "matplotlib", "pandas": "pandas",
    "imblearn": "imbalanced-learn",
}
reference_runtime = config_f2_guardada_68["runtime"]
instalar = []
for module, distribution in PACKAGE_MAP.items():
    expected = reference_runtime.get(distribution)
    try:
        installed = importlib.metadata.version(distribution)
    except importlib.metadata.PackageNotFoundError:
        installed = None
    if expected and expected != "not installed" and installed != expected:
        instalar.append(f"{distribution}=={expected}")
    elif installed is None:
        instalar.append(distribution)

if instalar:
    print("Aligning package versions:", ", ".join(instalar), flush=True)
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "--quiet", *instalar],
        check=True,
    )

# Reinstalling a package does not replace an already imported module.
loaded_conflicts = {}
for module, distribution in PACKAGE_MAP.items():
    expected = reference_runtime.get(distribution)
    loaded = sys.modules.get(module)
    if expected and loaded is not None and hasattr(loaded, "__version__"):
        if loaded.__version__ != expected:
            loaded_conflicts[distribution] = {
                "loaded": loaded.__version__, "required": expected
            }
if loaded_conflicts:
    raise RuntimeError(
        "The packages were aligned, but the runtime still has old versions loaded. "
        "Restart the Colab runtime and choose Run all again. "
        + str(loaded_conflicts)
    )

for subfolder in ("tracks", "src", "src/bsp"):
    entry = str(COURSE_DIR / subfolder)
    if entry not in sys.path:
        sys.path.insert(0, entry)

import sleep_edf
from sleep_edf import SleepEDFTrack
import numpy as np
import pandas as pd
import scipy
import sklearn
import mne
import matplotlib.pyplot as plt
from IPython.display import display
from sklearn.base import clone
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.exceptions import ConvergenceWarning
from sklearn.metrics import classification_report, ConfusionMatrixDisplay

assert Path(sleep_edf.__file__).resolve().parent == COURSE_DIR / "tracks", (
    "A different sleep_edf module is already loaded. Restart the runtime."
)

runtime_versions = {"python": platform.python_version()}
for distribution in PACKAGE_MAP.values():
    runtime_versions[distribution] = importlib.metadata.version(distribution)

def verificar_runtime_referencia_68():
    diferentes = {
        p: {"F2": reference_runtime[p], "current": runtime_versions[p]}
        for p in ("numpy", "scipy", "scikit-learn", "mne")
        if p in reference_runtime and runtime_versions[p] != reference_runtime[p]
    }
    if diferentes:
        raise RuntimeError("Package versions differ from the F2 reference: " + str(diferentes))

verificar_runtime_referencia_68()
print("Course code revision:", course_revision_loaded)
print("Runtime:", json.dumps(runtime_versions, sort_keys=True))
_SETUP_STAGES_68.add("runtime")


### 1.4 — Restore features, folds and epoch times

The first cell joins the 133 B0 caches in the saved F2 order, rebuilds the 16 features and checks folds A–E. The second restores the original epoch times so TC1 can check which epochs are consecutive. If annotations need to be read, the hypnogram file is copied to the temporary disk before MNE opens it.


In [ ]:
# Restore the cache entries in the order recorded in the F2 report.
_require_stage_68("runtime")
resultados_f2_68 = {
    fold: (ref_a_guardada_68 if fold == "A" else _read_json(
        EXPERIMENTS_DIR / f"F2_68_fixed_AE_fold_{fold}.json"
    ))
    for fold in "ABCDE"
}
FOLDS_68 = {k: list(v) for k, v in config_f2_guardada_68["fold_assignment"].items()}
assert FOLDS_68 == EXPECTED_FOLDS_68
DEVELOPMENT_GROUPS_68 = sorted(s for v in FOLDS_68.values() for s in v)
assert len(DEVELOPMENT_GROUPS_68) == len(set(DEVELOPMENT_GROUPS_68)) == 68
assert set(DEVELOPMENT_GROUPS_68).isdisjoint(HOLDOUT_GROUPS)

def identificar_gravacao_reposta_68(entrada):
    if isinstance(entrada, dict):
        entrada = entrada["record"]
    return Path(str(entrada)).name.removesuffix(".edf").removesuffix("-PSG")

registos_guardados_68 = config_f2_guardada_68["recordings"]
ordem_gravacoes_reposta_68 = [
    identificar_gravacao_reposta_68(r) for r in registos_guardados_68
]
assert len(ordem_gravacoes_reposta_68) == len(set(ordem_gravacoes_reposta_68)) == 133

def _listar_caches_b0():
    with os.scandir(CACHE_VIGGY) as entradas:
        return sorted(
            Path(e.path) for e in entradas
            if e.name.startswith("_cache_viggy_b0_features_") and e.name.endswith(".pkl")
        )

caches_b0_para_repor = _drive_io(_listar_caches_b0, "list the B0 caches")
assert len(caches_b0_para_repor) == 133, (
    f"Found {len(caches_b0_para_repor)} B0 caches; expected 133. "
    "Check the folder before continuing. No download will start."
)
cache_por_gravacao_68 = {}
for i, caminho in enumerate(caches_b0_para_repor, 1):
    local = _local_copy(caminho, "features_b0")
    with local.open("rb") as f:
        dados = pickle.load(f)
    assert {"record", "X", "y", "groups", "feature_names"} <= dados.keys()
    record = identificar_gravacao_reposta_68(dados["record"])
    assert record not in cache_por_gravacao_68, f"Duplicate cache for {record}."
    cache_por_gravacao_68[record] = dados
    if i % 25 == 0 or i == len(caches_b0_para_repor):
        print(f"Caches restored: {i}/{len(caches_b0_para_repor)}", flush=True)

assert set(cache_por_gravacao_68) == set(ordem_gravacoes_reposta_68)
b0_gravacoes_68 = [cache_por_gravacao_68[r] for r in ordem_gravacoes_reposta_68]
nomes_features_b0 = list(b0_gravacoes_68[0]["feature_names"])
nomes_features_f2_68 = list(config_f2_guardada_68["feature_names"])
assert len(nomes_features_b0) == 11
assert nomes_features_f2_68[:11] == nomes_features_b0
assert len(nomes_features_f2_68) == 16

_X, _y, _groups, _record_ids = [], [], [], []
for record, dados, registo in zip(
    ordem_gravacoes_reposta_68, b0_gravacoes_68, registos_guardados_68
):
    xx = np.asarray(dados["X"], dtype=float)
    yy = np.asarray(dados["y"]).astype(str)
    gg = np.asarray(dados["groups"]).astype(str)
    if gg.ndim == 0:
        gg = np.full(len(yy), gg.item())
    assert list(dados["feature_names"]) == nomes_features_b0
    assert xx.shape == (len(yy), 11)
    assert gg.shape == yy.shape == (len(yy),)
    assert np.isfinite(xx).all() and np.all(gg == record[:5])
    if isinstance(registo, dict) and "epochs" in registo:
        assert len(yy) == registo["epochs"]
    _X.append(xx); _y.append(yy); _groups.append(gg)
    _record_ids.append(np.full(len(yy), record))

X_b0_68 = np.vstack(_X)
y_b0_68 = np.concatenate(_y)
groups_b0_68 = np.concatenate(_groups)
record_ids_b0_68 = np.concatenate(_record_ids)
del _X, _y, _groups, _record_ids

B0_FEATURE_VERSION = config_f2_guardada_68.get("details", {}).get("base_feature_version", 1)
F2_FEATURE_VERSION = config_f2_guardada_68["feature_version"]
assert B0_FEATURE_VERSION == F2_FEATURE_VERSION == 1
bandas_f2_68 = ["delta", "theta", "alpha", "sigma", "beta"]
indices_bandas_f2_68 = [nomes_features_b0.index(f"eeg_{b}") for b in bandas_f2_68]
potencias_f2_68 = X_b0_68[:, indices_bandas_f2_68]
assert np.all(potencias_f2_68 >= 0)
_total_f2 = potencias_f2_68.sum(axis=1, keepdims=True)
_relativas_f2 = np.divide(
    potencias_f2_68, _total_f2,
    out=np.zeros_like(potencias_f2_68), where=_total_f2 > 0,
)
X_f2_68 = np.column_stack([X_b0_68, _relativas_f2])
assert X_b0_68.shape == (360621, 11)
assert X_f2_68.shape == (360621, 16)
assert np.isfinite(X_f2_68).all()
assert set(groups_b0_68) == set(DEVELOPMENT_GROUPS_68)

SPLITS_68 = {}
previsoes_f2_68 = np.empty_like(y_b0_68)
fold_por_epoca_68 = np.empty(len(y_b0_68), dtype="<U1")
_cobertura_68 = np.zeros(len(y_b0_68), dtype=np.uint8)
for fold in "ABCDE":
    teste = np.isin(groups_b0_68, FOLDS_68[fold])
    tr, te = np.flatnonzero(~teste), np.flatnonzero(teste)
    ref = resultados_f2_68[fold]
    assert ref["fold_assignment"] == FOLDS_68
    np.testing.assert_array_equal(te, ref["test_indices"])
    np.testing.assert_array_equal(y_b0_68[te], ref["report"]["y_true"])
    np.testing.assert_array_equal(groups_b0_68[te], ref["report"]["y_group"])
    assert set(groups_b0_68[tr]).isdisjoint(set(groups_b0_68[te]))
    assert np.all(_cobertura_68[te] == 0)
    assert len(ref["report"]["y_pred"]) == len(te)
    SPLITS_68[fold] = (tr, te)
    previsoes_f2_68[te] = ref["report"]["y_pred"]
    fold_por_epoca_68[te] = fold
    _cobertura_68[te] += 1
assert np.all(_cobertura_68 == 1)

def _erro_walk_68(exc):
    raise exc

def _indexar_psg_68():
    encontrados = {}
    for pasta, _, ficheiros in os.walk(TRAIN_DIR, onerror=_erro_walk_68):
        for nome in ficheiros:
            if nome.endswith("-PSG.edf"):
                record = identificar_gravacao_reposta_68(nome)
                if record in cache_por_gravacao_68:
                    assert record not in encontrados, f"Duplicate PSG: {record}"
                    encontrados[record] = Path(pasta) / nome
    return encontrados

psg_por_gravacao_68 = _drive_io(_indexar_psg_68, "locate the PSG files")
assert set(psg_por_gravacao_68) == set(ordem_gravacoes_reposta_68)
ficheiros_b0_68 = [psg_por_gravacao_68[r] for r in ordem_gravacoes_reposta_68]
limites_gravacoes_68 = np.r_[0, np.cumsum([len(d["y"]) for d in b0_gravacoes_68])]

print("F2 restored:", X_f2_68.shape)
print("Verified 68 participants, 133 recordings and folds A–E.")
_SETUP_STAGES_68.add("features")


In [ ]:
_require_stage_68("features")
EPOCH_TIMES_VERSION_68 = 1
EPOCH_SECONDS_68 = 30.0
CONSECUTIVE_TOL_68 = 1e-6
RK_TEMPOS_68 = {
    "Sleep stage W": "W", "Sleep stage 1": "N1", "Sleep stage 2": "N2",
    "Sleep stage 3": "N3", "Sleep stage 4": "N3", "Sleep stage R": "REM",
}
META_TEMPOS_68 = {
    "version": EPOCH_TIMES_VERSION_68,
    "course_revision": COURSE_REVISION,
    "epoch_seconds": EPOCH_SECONDS_68,
    "required_native_fs": 100.0,
    "time_origin": "PSG sample zero",
    "event_rounding": True,
    "stage_mapping": RK_TEMPOS_68,
    "mne": mne.__version__,
    "numpy": np.__version__,
}
FICHEIRO_TEMPOS_68 = Path(CACHE_VIGGY) / "viggy_epoch_times_68_v1.json"
_LOCAL_TIMES_PENDING = LOCAL_SESSION_DIR / "epoch_times_pending_v1.json"

def _ler_tempos_edf_68(psg, hyp_local, labels):
    # Read the PSG header and use a small local copy of the hypnogram.
    raw = mne.io.read_raw_edf(psg, preload=False, verbose="ERROR")
    try:
        fs = float(raw.info["sfreq"])
        assert abs(fs - 100.0) < 1e-6 and raw.first_samp == 0
        raw.set_annotations(mne.read_annotations(hyp_local), emit_warning=False)
        eventos, ids = mne.events_from_annotations(
            raw,
            event_id={k: i for i, k in enumerate(RK_TEMPOS_68)},
            use_rounding=True, chunk_duration=EPOCH_SECONDS_68, verbose="ERROR",
        )
        assert np.all(eventos[:, 0] >= 0)
        passo = int(EPOCH_SECONDS_68 * fs)
        eventos = eventos[eventos[:, 0] + passo <= raw.n_times]
        inverso = {codigo: RK_TEMPOS_68[nome] for nome, codigo in ids.items()}
        lidas = np.asarray([inverso[int(e[2])] for e in eventos]).astype(str)
        np.testing.assert_array_equal(lidas, labels, err_msg=f"Labels: {psg.name}")
        return eventos[:, 0].astype(float) / fs, float(raw.n_times / fs)
    finally:
        raw.close()

def _validar_tempos_68(tempos, duracao, labels):
    assert tempos.shape == labels.shape and len(labels) > 0
    assert np.isfinite(tempos).all() and np.isfinite(duracao)
    assert np.all(np.diff(tempos) > 0) and tempos[0] >= 0
    assert tempos[-1] + EPOCH_SECONDS_68 <= duracao + CONSECUTIVE_TOL_68

def preparar_tempos_sessao_68():
    cache = {"metadata": META_TEMPOS_68, "records": {}}
    pendente_reposto = False
    if _drive_is_file(FICHEIRO_TEMPOS_68):
        anterior = _read_json(FICHEIRO_TEMPOS_68)
        if anterior.get("metadata") == META_TEMPOS_68:
            cache = anterior
    if _LOCAL_TIMES_PENDING.is_file():
        pendente = json.loads(_LOCAL_TIMES_PENDING.read_text(encoding="utf-8"))
        if pendente.get("metadata") == META_TEMPOS_68:
            cache["records"].update(pendente["records"])
            pendente_reposto = True

    tempos_por_gravacao, blocos_t, blocos_prev = {}, [], []
    calculadas = reutilizadas = posicao = 0
    alterado = pendente_reposto

    try:
        for numero, (psg, dados) in enumerate(zip(ficheiros_b0_68, b0_gravacoes_68), 1):
            record = identificar_gravacao_reposta_68(dados["record"])
            assert psg.name == f"{record}-PSG.edf"
            assert record[:5] in DEVELOPMENT_GROUPS_68
            labels = np.asarray(dados["y"]).astype(str)

            hyp = psg.with_name(psg.name.replace("-PSG.edf", "-Hypnogram.edf"))
            if not _drive_is_file(hyp):
                candidatos = _drive_io(
                    lambda: sorted(psg.parent.glob(f"{record[:6]}*Hypnogram.edf")),
                    f"locate annotations for {record}",
                )
                assert len(candidatos) == 1, f"Ambiguous annotations: {record}"
                hyp = candidatos[0]

            assinatura = {
                "psg": _file_signature(psg),
                "hypnogram": _file_signature(hyp),
                "labels_sha256": hashlib.sha256("\n".join(labels).encode()).hexdigest(),
            }
            entrada = cache["records"].get(record)
            if entrada is not None and entrada.get("signature") == assinatura:
                tempos = np.asarray(entrada["onsets_seconds"], dtype=float)
                duracao = float(entrada["duration_seconds"])
                reutilizadas += 1
            else:
                hyp_local = _local_copy(hyp, "annotations")
                tempos, duracao = _drive_io(
                    lambda: _ler_tempos_edf_68(psg, hyp_local, labels),
                    f"read the header for {record}",
                )
                _validar_tempos_68(tempos, duracao, labels)
                assert _file_signature(psg) == assinatura["psg"]
                assert _file_signature(hyp) == assinatura["hypnogram"]
                cache["records"][record] = {
                    "signature": assinatura,
                    "onsets_seconds": tempos.tolist(),
                    "duration_seconds": duracao,
                }
                calculadas += 1
                alterado = True

            _validar_tempos_68(tempos, duracao, labels)
            fim = posicao + len(labels)
            assert np.all(record_ids_b0_68[posicao:fim] == record)
            np.testing.assert_array_equal(y_b0_68[posicao:fim], labels)
            posicao = fim

            anterior_valida = np.zeros(len(tempos), dtype=bool)
            anterior_valida[1:] = np.isclose(
                np.diff(tempos), EPOCH_SECONDS_68, rtol=0, atol=CONSECUTIVE_TOL_68
            )
            tempos_por_gravacao[record] = (tempos, duracao)
            blocos_t.append(tempos)
            blocos_prev.append(anterior_valida)

            # Save the first recording, then save at regular checkpoints.
            if numero == 1 or numero % 25 == 0 or numero == len(ficheiros_b0_68):
                if alterado:
                    _atomic_json(FICHEIRO_TEMPOS_68, cache)
                    alterado = False
                if numero % 25 == 0 or numero == len(ficheiros_b0_68):
                    print(f"Epoch times checked: {numero}/{len(ficheiros_b0_68)}", flush=True)
    except (Exception, KeyboardInterrupt):
        # Keep the session work if Drive becomes unavailable.
        _LOCAL_TIMES_PENDING.write_text(
            json.dumps(cache, ensure_ascii=False), encoding="utf-8"
        )
        raise

    assert posicao == len(y_b0_68)
    if _LOCAL_TIMES_PENDING.is_file():
        _LOCAL_TIMES_PENDING.unlink()  # Only the temporary file on local disk.
    return tempos_por_gravacao, np.concatenate(blocos_t), np.concatenate(blocos_prev), calculadas, reutilizadas

(
    tempos_epocas_68, onsets_epocas_68, anterior_consecutiva_68,
    calculados_tempos_68, reutilizados_tempos_68,
) = preparar_tempos_sessao_68()

# Keep the legacy name for compatibility. This does not apply a Wake crop.
tempos_c1_68 = tempos_epocas_68
print(
    f"Epoch times ready: {len(tempos_epocas_68)} recordings | "
    f"{calculados_tempos_68} read now | {reutilizados_tempos_68} reused."
)
print("Epoch-time cache:", FICHEIRO_TEMPOS_68)
_SETUP_STAGES_68.add("times")


### 1.5 — Prepare evaluation helpers and TC1 context

TC1 has **16 current features + 16 previous features + one availability flag**. The previous epoch must be in the same recording and start 30 seconds earlier, within a 1 μs tolerance. At a recording start or gap, TC1 copies the current features and sets the flag to zero. Stage labels and future epochs are not used as inputs.


In [ ]:
import copy
import io
import re
import warnings
from datetime import datetime, timezone

_require_stage_68("times")

ESTAGIOS_68 = ["W", "N1", "N2", "N3", "REM"]
METRICAS_68 = ["cohens_kappa", "macro_f1", "accuracy", "balanced_accuracy"]

track_ref_68 = SleepEDFTrack(cfg={})
track_b0_68 = track_ref_68
track_c1_68 = track_ref_68
modelo_base_f2_68 = Pipeline([
    ("scale", StandardScaler(**config_f2_guardada_68["scaler_params"])),
    ("clf", RandomForestClassifier(**config_f2_guardada_68["classifier_params"])),
])
modelo_base_b0 = modelo_base_f2_68
relatorio_f2_68 = track_ref_68._make_report(
    y_b0_68, previsoes_f2_68, groups=groups_b0_68, folds=fold_por_epoca_68
)
tabela_f2_68 = pd.DataFrame([
    {"Fold": f, **{m: resultados_f2_68[f]["report"][m] for m in METRICAS_68},
     "Treino (min)": resultados_f2_68[f]["fit_seconds"] / 60}
    for f in "ABCDE"
]).set_index("Fold")
_RESULTADOS_DA_SESSAO_68 = globals().get("_RESULTADOS_DA_SESSAO_68", {})

# Translate display labels without changing saved report fields.
_ENGLISH_DISPLAY_LABELS = {'Treino (min)': 'Fit time (min)', 'Diferença': 'Difference', 'TC1 + corte': 'TC1 + Wake crop', 'Treino antes': 'Training epochs before crop', 'Treino depois': 'Training epochs after crop', 'Wake retiradas': 'Wake epochs removed', 'Teste completo': 'Complete test epochs', 'Estágio': 'Stage', 'Épocas': 'Epochs', 'Recall TC1 + corte': 'Recall TC1 + Wake crop', 'F1 TC1 + corte': 'F1 TC1 + Wake crop', 'Diferença F1': 'F1 change', 'Diferença recall': 'Recall change', 'Diferença kappa': 'Kappa change', 'Kappa TC1': 'TC1 kappa', 'Gravação': 'Recording', 'Com LP40': 'Filtered with LP40', 'Com LP40 (%)': 'Filtered with LP40 (%)', 'Mediana 48–50 (%)': 'Median 48–50 (%)', '48–50 antes (%)': '48–50 before (%)', '48–50 depois (%)': '48–50 after (%)', '0.5–30 mantida (%)': '0.5–30 retained (%)', 'Grupo': 'Group', 'Modelo': 'Model', 'Cumpre critério >=10%': 'Meets the ≥10% criterion', 'Restantes épocas': 'Remaining epochs', 'Recall +LP40': 'Recall + LP40', 'F1 +LP40': 'F1 + LP40'}

def _display_en(value):
    if isinstance(value, pd.DataFrame):
        names = _ENGLISH_DISPLAY_LABELS
        value = value.rename(index=names, columns=names).rename_axis(
            index=lambda name: names.get(name, name),
            columns=lambda name: names.get(name, name),
        )
    display(value)


def assinatura_arrays_68(*arrays):
    h = hashlib.sha256()
    for arr in arrays:
        a = np.asarray(arr)
        if a.dtype.kind in "OUS":
            a = a.astype(str)
        a = np.ascontiguousarray(a)
        h.update(str(a.shape).encode())
        h.update(a.dtype.str.encode())
        h.update(memoryview(a).cast("B"))
    return h.hexdigest()

def juntar_previsoes_68(resultados):
    assert set(resultados) == set("ABCDE")
    previsoes = np.empty_like(y_b0_68)
    cobertura = np.zeros(len(y_b0_68), dtype=np.uint8)
    folds = np.empty(len(y_b0_68), dtype="<U1")
    for fold in "ABCDE":
        r = resultados[fold]
        te = np.asarray(r["test_indices"], dtype=int)
        rep = r["report"]
        np.testing.assert_array_equal(te, SPLITS_68[fold][1])
        np.testing.assert_array_equal(rep["y_true"], y_b0_68[te])
        np.testing.assert_array_equal(rep["y_group"], groups_b0_68[te])
        pred = np.asarray(rep["y_pred"])
        assert pred.shape == te.shape and np.isin(pred, ESTAGIOS_68).all()
        assert np.all(cobertura[te] == 0)
        previsoes[te] = pred
        folds[te] = fold
        cobertura[te] += 1
    assert np.all(cobertura == 1)
    rep = track_ref_68._make_report(
        y_b0_68, previsoes, groups=groups_b0_68, folds=folds
    )
    return previsoes, rep

def avaliar_nova_experiencia_68(nome, X, nomes_features, receita, modelo=None):
    """Perform five fits only when this function is called explicitly."""
    assert SESSAO_PRONTA, "Complete section 1 first."
    verificar_runtime_referencia_68()
    assert re.fullmatch(r"[A-Za-z0-9_-]+", nome), "Use a simple experiment identifier."
    assert nome not in {
        "B0_68_fixed_AE", "F2_68_fixed_AE", "LP40_68_fixed_AE",
        "C1_68_crop30_fixed_AE", "LR1_68_fixed_AE",
    }, "Existing experiment identifiers are reserved."

    X = np.asarray(X)
    assert X.shape == (len(y_b0_68), len(nomes_features))
    assert np.isfinite(X).all()
    base = clone(modelo_base_f2_68 if modelo is None else modelo)
    assert isinstance(base, Pipeline)
    manifesto_modelo = [
        {"step": n, "class": f"{type(e).__module__}.{type(e).__qualname__}",
         "params": e.get_params(deep=False)}
        for n, e in base.steps
    ]
    meta = {
        "experiment": nome, "schema_version": 1,
        "course_revision": COURSE_REVISION,
        "reference_experiment": "F2_68_fixed_AE",
        "split_unit": "subject", "cv_method": "five fixed subject folds A-E",
        "fold_assignment": FOLDS_68, "n_subjects": 68,
        "n_recordings": 133, "n_epochs": len(y_b0_68),
        "feature_names": list(nomes_features), "recipe": receita,
        "training_selection": "all original retained training epochs",
        "data_sha256": assinatura_arrays_68(X, y_b0_68, groups_b0_68, record_ids_b0_68),
        "model": manifesto_modelo, "runtime": runtime_versions,
        "feature_cache_runtime": reference_runtime,
    }
    meta = json.loads(json.dumps(meta, default=converter_numpy))
    resultados = {}
    for fold in "ABCDE":
        tr, te = SPLITS_68[fold]
        assert set(groups_b0_68[te]) == set(FOLDS_68[fold])
        assert set(groups_b0_68[tr]).isdisjoint(set(groups_b0_68[te]))
        destino = EXPERIMENTS_DIR / f"{nome}_fold_{fold}.json"
        novo = False
        if _drive_is_file(destino):
            r = _read_json(destino)
        elif (nome, fold) in _RESULTADOS_DA_SESSAO_68:
            r = _RESULTADOS_DA_SESSAO_68[(nome, fold)]
            novo = True
        else:
            print(f"{nome} — fold {fold}: fitting...", flush=True)
            clf = clone(base)
            inicio = time.perf_counter()
            with warnings.catch_warnings():
                warnings.simplefilter("error", ConvergenceWarning)
                clf.fit(X[tr], y_b0_68[tr])
            fit_seconds = time.perf_counter() - inicio
            estimator = clf.steps[-1][1]
            n_iter = getattr(estimator, "n_iter_", None)
            if n_iter is not None and hasattr(estimator, "max_iter"):
                assert np.all(np.asarray(n_iter) < estimator.max_iter), (
                    "The fit reached max_iter. Resolve this before accepting the result."
                )
            inicio = time.perf_counter()
            pred = clf.predict(X[te])
            predict_seconds = time.perf_counter() - inicio
            r = {
                "metadata": copy.deepcopy(meta), "fold": fold,
                "created_at_utc": datetime.now(timezone.utc).isoformat(),
                "train_indices": tr, "test_indices": te,
                "fit_seconds": fit_seconds, "predict_seconds": predict_seconds,
                "n_iter": n_iter, "converged": True,
                "report": track_ref_68._make_report(
                    y_b0_68[te], pred, groups=groups_b0_68[te],
                    folds=np.full(len(te), fold),
                ),
            }
            novo = True
        assert r["metadata"] == meta, (
            f"{nome}, fold {fold}: different configuration or data. "
            "Use a new identifier or version. The previous file has been preserved."
        )
        assert r["fold"] == fold and r["converged"]
        np.testing.assert_array_equal(r["train_indices"], tr)
        np.testing.assert_array_equal(r["test_indices"], te)
        np.testing.assert_array_equal(r["report"]["y_true"], y_b0_68[te])
        np.testing.assert_array_equal(r["report"]["y_group"], groups_b0_68[te])
        assert len(r["report"]["y_pred"]) == len(te)
        _RESULTADOS_DA_SESSAO_68[(nome, fold)] = r
        if novo:
            _atomic_json(destino, r)
        resultados[fold] = r
        print(
            f"Fold {fold}: kappa {r['report']['cohens_kappa']:.3f} | "
            f"fit {r['fit_seconds']/60:.2f} min"
            + ("" if novo else " | reused"),
            flush=True,
        )
    tabela = pd.DataFrame([
        {"Fold": f, **{m: resultados[f]["report"][m] for m in METRICAS_68},
         "Treino (min)": resultados[f]["fit_seconds"] / 60}
        for f in "ABCDE"
    ]).set_index("Fold")
    previsoes, relatorio = juntar_previsoes_68(resultados)
    return resultados, tabela, previsoes, relatorio

def apresentar_experiencia_68(nome, resultados, tabela, previsoes, relatorio, guardar=True):
    """Display and export the results without fitting again."""
    print("METRICS BY FOLD")
    _display_en(tabela.round(3))
    print(
        f"Mean fold kappa ± SD: {tabela['cohens_kappa'].mean():.3f}"
        f" ± {tabela['cohens_kappa'].std(ddof=1):.3f}"
    )
    print(f"Total fit time: {tabela['Treino (min)'].sum():.1f} min")
    comparacao = pd.DataFrame({
        "F2": {m: relatorio_f2_68[m] for m in METRICAS_68},
        nome: {m: relatorio[m] for m in METRICAS_68},
    })
    comparacao["Diferença"] = comparacao[nome] - comparacao["F2"]
    print("\nPOOLED METRICS")
    _display_en(comparacao.round(3))
    print("\nRESULTS BY STAGE")
    print(classification_report(
        y_b0_68, previsoes, labels=ESTAGIOS_68, digits=3, zero_division=0
    ))
    classes = classification_report(
        y_b0_68, previsoes, labels=ESTAGIOS_68, output_dict=True, zero_division=0
    )
    sujeito = pd.DataFrame({
        "F2": pd.DataFrame(relatorio_f2_68["per_group"]).set_index("group")["cohens_kappa"],
        nome: pd.DataFrame(relatorio["per_group"]).set_index("group")["cohens_kappa"],
    })
    assert len(sujeito) == 68 and not sujeito.isna().any().any()
    sujeito["Diferença"] = sujeito[nome] - sujeito["F2"]
    dif = sujeito["Diferença"]
    print(
        "\nParticipants — kappa before rounding:",
        f"{int((dif > 0).sum())} improved |",
        f"{int((dif < 0).sum())} decreased |",
        f"{int((dif == 0).sum())} tied",
    )
    fig, axes = plt.subplots(1, 2, figsize=(13, 5.5))
    for ax, pred, titulo in [
        (axes[0], previsoes_f2_68, "F2 — reference"),
        (axes[1], previsoes, nome),
    ]:
        disp = ConfusionMatrixDisplay.from_predictions(
            y_b0_68, pred, labels=ESTAGIOS_68, normalize="true",
            values_format=".3f", cmap="Blues", colorbar=False, ax=ax,
        )
        disp.im_.set_clim(0, 1)
        ax.set_title(titulo)
    fig.suptitle("68 subjects — fixed folds A–E — complete test recordings")
    fig.tight_layout(rect=(0, 0, 1, .94))
    plt.show()
    plt.close(fig)

    if guardar:
        exportacao = {
            **resultados["A"]["metadata"],
            "cv_mean_sd_across_folds": {
                m: {"mean": float(tabela[m].mean()), "sd": float(tabela[m].std(ddof=1))}
                for m in METRICAS_68
            },
            "sd_ddof": 1, "fold_metrics": tabela.reset_index().to_dict(orient="records"),
            "total_fit_seconds": sum(float(r["fit_seconds"]) for r in resultados.values()),
            "total_predict_seconds": sum(float(r["predict_seconds"]) for r in resultados.values()),
            "report": relatorio, "classification_report": classes,
            "comparison_vs_F2_global": comparacao.rename_axis("metric").reset_index().to_dict(orient="records"),
            "comparison_vs_F2_subject_kappa": sujeito.reset_index().to_dict(orient="records"),
            "working_decision": "Pending review; no automatic replacement of F2.",
        }
        destino = EXPERIMENTS_DIR / f"{nome}.json"
        if not _drive_is_file(destino):
            _atomic_json(destino, exportacao)
            print("Report saved:", destino)
        else:
            print("Existing report preserved:", destino)
        imagem = EXPERIMENTS_DIR / f"{nome}_confusion.png"
        if _drive_is_file(imagem):
            print("Existing image preserved:", imagem)
            return comparacao, sujeito
        fig_ind, ax = plt.subplots(figsize=(7, 6))
        disp = ConfusionMatrixDisplay.from_predictions(
            y_b0_68, previsoes, labels=ESTAGIOS_68, normalize="true",
            values_format=".3f", cmap="Blues", colorbar=False, ax=ax,
        )
        disp.im_.set_clim(0, 1)
        ax.set_title(f"{nome} — 68 subjects — fixed folds A–E")
        fig_ind.tight_layout()
        buffer = io.BytesIO()
        fig_ind.savefig(buffer, format="png", dpi=200, bbox_inches="tight")
        plt.close(fig_ind)
        _atomic_bytes(imagem, buffer.getvalue())
        print("Image saved:", imagem)
    return comparacao, sujeito

_SETUP_STAGES_68.add("helpers")


In [ ]:
# Build context with fixed operations per recording; no fitting.
_require_stage_68("helpers")
TC1_FEATURE_VERSION = 1
NOME_TC1_68 = "TC1_68_fixed_AE"
TC1_CONFIG_68 = {
    "version": TC1_FEATURE_VERSION,
    "base_features": "F2 version 1, original 11 plus five relative EEG powers",
    "previous_epochs": 1,
    "direction": "past only",
    "epoch_seconds": EPOCH_SECONDS_68,
    "consecutive_tolerance_seconds": CONSECUTIVE_TOL_68,
    "require_same_recording": True,
    "missing_previous": "copy current features and set availability flag to zero",
    "availability_flag": True,
    "additional_filter": None,
    "training_selection": "all retained epochs",
    "label_inputs": False,
}

def preparar_contexto_anterior_68(X_atual=None):
    X_atual = X_f2_68 if X_atual is None else np.asarray(X_atual)
    assert X_atual.shape == X_f2_68.shape
    validas = np.asarray(anterior_consecutiva_68, dtype=bool)
    assert validas.shape == (len(X_atual),) and not validas[0]
    indices = np.flatnonzero(validas)
    assert np.all(record_ids_b0_68[indices] == record_ids_b0_68[indices - 1])
    np.testing.assert_allclose(
        onsets_epocas_68[indices] - onsets_epocas_68[indices - 1],
        EPOCH_SECONDS_68, rtol=0, atol=CONSECUTIVE_TOL_68,
    )
    anteriores = X_atual.copy()
    anteriores[indices] = X_atual[indices - 1]
    X_contexto = np.column_stack([X_atual, anteriores, validas.astype(float)])
    nomes = (
        list(nomes_features_f2_68)
        + [f"previous_{n}" for n in nomes_features_f2_68]
        + ["previous_epoch_available"]
    )
    assert X_contexto.shape == (len(y_b0_68), 33)
    assert np.isfinite(X_contexto).all()
    np.testing.assert_array_equal(X_contexto[:, :16], X_atual)
    np.testing.assert_array_equal(X_contexto[indices, 16:32], X_atual[indices - 1])
    return X_contexto, nomes



for _stage in ("drive", "runtime", "features", "times", "helpers"):
    _require_stage_68(_stage)
assert X_b0_68.shape == (360621, 11)
assert X_f2_68.shape == (360621, 16)
assert len(onsets_epocas_68) == len(anterior_consecutiva_68) == len(y_b0_68)
assert len(tempos_epocas_68) == len(ficheiros_b0_68) == 133
assert len(set(groups_b0_68)) == 68
assert FOLDS_68 == EXPECTED_FOLDS_68
assert not set(groups_b0_68).intersection(HOLDOUT_GROUPS)
assert modelo_base_f2_68.named_steps["clf"].n_estimators == 200
assert modelo_base_f2_68.named_steps["clf"].class_weight == "balanced"
assert modelo_base_f2_68.named_steps["clf"].random_state == 0
verificar_runtime_referencia_68()

X_tc1_68, nomes_features_tc1_68 = preparar_contexto_anterior_68()
track = trk = track_ref_68
SESSAO_PRONTA = True

print("SESSION READY")
print(f"F2: {X_f2_68.shape} | TC1: {X_tc1_68.shape}")
print("68 participants | 133 recordings | folds A–E")
print("Epochs with a consecutive predecessor:", int(anterior_consecutiva_68.sum()))


## 2 — Results

### 2.1 — Exploration with eight participants

**Exploratory results.** SC400–SC407, 16 recordings, 44,303 epochs and eight leave-one-subject-out (LOSO) folds. Each test fold is one participant. These scores are separate from the later 68-participant comparison.

| Experiment | Change | Mean kappa ± SD across participants | Pooled kappa | Macro-F1 | Accuracy |
|---|---|---:|---:|---:|---:|
| B0 | 11 original features + Random Forest | 0.824 ± 0.068 | 0.825 | 0.751 | 0.917 |
| F1 | Replace absolute band powers with relative band powers | 0.828 ± 0.054 | 0.829 | 0.758 | 0.918 |
| F2 | Original features + five relative band powers | 0.837 ± 0.052 | 0.837 | 0.762 | 0.922 |
| C1 | Crop distant Wake during training only | 0.813 ± 0.051 | 0.813 | 0.758 | 0.906 |
| P1 | 35 Hz EEG low-pass filter with F2 features | 0.833 ± 0.054 | 0.833 | 0.757 | 0.920 |

F2 became the reference for this phase. The earlier SC400–SC402 pilot gave mean kappa **0.720 ± 0.119** and pooled kappa **0.729** on a different set of participants. The proposed eight-participant M1 experiment was not run.


### 2.2 — Comparisons with 68 participants

All nine experiments use 133 recordings, **360,621 valid test epochs** and the same five folds grouped by participant. Test recordings retain all valid epochs, including Wake. Pooled kappa combines all test predictions; mean kappa gives each fold equal weight. SD describes variation across the five folds.

| Experiment | Change | Pooled kappa | Mean kappa ± SD | Macro-F1 | Accuracy | Balanced accuracy | Fit time (min) |
|---|---|---:|---:|---:|---:|---:|---:|
| B0 | 11 features + Random Forest | 0.728 | 0.728 ± 0.012 | 0.651 | 0.873 | 0.628 | 35.2 |
| F2 | B0 + five relative band powers | 0.735 | 0.735 ± 0.017 | 0.659 | 0.876 | 0.638 | 46.3 |
| LP40 | F2 with a 40 Hz EEG low-pass filter | 0.736 | 0.736 ± 0.018 | 0.661 | 0.876 | 0.641 | 44.6 |
| C1 | F2 with Wake cropping during training only | 0.729 | 0.729 ± 0.024 | 0.666 | 0.868 | 0.653 | 20.0 |
| LR1 | F2 with logistic regression | 0.591 | 0.592 ± 0.024 | 0.600 | 0.768 | 0.705 | 2.8 |
| **TC1** | F2 + features from the preceding epoch | 0.751 | **0.751 ± 0.020** | 0.677 | 0.883 | 0.654 | 35.9 |
| TC1 + Wake crop | TC1 with the same training-only crop as C1 | 0.750 | 0.750 ± 0.026 | **0.688** | 0.879 | 0.672 | 14.5 |
| SEL40 | F2 with LP40 only on epochs meeting the spectral criterion | 0.736 | 0.736 ± 0.017 | 0.660 | 0.876 | 0.640 | 26.3 |
| TC1 + LP40 | TC1 with LP40 on all EEG epochs | 0.752 | 0.751 ± 0.022 | 0.678 | 0.883 | 0.656 | 65.5 |

- **Temporal context:** TC1 improved kappa for 62 of 68 participants compared with F2.
- **Wake cropping:** N1 recall rose from 15.3% to 22.6% with TC1 + Wake crop. Pooled kappa changed little. Test epochs were unchanged.
- **Filtering:** SEL40 filtered 25,446 epochs (7.06%) across 74 recordings. On those same epochs, kappa rose from 0.524 with F2 to 0.538 with SEL40, but the pooled gain was small. Adding LP40 to TC1 left mean kappa unchanged to three decimals and reduced N1 recall from 15.3% to 14.8%. The measured fit times come from separate runs. TC1 without LP40 remains the reference.

The tables show previous results. Per-fold and consolidated JSON files are stored in the experiments folder under CM2013_data.


### 2.3 — Fixed A–E split

Each list gives the test participants for that fold. Training uses the other four groups. Both nights from one participant always stay in the same fold.

| Test fold | Participants |
|---|---|
| A | SC400, SC405, SC411, SC417, SC422, SC428, SC434, SC441, SC446, SC452, SC458, SC463, SC471, SC477 |
| B | SC401, SC406, SC412, SC418, SC423, SC429, SC435, SC442, SC447, SC453, SC459, SC465, SC473, SC481 |
| C | SC402, SC407, SC413, SC419, SC425, SC430, SC436, SC443, SC449, SC454, SC460, SC466, SC474, SC482 |
| D | SC403, SC409, SC414, SC420, SC426, SC431, SC437, SC444, SC450, SC455, SC461, SC467, SC475 |
| E | SC404, SC410, SC415, SC421, SC427, SC433, SC438, SC445, SC451, SC457, SC462, SC470, SC476 |

**Final hold-out:** SC408, SC416, SC424, SC432, SC440, SC448, SC456, SC464, SC472 and SC480.


### 2.4 — Recorded confusion matrices

The tables below are visible without running the notebook. Rows are **reference stages** and columns are **predicted stages**. Each row shows percentages for a true stage, and **bold values** mark correct predictions (recall). Values are rounded, so rows may not add to exactly 100%.

The 68-participant matrices use the same complete A–E test epochs. The earlier eight-participant matrices are kept separately in **5.1**.


#### F2 and SEL40

**F2**

| Reference stage | Pred. W | Pred. N1 | Pred. N2 | Pred. N3 | Pred. REM |
|---|---:|---:|---:|---:|---:|
| W | **98.0%** | 0.7% | 0.9% | 0.1% | 0.4% |
| N1 | 41.2% | **13.5%** | 30.3% | 0.2% | 14.9% |
| N2 | 7.7% | 2.3% | **81.3%** | 2.2% | 6.4% |
| N3 | 6.6% | 0.0% | 24.9% | **68.1%** | 0.5% |
| REM | 13.8% | 6.1% | 21.7% | 0.2% | **58.1%** |

**SEL40**

| Reference stage | Pred. W | Pred. N1 | Pred. N2 | Pred. N3 | Pred. REM |
|---|---:|---:|---:|---:|---:|
| W | **97.9%** | 0.7% | 1.0% | 0.1% | 0.4% |
| N1 | 40.8% | **13.5%** | 30.1% | 0.2% | 15.5% |
| N2 | 7.6% | 2.3% | **81.4%** | 2.3% | 6.4% |
| N3 | 6.3% | 0.0% | 24.7% | **68.6%** | 0.4% |
| REM | 14.4% | 7.1% | 19.3% | 0.2% | **58.9%** |


#### TC1 and its two variants

**TC1**

| Reference stage | Pred. W | Pred. N1 | Pred. N2 | Pred. N3 | Pred. REM |
|---|---:|---:|---:|---:|---:|
| W | **98.1%** | 0.7% | 0.9% | 0.1% | 0.3% |
| N1 | 41.7% | **15.3%** | 29.1% | 0.2% | 13.7% |
| N2 | 7.0% | 1.9% | **83.2%** | 2.1% | 5.8% |
| N3 | 5.9% | 0.0% | 24.9% | **68.9%** | 0.3% |
| REM | 12.2% | 5.3% | 20.9% | 0.1% | **61.5%** |

**TC1 + Wake crop**

| Reference stage | Pred. W | Pred. N1 | Pred. N2 | Pred. N3 | Pred. REM |
|---|---:|---:|---:|---:|---:|
| W | **96.4%** | 1.3% | 1.7% | 0.1% | 0.4% |
| N1 | 31.3% | **22.6%** | 32.0% | 0.2% | 14.0% |
| N2 | 4.2% | 2.8% | **85.1%** | 2.1% | 5.8% |
| N3 | 3.5% | 0.0% | 25.9% | **70.3%** | 0.3% |
| REM | 8.4% | 7.3% | 22.5% | 0.1% | **61.7%** |

**TC1 + LP40**

| Reference stage | Pred. W | Pred. N1 | Pred. N2 | Pred. N3 | Pred. REM |
|---|---:|---:|---:|---:|---:|
| W | **98.0%** | 0.6% | 1.0% | 0.0% | 0.3% |
| N1 | 41.5% | **14.8%** | 28.9% | 0.1% | 14.6% |
| N2 | 7.0% | 1.7% | **83.4%** | 2.1% | 5.8% |
| N3 | 5.2% | 0.0% | 24.5% | **70.0%** | 0.3% |
| REM | 13.3% | 6.0% | 18.7% | 0.1% | **61.9%** |


#### B0, LP40, C1 and LR1

**B0**

| Reference stage | Pred. W | Pred. N1 | Pred. N2 | Pred. N3 | Pred. REM |
|---|---:|---:|---:|---:|---:|
| W | **98.0%** | 0.7% | 0.9% | 0.1% | 0.4% |
| N1 | 41.3% | **13.1%** | 31.1% | 0.2% | 14.3% |
| N2 | 8.2% | 2.2% | **80.9%** | 2.2% | 6.4% |
| N3 | 7.4% | 0.1% | 25.5% | **66.7%** | 0.4% |
| REM | 14.2% | 5.9% | 24.3% | 0.2% | **55.3%** |

**LP40**

| Reference stage | Pred. W | Pred. N1 | Pred. N2 | Pred. N3 | Pred. REM |
|---|---:|---:|---:|---:|---:|
| W | **97.9%** | 0.6% | 1.0% | 0.0% | 0.4% |
| N1 | 41.3% | **12.9%** | 30.2% | 0.2% | 15.4% |
| N2 | 7.7% | 2.3% | **81.4%** | 2.2% | 6.4% |
| N3 | 6.1% | 0.0% | 24.4% | **69.0%** | 0.6% |
| REM | 14.5% | 7.0% | 19.0% | 0.2% | **59.3%** |

**C1**

| Reference stage | Pred. W | Pred. N1 | Pred. N2 | Pred. N3 | Pred. REM |
|---|---:|---:|---:|---:|---:|
| W | **95.9%** | 1.5% | 1.8% | 0.1% | 0.6% |
| N1 | 32.0% | **19.5%** | 32.7% | 0.2% | 15.5% |
| N2 | 4.8% | 3.2% | **83.1%** | 2.2% | 6.6% |
| N3 | 4.6% | 0.0% | 25.7% | **69.0%** | 0.6% |
| REM | 9.7% | 8.2% | 22.8% | 0.2% | **59.0%** |

**LR1**

| Reference stage | Pred. W | Pred. N1 | Pred. N2 | Pred. N3 | Pred. REM |
|---|---:|---:|---:|---:|---:|
| W | **82.0%** | 11.7% | 1.8% | 2.5% | 2.1% |
| N1 | 7.9% | **45.6%** | 21.9% | 3.0% | 21.6% |
| N2 | 3.0% | 7.0% | **65.0%** | 12.2% | 12.7% |
| N3 | 2.8% | 0.2% | 6.4% | **90.1%** | 0.5% |
| REM | 2.7% | 16.6% | 8.7% | 2.1% | **69.9%** |


#### Optional plots from saved predictions

After setup, choose a comparison in **MOSTRAR_MATRIZES_68** (section 1.1). The options are listed below. **F2 vs SEL40** is the default. If the SEL40 JSON is missing, the viewer checks the five fold reports and then the saved PNG. It reads results without retraining or exporting anything.


In [ ]:
# Read saved validation predictions without fitting or exporting.
MATRIZES_DISPONIVEIS_68 = {
    "F2 vs SEL40": (
        ("F2", "F2_68_fixed_AE"), ("SEL40", "SEL40_68_fixed_AE")
    ),
    "B0 vs F2": (
        ("B0", "B0_68_fixed_AE"), ("F2", "F2_68_fixed_AE")
    ),
    "F2 vs LP40": (
        ("F2", "F2_68_fixed_AE"), ("LP40", "LP40_68_fixed_AE")
    ),
    "F2 vs C1": (
        ("F2", "F2_68_fixed_AE"), ("C1", "C1_68_crop30_fixed_AE")
    ),
    "F2 vs LR1": (
        ("F2", "F2_68_fixed_AE"), ("LR1", "LR1_68_fixed_AE")
    ),
    "F2 vs TC1": (
        ("F2", "F2_68_fixed_AE"), ("TC1", "TC1_68_fixed_AE")
    ),
    "TC1 vs TC1 + Wake crop": (
        ("TC1", "TC1_68_fixed_AE"), ("TC1 + Wake crop", "TC1_C1_68_crop30_fixed_AE")
    ),
    "TC1 vs TC1 + LP40": (
        ("TC1", "TC1_68_fixed_AE"), ("TC1 + LP40", "TC1_LP40_68_fixed_AE")
    ),
}


def ler_previsoes_guardadas_68(nome):
    """Check that saved predictions match the test epochs."""
    if nome == "F2_68_fixed_AE":
        # Already checked and assembled from all five folds during setup.
        return previsoes_f2_68

    caminho = EXPERIMENTS_DIR / f"{nome}.json"
    if _drive_is_file(caminho):
        guardado = _read_json(caminho)
        assert guardado["experiment"] == nome, caminho.name
        assert guardado["course_revision"] == COURSE_REVISION, caminho.name
        assert guardado["fold_assignment"] == FOLDS_68, caminho.name
        rep = guardado["report"]
        assert rep["labels"] == ESTAGIOS_68, caminho.name
        np.testing.assert_array_equal(rep["y_true"], y_b0_68, err_msg=caminho.name)
        np.testing.assert_array_equal(rep["y_group"], groups_b0_68, err_msg=caminho.name)
        np.testing.assert_array_equal(rep["y_fold"], fold_por_epoca_68, err_msg=caminho.name)
        pred = np.asarray(rep["y_pred"])
        assert pred.shape == y_b0_68.shape and np.isin(pred, ESTAGIOS_68).all()
        return pred

    # SEL40 can also be restored from the five fold checkpoints.
    if nome == "SEL40_68_fixed_AE":
        caminhos = {f: EXPERIMENTS_DIR / f"{nome}_fold_{f}.json" for f in "ABCDE"}
        if all(_drive_is_file(p) for p in caminhos.values()):
            resultados = {f: _read_json(p) for f, p in caminhos.items()}
            meta = resultados["A"]["metadata"]
            assert meta["experiment"] == nome
            assert meta["course_revision"] == COURSE_REVISION
            assert meta["fold_assignment"] == FOLDS_68
            for f, r in resultados.items():
                assert r["fold"] == f and r["metadata"] == meta
            return juntar_previsoes_68(resultados)[0]
    return None


def mostrar_matrizes_guardadas_68(comparacao):
    """Show row percentages on the same colour scale."""
    assert SESSAO_PRONTA, "Run section 1 first."
    if comparacao not in MATRIZES_DISPONIVEIS_68:
        raise ValueError(f"Choose a comparison: {', '.join(MATRIZES_DISPONIVEIS_68)}")

    modelos = MATRIZES_DISPONIVEIS_68[comparacao]
    preds = [ler_previsoes_guardadas_68(nome) for _, nome in modelos]
    if any(p is None for p in preds):
        from IPython.display import Image

        for (_, nome), pred in zip(modelos, preds):
            if pred is not None:
                continue
            sufixo = "_confusion_vs_TC1.png" if nome == "TC1_LP40_68_fixed_AE" else "_confusion.png"
            imagem = EXPERIMENTS_DIR / f"{nome}{sufixo}"
            if _drive_is_file(imagem):
                print(f"Saved image: {imagem.name}")
                _display_en(Image(data=_local_copy(imagem, "figures").read_bytes(), format="png"))
            else:
                print(f"No saved predictions or image for {nome} in {EXPERIMENTS_DIR}.")
        return

    fig, axes = plt.subplots(1, 2, figsize=(12, 5), layout="constrained")
    for ax, (titulo, _), pred in zip(axes, modelos, preds):
        disp = ConfusionMatrixDisplay.from_predictions(
            y_b0_68, pred, labels=ESTAGIOS_68, normalize="true",
            values_format=".1%", cmap="Blues", colorbar=False, ax=ax,
        )
        disp.im_.set_clim(0, 1)
        ax.set(title=titulo, xlabel="Predicted stage", ylabel="Reference stage")
    fig.suptitle("68 participants · folds A–E · row-normalised percentages")
    plt.show()
    plt.close(fig)


if MOSTRAR_MATRIZES_68 is not None:
    mostrar_matrizes_guardadas_68(MOSTRAR_MATRIZES_68)


## 3 — Optional training

These cells require section 1. Set the relevant switch in **1.1** to **True** to run a training recipe. Fold results are saved as each fold finishes, and matching existing results are reused. Section **2.4** displays the recorded matrices independently of training.

### 3.1 — TC1: temporal context

**RUN_TC1_68** uses the 33 features from section 1.5 and the F2 Random Forest. It uses all training epochs and no additional EEG filter.


In [ ]:
# 3.1 — TC1: temporal context (33 features) + Random Forest.
if RUN_TC1_68:
    (
        resultados_tc1_68, tabela_tc1_68,
        previsoes_tc1_68, relatorio_tc1_68,
    ) = avaliar_nova_experiencia_68(
        nome=NOME_TC1_68,
        X=X_tc1_68,
        nomes_features=nomes_features_tc1_68,
        receita=TC1_CONFIG_68,
    )
    comparacao_global_tc1_68, comparacao_participantes_tc1_68 = (
        apresentar_experiencia_68(
            NOME_TC1_68, resultados_tc1_68, tabela_tc1_68,
            previsoes_tc1_68, relatorio_tc1_68, guardar=True,
        )
    )


### 3.2 — TC1 + Wake crop

**RUN_TC1_C1_68** reuses the C1 selection: the interval from the first to the last sleep stage plus 30 minutes on each side. Temporal features are computed before cropping. Only training loses distant Wake; the test recordings remain complete.


In [ ]:
def treinar_tc1_c1_68():
    assert SESSAO_PRONTA, "Run section 1 first."
    verificar_runtime_referencia_68()

    refs = {
        f: _read_json(EXPERIMENTS_DIR / f"TC1_68_fixed_AE_fold_{f}.json")
        for f in "ABCDE"
    }

    meta_ref = refs["A"]["metadata"]
    base = clone(modelo_base_f2_68)

    modelo_atual = [
        {
            "step": n,
            "class": f"{type(e).__module__}.{type(e).__qualname__}",
            "params": e.get_params(deep=False),
        }
        for n, e in base.steps
    ]
    modelo_atual = json.loads(
        json.dumps(modelo_atual, default=converter_numpy)
    )

    # Verify that features and model match TC1.
    assert meta_ref["experiment"] == "TC1_68_fixed_AE"
    assert meta_ref["runtime"] == runtime_versions
    assert meta_ref["course_revision"] == COURSE_REVISION
    assert meta_ref["fold_assignment"] == FOLDS_68
    assert meta_ref["feature_names"] == list(nomes_features_tc1_68)
    assert meta_ref["recipe"] == TC1_CONFIG_68
    assert meta_ref["model"] == modelo_atual
    assert np.isfinite(X_tc1_68).all()

    assert meta_ref["data_sha256"] == assinatura_arrays_68(
        X_tc1_68, y_b0_68, groups_b0_68, record_ids_b0_68
    )

    for f in "ABCDE":
        assert refs[f]["metadata"] == meta_ref
        np.testing.assert_array_equal(
            refs[f]["train_indices"], SPLITS_68[f][0]
        )

    juntar_previsoes_68(refs)

    meta = {
        **copy.deepcopy(meta_ref),
        "experiment": NOME_TC1_C1_68,
        "schema_version": 2,
        "reference_experiment": "TC1_68_fixed_AE",
        "recipe": copy.deepcopy(TC1_C1_CONFIG_68),
        "training_selection": copy.deepcopy(CORTE_TC1_C1_68),
        "selected_splits_sha256": {
            f: assinatura_arrays_68(*SPLITS_TC1_C1_68[f])
            for f in "ABCDE"
        },
    }

    resultados = {}

    for fold in "ABCDE":
        treino, teste = SPLITS_TC1_C1_68[fold]
        original, teste_original = SPLITS_68[fold]

        np.testing.assert_array_equal(teste, teste_original)
        assert np.isin(treino, original).all()
        assert np.all(
            y_b0_68[np.setdiff1d(original, treino)] == "W"
        )
        assert (
            set(groups_b0_68[treino])
            == set(groups_b0_68[original])
        )
        assert np.intersect1d(treino, teste).size == 0

        destino = (
            EXPERIMENTS_DIR
            / f"{NOME_TC1_C1_68}_fold_{fold}.json"
        )
        existe = _drive_is_file(destino)
        chave = (NOME_TC1_C1_68, fold)

        r = (
            _read_json(destino)
            if existe
            else _RESULTADOS_DA_SESSAO_68.get(chave)
        )

        if r is None:
            print(
                f"TC1 + Wake crop — fold {fold}: "
                f"fitting with {len(treino)} epochs...",
                flush=True,
            )

            # Fit a fresh scaler and classifier for this fold.
            modelo = clone(base)

            inicio = time.perf_counter()
            modelo.fit(X_tc1_68[treino], y_b0_68[treino])
            tempo_treino = time.perf_counter() - inicio

            inicio = time.perf_counter()
            pred = modelo.predict(X_tc1_68[teste])
            tempo_previsao = time.perf_counter() - inicio

            r = {
                "metadata": copy.deepcopy(meta),
                "fold": fold,
                "created_at_utc": datetime.now(timezone.utc).isoformat(),
                "train_indices": treino.copy(),
                "test_indices": teste.copy(),
                "training_windows": TC1_C1_JANELAS_68[fold],
                "fit_seconds": tempo_treino,
                "predict_seconds": tempo_previsao,
                "n_iter": None,
                "converged": True,
                "report": track_ref_68._make_report(
                    y_b0_68[teste],
                    pred,
                    groups=groups_b0_68[teste],
                    folds=np.full(len(teste), fold),
                ),
            }

        # Check results restored from Drive or memory.
        assert r["metadata"] == meta, (
            "The saved result uses a different configuration and has been preserved."
        )
        assert r["fold"] == fold and r["converged"]
        assert r["training_windows"] == TC1_C1_JANELAS_68[fold]

        np.testing.assert_array_equal(r["train_indices"], treino)
        np.testing.assert_array_equal(r["test_indices"], teste)
        np.testing.assert_array_equal(
            r["report"]["y_true"], y_b0_68[teste]
        )
        np.testing.assert_array_equal(
            r["report"]["y_group"], groups_b0_68[teste]
        )
        assert np.isin(r["report"]["y_pred"], ESTAGIOS_68).all()

        # Keep a copy in memory before writing to Drive.
        _RESULTADOS_DA_SESSAO_68[chave] = r

        if not existe:
            _atomic_json(destino, r)

        resultados[fold] = r

        print(
            f"Fold {fold}: kappa {r['report']['cohens_kappa']:.3f} | "
            f"fit {r['fit_seconds'] / 60:.2f} min"
            + (" | reused" if existe else ""),
            flush=True,
        )

    tabela = pd.DataFrame([
        {
            "Fold": f,
            **{
                m: resultados[f]["report"][m]
                for m in METRICAS_68
            },
            "Treino (min)": resultados[f]["fit_seconds"] / 60,
        }
        for f in "ABCDE"
    ]).set_index("Fold")

    comparacao = pd.DataFrame({
        "TC1": [
            refs[f]["report"]["cohens_kappa"]
            for f in "ABCDE"
        ],
        "TC1 + corte": tabela["cohens_kappa"].to_numpy(),
    }, index=tabela.index)

    comparacao["Diferença"] = (
        comparacao["TC1 + corte"] - comparacao["TC1"]
    )

    _display_en(tabela.round(3))

    print(
        f"Mean kappa ± SD: "
        f"{tabela['cohens_kappa'].mean():.3f} "
        f"± {tabela['cohens_kappa'].std(ddof=1):.3f}"
    )
    print(
        f"Total fit time: {tabela['Treino (min)'].sum():.1f} min"
    )

    print("COMPARISON WITH TC1 ON THE SAME FOLDS")
    _display_en(comparacao.round(3))

    return resultados, tabela, comparacao


if RUN_TC1_C1_68:
    assert SESSAO_PRONTA, "Run section 1 first."
    assert X_tc1_68.shape == (len(y_b0_68), 33)

    NOME_TC1_C1_68 = "TC1_C1_68_crop30_fixed_AE"

    CORTE_TC1_C1_68 = {
        "version": 1,
        "margin_minutes": 30.0,
        "epoch_seconds": 30.0,
        "time_basis": "original PSG event times in seconds",
        "selection": "complete epochs inside the sleep window plus margins",
        "scope": "training only",
        "no_sleep_policy": "keep all epochs",
    }

    TC1_C1_CONFIG_68 = {
        "version": 1,
        "features": dict(TC1_CONFIG_68),
        "training_selection": dict(CORTE_TC1_C1_68),
        "selection_source": "C1_68_crop30_fixed_AE",
        "context_order": "build TC1 before training selection",
    }

    SPLITS_TC1_C1_68 = {}
    TC1_C1_JANELAS_68 = {}
    _resumo_tc1_c1 = []

    for fold in "ABCDE":
        r = _read_json(
            EXPERIMENTS_DIR / f"C1_68_crop30_fixed_AE_fold_{fold}.json"
        )
        treino_original, teste = SPLITS_68[fold]
        fit = np.asarray(r["train_indices"], dtype=int)

        # Check the saved configuration and data order.
        assert r["fold"] == fold
        assert r["course_revision"] == COURSE_REVISION
        assert r["fold_assignment"] == FOLDS_68
        assert r["recordings"] == config_f2_guardada_68["recordings"]
        assert r["training_selection"] == CORTE_TC1_C1_68

        np.testing.assert_array_equal(r["test_indices"], teste)
        np.testing.assert_array_equal(
            r["report"]["y_true"], y_b0_68[teste]
        )
        np.testing.assert_array_equal(
            r["report"]["y_group"], groups_b0_68[teste]
        )

        # Check that only training Wake was removed.
        assert treino.ndim == 1
        assert len(np.unique(treino)) == len(treino)
        assert np.isin(treino, treino_original).all()
        assert np.intersect1d(treino, teste).size == 0
        assert (
            set(groups_b0_68[treino])
            == set(groups_b0_68[treino_original])
        )

        retiradas = np.setdiff1d(treino_original, treino)
        assert np.all(y_b0_68[retiradas] == "W")

        SPLITS_TC1_C1_68[fold] = (treino.copy(), teste.copy())
        TC1_C1_JANELAS_68[fold] = r["training_windows"]

        _resumo_tc1_c1.append({
            "Fold": fold,
            "Treino antes": len(treino_original),
            "Treino depois": len(treino),
            "Wake retiradas": len(retiradas),
            "Teste completo": len(teste),
        })

    resumo_tc1_c1_68 = (
        pd.DataFrame(_resumo_tc1_c1).set_index("Fold")
    )

    _display_en(resumo_tc1_c1_68)
    print("TC1 + Wake crop is ready. No model has been fitted yet.")

    (
        resultados_tc1_c1_68,
        tabela_tc1_c1_68,
        comparacao_tc1_c1_68,
    ) = treinar_tc1_c1_68()


In [ ]:
# Export only if the JSON report or figure is missing.
if RUN_TC1_C1_68:
    _destino_tc1_c1 = EXPERIMENTS_DIR / "TC1_C1_68_crop30_fixed_AE.json"
    _imagem_tc1_c1 = EXPERIMENTS_DIR / "TC1_C1_68_crop30_fixed_AE_confusion.png"
    if not (_drive_is_file(_destino_tc1_c1) and _drive_is_file(_imagem_tc1_c1)):
        from sklearn.metrics import confusion_matrix

        assert SESSAO_PRONTA, "Run section 1 first."
        NOME_TC1_C1_68 = "TC1_C1_68_crop30_fixed_AE"

        resultados_tc1_68 = {
            f: _read_json(
                EXPERIMENTS_DIR / f"TC1_68_fixed_AE_fold_{f}.json"
            )
            for f in "ABCDE"
        }

        resultados_tc1_c1_68 = {
            f: _read_json(
                EXPERIMENTS_DIR / f"{NOME_TC1_C1_68}_fold_{f}.json"
            )
            for f in "ABCDE"
        }

        # Check the five folds and their shared configuration.
        for nome, resultados in [
            ("TC1_68_fixed_AE", resultados_tc1_68),
            (NOME_TC1_C1_68, resultados_tc1_c1_68),
        ]:
            meta = resultados["A"]["metadata"]
            assert meta["experiment"] == nome
            assert meta["course_revision"] == COURSE_REVISION
            assert meta["fold_assignment"] == FOLDS_68

            for f in "ABCDE":
                assert resultados[f]["metadata"] == meta
                assert resultados[f]["fold"] == f

        previsoes_tc1_68, relatorio_tc1_68 = juntar_previsoes_68(
            resultados_tc1_68
        )
        previsoes_tc1_c1_68, relatorio_tc1_c1_68 = juntar_previsoes_68(
            resultados_tc1_c1_68
        )

        # Calculate pooled metrics from the assembled test predictions.
        comparacao_global_tc1_c1_68 = pd.DataFrame({
            "TC1": {
                m: relatorio_tc1_68[m]
                for m in METRICAS_68
            },
            "TC1 + corte": {
                m: relatorio_tc1_c1_68[m]
                for m in METRICAS_68
            },
        })
        comparacao_global_tc1_c1_68["Diferença"] = (
            comparacao_global_tc1_c1_68["TC1 + corte"]
            - comparacao_global_tc1_c1_68["TC1"]
        )

        print("COMPARISON — POOLED METRICS")
        _display_en(comparacao_global_tc1_c1_68.round(3))

        # Results by stage.
        classes_tc1_68 = classification_report(
            y_b0_68,
            previsoes_tc1_68,
            labels=ESTAGIOS_68,
            output_dict=True,
            zero_division=0,
        )
        classes_tc1_c1_68 = classification_report(
            y_b0_68,
            previsoes_tc1_c1_68,
            labels=ESTAGIOS_68,
            output_dict=True,
            zero_division=0,
        )

        print("TC1 + WAKE CROP — RESULTS BY STAGE")
        print(classification_report(
            y_b0_68,
            previsoes_tc1_c1_68,
            labels=ESTAGIOS_68,
            digits=3,
            zero_division=0,
        ))

        comparacao_estagios_tc1_c1_68 = pd.DataFrame([
            {
                "Estágio": e,
                "Recall TC1": classes_tc1_68[e]["recall"],
                "Recall TC1 + corte": classes_tc1_c1_68[e]["recall"],
                "F1 TC1": classes_tc1_68[e]["f1-score"],
                "F1 TC1 + corte": classes_tc1_c1_68[e]["f1-score"],
            }
            for e in ESTAGIOS_68
        ]).set_index("Estágio")

        comparacao_estagios_tc1_c1_68["Diferença F1"] = (
            comparacao_estagios_tc1_c1_68["F1 TC1 + corte"]
            - comparacao_estagios_tc1_c1_68["F1 TC1"]
        )
        _display_en(comparacao_estagios_tc1_c1_68.round(3))

        # Compare participants before rounding.
        comparacao_participantes_tc1_c1_68 = pd.DataFrame({
            "TC1": (
                pd.DataFrame(relatorio_tc1_68["per_group"])
                .set_index("group")["cohens_kappa"]
            ),
            "TC1 + corte": (
                pd.DataFrame(relatorio_tc1_c1_68["per_group"])
                .set_index("group")["cohens_kappa"]
            ),
        })
        assert len(comparacao_participantes_tc1_c1_68) == 68
        assert not comparacao_participantes_tc1_c1_68.isna().any().any()

        comparacao_participantes_tc1_c1_68["Diferença"] = (
            comparacao_participantes_tc1_c1_68["TC1 + corte"]
            - comparacao_participantes_tc1_c1_68["TC1"]
        )
        dif = comparacao_participantes_tc1_c1_68["Diferença"]

        print(
            "Participants — kappa before rounding:",
            int((dif > 0).sum()), "improved |",
            int((dif < 0).sum()), "decreased |",
            int((dif == 0).sum()), "tied",
        )

        # Show matrices side by side on the same scale.
        fig, axes = plt.subplots(1, 2, figsize=(13, 5.5))

        for ax, pred, titulo in [
            (axes[0], previsoes_tc1_68, "TC1 — full training"),
            (
                axes[1],
                previsoes_tc1_c1_68,
                "TC1 + Wake crop — training only",
            ),
        ]:
            disp = ConfusionMatrixDisplay.from_predictions(
                y_b0_68,
                pred,
                labels=ESTAGIOS_68,
                normalize="true",
                values_format=".1%",
                cmap="Blues",
                colorbar=False,
                ax=ax,
            )
            disp.im_.set_clim(0, 1)
            ax.set_title(titulo)

        fig.suptitle(
            "68 subjects — fixed folds A–E — complete test recordings"
        )
        fig.tight_layout(rect=(0, 0, 1, .94))
        plt.show()
        plt.close(fig)

        # Consolidate fold metrics.
        tabela_tc1_c1_68 = pd.DataFrame([
            {
                "Fold": f,
                **{
                    m: resultados_tc1_c1_68[f]["report"][m]
                    for m in METRICAS_68
                },
                "Treino (min)": (
                    resultados_tc1_c1_68[f]["fit_seconds"] / 60
                ),
            }
            for f in "ABCDE"
        ]).set_index("Fold")

        exportacao_tc1_c1_68 = {
            **resultados_tc1_c1_68["A"]["metadata"],
            "cv_mean_sd_across_folds": {
                m: {
                    "mean": float(tabela_tc1_c1_68[m].mean()),
                    "sd": float(tabela_tc1_c1_68[m].std(ddof=1)),
                }
                for m in METRICAS_68
            },
            "sd_ddof": 1,
            "fold_metrics": (
                tabela_tc1_c1_68.reset_index().to_dict(orient="records")
            ),
            "total_fit_seconds": sum(
                r["fit_seconds"]
                for r in resultados_tc1_c1_68.values()
            ),
            "total_predict_seconds": sum(
                r["predict_seconds"]
                for r in resultados_tc1_c1_68.values()
            ),
            "report": relatorio_tc1_c1_68,
            "classification_report": classes_tc1_c1_68,
            "reference_classification_report": classes_tc1_68,
            "confusion_matrix_labels": ESTAGIOS_68,
            "confusion_matrix_counts": confusion_matrix(
                y_b0_68,
                previsoes_tc1_c1_68,
                labels=ESTAGIOS_68,
            ),
            "confusion_matrix_normalized_true": confusion_matrix(
                y_b0_68,
                previsoes_tc1_c1_68,
                labels=ESTAGIOS_68,
                normalize="true",
            ),
            "comparison_vs_TC1_global": (
                comparacao_global_tc1_c1_68.rename_axis("metric")
                .reset_index().to_dict(orient="records")
            ),
            "comparison_vs_TC1_per_stage": (
                comparacao_estagios_tc1_c1_68.reset_index()
                .to_dict(orient="records")
            ),
            "comparison_vs_TC1_subject_kappa": (
                comparacao_participantes_tc1_c1_68.reset_index()
                .to_dict(orient="records")
            ),
            "working_decision": (
                "Pending review of per-stage results; "
                "no automatic model selection."
            ),
        }

        destino = EXPERIMENTS_DIR / f"{NOME_TC1_C1_68}.json"
        if not _drive_is_file(destino):
            _atomic_json(destino, exportacao_tc1_c1_68)

        # Save the single-model matrix for the experiment table.
        fig_ind, ax = plt.subplots(figsize=(7, 6))
        disp = ConfusionMatrixDisplay.from_predictions(
            y_b0_68,
            previsoes_tc1_c1_68,
            labels=ESTAGIOS_68,
            normalize="true",
            values_format=".1%",
            cmap="Blues",
            colorbar=False,
            ax=ax,
        )
        disp.im_.set_clim(0, 1)
        ax.set_title("TC1 + Wake crop — 68 subjects — fixed folds A–E")
        fig_ind.tight_layout()

        buffer = io.BytesIO()
        fig_ind.savefig(
            buffer, format="png", dpi=200, bbox_inches="tight"
        )
        plt.close(fig_ind)

        imagem = (
            EXPERIMENTS_DIR / f"{NOME_TC1_C1_68}_confusion.png"
        )
        if not _drive_is_file(imagem):
            _atomic_bytes(imagem, buffer.getvalue())

        print("Report saved:", destino)
        print("Image saved:", imagem)


### 3.3 — TC1 + LP40

**RUN_TC1_LP40_68** restores the original LP40 caches. It replaces only the EEG features in TC1's current and previous blocks. EOG, EMG and the test epochs stay the same. The same cache loader is used by SEL40 for the recordings that meet the selection rule.


In [ ]:
# Cache hashes in the original 133-recording order.
codigos_lp40 = """
b030fdacb58c 9c9b545aef4e 60dc679ba963 7e15ac284305 5316b789b31a 4d5592aefef2 7f22feb5b2d9 683c0c993183
167530e36088 68303cfdc651 06e7d38fc3c2 dd86b22539cc bc201fc31fc3 24b657561314 7ec6634cb7b3 176388bb512b
702aff11fe11 8713427820ff ebddef574b4c 38a423f895c2 5bbcefadbd1d 3ce20b013faf 0aa2bf7a10a0 e76d32ccb534
07859be07fc3 f92440336d71 a2f56069f8bc 4ff1b0747964 963cbd22e618 32bc416c5b8e a1ee80fbcab5 e32853cb010f
d0042238500a b20b1ab6df66 78a20b0b9ae7 7dc350d6b3ce 2c8ab97b49ca b171d3e864be 600efbd9dbe8 20fab70479de
005c4c639f5c 218d06880792 c51839020ad9 3bcf74123583 3b57dee659ef d54a2ed0f710 cd0329969ff5 636cd657baa6
8a0504b294e1 b114c4ac16c1 118f82ae4896 40947f4cd7a2 e10367e5ad9e b5da7a51fb34 6b4fbc136e19 39b343a047dd
ff6577ca4cfe 0a49ef329150 d4be67cd33dc 1a8e73c865aa 7dedcb20912e 1b3426dd89c4 6b447a92cc6e 902775824b7a
91417780b9ff d3fc718c781d 214e025f06ab ba3772af3e51 73429e0447c5 a8d00b4ad4c5 4605994855be 7978e710c129
3a81ad1a3211 e6cae5bef5c8 b51d229f24b4 6817b7bf54fd 3d5222e62b8d 46fe936f9c32 d1fbc2bf5fc2 ac6b754e8b63
19ff879c5a3d 3a91c1d2881a 9a0e268144b4 7f094fc1dbbd 93df6f933611 7bf01051faac 3ac8efccc2fd f0ced7214af1
5ee43bdc94d2 76902d12e2b5 3603a23865d3 aad8647a807c 6299e1dcb768 6f61e53ed800 a6baae088583 b097c5e45815
5d749926dd1c ad834db76122 34fb16a9c2f7 bab7dd64cddc f277e6905f45 1a77e0535c70 bfa88b7fe3ae 8e4c87d8ba0e
6325f103d470 e473096c4591 60bb0813ffc6 a63866283e44 f5d28d30274f 06c1c4482597 c9d6c045d87d e746ebf73c63
0e28df266a2e e8c1f93fc6c8 30bbe18ae68a 1bac1351d6e9 2dec7fc0be73 14b65e242d35 42d76010a28b 029f8a834b2d
bd921d9a5386 fcaaae6b3254 cecd8abc8a5a 9a98cfc33566 cc2d9b260994 218f49231a91 d4e759dbb4c0 6eb9b07fe362
41e3a5200168 9599603e17a3 2005149db4f1 d0f6f077b43f cb84996d6665
""".split()


def repor_lp40_68(gravacoes_pedidas=None):
    """Read the original LP40 caches without filtering the EDF files again."""
    gravacoes = sorted(set(record_ids_b0_68))
    assert len(gravacoes) == len(codigos_lp40) == 133
    pedidas = set(gravacoes if gravacoes_pedidas is None else gravacoes_pedidas)
    assert pedidas <= set(gravacoes)
    nomes = list(nomes_features_f2_68)
    aux = [nomes.index(n) for n in ("eog_movement", "emg_rms")]
    X = X_f2_68.copy()
    fontes = {}

    # Use only the original entries; ignore duplicates.
    for record, codigo in zip(gravacoes, codigos_lp40):
        if record not in pedidas:
            continue
        caminho = Path(CACHE_VIGGY) / f"_cache_viggy_lp40_features_{codigo}.pkl"
        if not _drive_is_file(caminho):
            raise FileNotFoundError(
                f"Missing original LP40 cache for {record}: {caminho.name}. "
                "No alternate version was used and no features were recomputed."
            )
        with _local_copy(caminho, "features_lp40").open("rb") as ficheiro:
            dados = pickle.load(ficheiro)
        assert dados["record"] == record, caminho.name
        linhas = record_ids_b0_68 == record
        xx = np.asarray(dados["X"], dtype=float)
        assert xx.shape == (int(linhas.sum()), len(nomes)) and np.isfinite(xx).all(), record
        assert list(dados["feature_names"]) == nomes, record
        np.testing.assert_array_equal(dados["y"], y_b0_68[linhas], err_msg=record)
        np.testing.assert_array_equal(dados["groups"], groups_b0_68[linhas], err_msg=record)
        np.testing.assert_allclose(
            xx[:, aux], X_f2_68[linhas][:, aux], rtol=1e-12, atol=0, err_msg=record
        )
        potencias = xx[:, indices_bandas_f2_68]
        assert np.all(potencias >= 0), record
        total = potencias.sum(axis=1, keepdims=True)
        relativas = np.divide(
            potencias, total, out=np.zeros_like(potencias), where=total > 0
        )
        np.testing.assert_allclose(xx[:, 11:], relativas, rtol=1e-12, atol=0, err_msg=record)
        X[linhas] = xx
        fontes[record] = caminho.name

    assert set(fontes) == pedidas
    print(f"Original LP40 caches reused: {len(fontes)}")
    return X, fontes


In [ ]:
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, cohen_kappa_score, f1_score,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler


def correr_tc1_lp40_68():
    nome = "TC1_LP40_68_fixed_AE"
    pasta = Path(EXPERIMENTS_DIR)
    ref = _read_json(pasta / "TC1_68_fixed_AE.json")
    rel = ref["report"]

    X = X_tc1_lp40_68
    y = np.asarray(y_b0_68, dtype=str)
    grupos = np.asarray(groups_b0_68, dtype=str)
    folds_ref = np.asarray(rel["y_fold"], dtype=str)
    rotulos = rel["labels"]

    assert X.shape == (len(y), 33) and np.isfinite(X).all()
    assert list(nomes_features_tc1_lp40_68) == ref["feature_names"]
    assert len(fontes_lp40_68) == ref["n_recordings"]
    np.testing.assert_array_equal(y, rel["y_true"])
    np.testing.assert_array_equal(grupos, rel["y_group"])
    assert set(SPLITS_68) == set(folds_ref) == set("ABCDE")

    for fold in "ABCDE":
        treino, teste = SPLITS_68[fold]
        np.testing.assert_array_equal(treino, np.flatnonzero(folds_ref != fold))
        np.testing.assert_array_equal(teste, np.flatnonzero(folds_ref == fold))

    runtime = {
        "numpy": np.__version__,
        "scipy": scipy.__version__,
        "scikit-learn": sklearn.__version__,
    }
    assert all(ref["runtime"][k] == v for k, v in runtime.items()), (
        "Versions differ from the original TC1 run. Check section 1."
    )

    params = {passo["step"]: passo["params"] for passo in ref["model"]}
    modelo_base = Pipeline([
        ("scale", StandardScaler(**params["scale"])),
        ("clf", RandomForestClassifier(**params["clf"])),
    ])

    def painel(real, previsto):
        return {
            "cohens_kappa": float(cohen_kappa_score(real, previsto)),
            "macro_f1": float(f1_score(
                real, previsto, labels=rotulos, average="macro", zero_division=0
            )),
            "accuracy": float(accuracy_score(real, previsto)),
            "balanced_accuracy": float(balanced_accuracy_score(real, previsto)),
        }

    # Identify the data so only compatible results are reused.
    h = hashlib.sha256()
    for a in (X, y, grupos, folds_ref):
        a = np.ascontiguousarray(a)
        h.update(f"{a.shape}:{a.dtype}|".encode())
        h.update(a.view(np.uint8))

    meta = {
        "experiment": nome,
        "reference_experiment": ref["experiment"],
        "course_revision": ref["course_revision"],
        "data_sha256": h.hexdigest(),
        "feature_names": ref["feature_names"],
        "fold_assignment": ref["fold_assignment"],
        "model": ref["model"],
        "runtime": runtime,
        "training_selection": "all retained epochs",
        "additional_filter": "EEG-only LP40, Butterworth order 4 per pass, sosfiltfilt per epoch",
        "source_lp40_caches": fontes_lp40_68,
    }

    resultados = {}
    oof = np.empty_like(y)
    pred_ref = np.asarray(rel["y_pred"], dtype=str)
    tabela = []

    for fold in "ABCDE":
        treino, teste = SPLITS_68[fold]
        caminho = pasta / f"{nome}_fold_{fold}.json"

        if _drive_io(caminho.is_file, f"verificar fold {fold}"):
            r = _read_json(caminho)
            assert r["metadata"] == meta and r["fold"] == fold, (
                f"The saved file for fold {fold} uses a different configuration."
            )
            pred = np.asarray(r["y_pred"], dtype=str)
            print(f"Fold {fold}: reused the saved result.", flush=True)

        else:
            print(f"Fold {fold}: fitting with {len(treino)} epochs...", flush=True)
            modelo = clone(modelo_base)

            inicio = time.perf_counter()
            modelo.fit(X[treino], y[treino])
            segundos_treino = time.perf_counter() - inicio

            inicio = time.perf_counter()
            pred = modelo.predict(X[teste])
            segundos_previsao = time.perf_counter() - inicio

            r = {
                "metadata": meta,
                "fold": fold,
                "metrics": painel(y[teste], pred),
                "fit_seconds": segundos_treino,
                "predict_seconds": segundos_previsao,
                "y_pred": pred.tolist(),
            }

            _atomic_json(caminho, r)
            print(
                f"Fold {fold}: kappa {r['metrics']['cohens_kappa']:.3f} | "
                f"fit {segundos_treino / 60:.2f} min",
                flush=True,
            )

        assert pred.shape == teste.shape and np.isin(pred, rotulos).all()
        oof[teste] = pred
        resultados[fold] = r

        kappa_ref = cohen_kappa_score(y[teste], pred_ref[teste])
        tabela.append({
            "Fold": fold,
            "Kappa TC1": kappa_ref,
            **r["metrics"],
            "Diferença kappa": r["metrics"]["cohens_kappa"] - kappa_ref,
            "Treino (min)": r["fit_seconds"] / 60,
        })

    tabela = pd.DataFrame(tabela).set_index("Fold")
    _display_en(tabela.round(3))

    k = tabela["cohens_kappa"]
    print(f"Mean kappa ± SD: {k.mean():.3f} ± {k.std(ddof=1):.3f}")
    print(f"Total fit time: {tabela['Treino (min)'].sum():.1f} min")

    globais = pd.DataFrame({
        "TC1": painel(y, pred_ref),
        "TC1 + LP40": painel(y, oof),
    })
    globais["Diferença"] = globais["TC1 + LP40"] - globais["TC1"]
    _display_en(globais.round(3))
    return resultados, oof


if RUN_TC1_LP40_68:
    assert SESSAO_PRONTA, "Run section 1 first."
    X_lp40_68, fontes_lp40_68 = repor_lp40_68()
    _eeg_lp40 = [i for i, n in enumerate(nomes_features_f2_68) if n.startswith("eeg_")]
    assert len(_eeg_lp40) == 14
    _base_com_lp40 = X_f2_68.copy()
    _base_com_lp40[:, _eeg_lp40] = X_lp40_68[:, _eeg_lp40]
    X_tc1_lp40_68, nomes_features_tc1_lp40_68 = preparar_contexto_anterior_68(_base_com_lp40)
    del _base_com_lp40
    print(f"TC1 + LP40 features ready: {X_tc1_lp40_68.shape}")
    resultados_tc1_lp40_68, pred_tc1_lp40_68 = correr_tc1_lp40_68()


In [ ]:
# Export only if the JSON report or figure is missing.
if RUN_TC1_LP40_68:
    _destino_tc1_lp40 = EXPERIMENTS_DIR / "TC1_LP40_68_fixed_AE.json"
    _imagem_tc1_lp40 = EXPERIMENTS_DIR / "TC1_LP40_68_fixed_AE_confusion_vs_TC1.png"
    if not (_drive_is_file(_destino_tc1_lp40) and _drive_is_file(_imagem_tc1_lp40)):
        from sklearn.metrics import confusion_matrix

        assert SESSAO_PRONTA, "Run section 1 first."
        nome = "TC1_LP40_68_fixed_AE"
        pasta = Path(EXPERIMENTS_DIR)
        ref = _read_json(pasta / "TC1_68_fixed_AE.json")
        base = ref["report"]
        rotulos = ESTAGIOS_68

        # Assemble test predictions in the original epoch order.
        y = np.asarray(y_b0_68, dtype=str)
        grupos = np.asarray(groups_b0_68, dtype=str)
        folds = np.asarray(base["y_fold"], dtype=str)
        pred_tc1 = np.asarray(base["y_pred"], dtype=str)
        np.testing.assert_array_equal(y, base["y_true"])
        np.testing.assert_array_equal(grupos, base["y_group"])
        assert base["labels"] == rotulos and set(folds) == set("ABCDE")

        runs = {f: _read_json(pasta / f"{nome}_fold_{f}.json") for f in "ABCDE"}
        meta = runs["A"]["metadata"]
        assert meta["experiment"] == nome and meta["reference_experiment"] == ref["experiment"]
        assert meta["fold_assignment"] == ref["fold_assignment"] == FOLDS_68
        assert meta["model"] == ref["model"] and meta["feature_names"] == ref["feature_names"]

        pred_lp40 = np.empty_like(y)
        for f, r in runs.items():
            assert r["fold"] == f and r["metadata"] == meta
            idx = np.flatnonzero(folds == f)
            np.testing.assert_array_equal(idx, SPLITS_68[f][1])
            p = np.asarray(r["y_pred"], dtype=str)
            assert p.shape == idx.shape and np.isin(p, rotulos).all()
            pred_lp40[idx] = p

        # Calculate differences before rounding.
        def painel_global_tc1_lp40(pred):
            return {
                "cohens_kappa": cohen_kappa_score(y, pred),
                "macro_f1": f1_score(y, pred, labels=rotulos, average="macro", zero_division=0),
                "accuracy": accuracy_score(y, pred),
                "balanced_accuracy": balanced_accuracy_score(y, pred),
            }

        modelos = {"TC1": pred_tc1, "TC1 + LP40": pred_lp40}
        globais = pd.DataFrame({n: painel_global_tc1_lp40(p) for n, p in modelos.items()})
        globais["Diferença"] = globais["TC1 + LP40"] - globais["TC1"]

        classes = {
            n: classification_report(y, p, labels=rotulos, output_dict=True, zero_division=0)
            for n, p in modelos.items()
        }
        a, b = classes["TC1"], classes["TC1 + LP40"]
        por_estagio = pd.DataFrame([
            {"Estágio": e, "Épocas": int(b[e]["support"]),
             "Recall TC1": a[e]["recall"], "Recall +LP40": b[e]["recall"],
             "Diferença recall": b[e]["recall"] - a[e]["recall"],
             "F1 TC1": a[e]["f1-score"], "F1 +LP40": b[e]["f1-score"],
             "Diferença F1": b[e]["f1-score"] - a[e]["f1-score"]}
            for e in rotulos
        ])

        relatorio = track_ref_68._make_report(y, pred_lp40, groups=grupos, folds=folds)
        por_sujeito = pd.DataFrame({
            n: pd.DataFrame(rep["per_group"]).set_index("group")["cohens_kappa"]
            for n, rep in [("TC1", base), ("TC1 + LP40", relatorio)]
        })
        assert len(por_sujeito) == ref["n_subjects"] and not por_sujeito.isna().any().any()
        por_sujeito["Diferença"] = por_sujeito["TC1 + LP40"] - por_sujeito["TC1"]

        tabela_folds = pd.DataFrame([
            {"Fold": f, **r["metrics"], "Treino (min)": r["fit_seconds"] / 60}
            for f, r in runs.items()
        ])
        matrizes = {
            n: confusion_matrix(y, p, labels=rotulos, normalize="true")
            for n, p in modelos.items()
        }

        exportacao = {
            **meta,
            **{k: ref[k] for k in ("schema_version", "split_unit", "cv_method",
                                   "n_subjects", "n_recordings", "n_epochs")},
            "recipe": {**ref["recipe"], "additional_filter": meta["additional_filter"]},
            "cv_mean_sd_across_folds": {
                m: {"mean": float(tabela_folds[m].mean()), "sd": float(tabela_folds[m].std(ddof=1))}
                for m in METRICAS_68
            },
            "sd_ddof": 1,
            "fold_metrics": tabela_folds.to_dict("records"),
            "total_fit_seconds": sum(r["fit_seconds"] for r in runs.values()),
            "total_predict_seconds": sum(r["predict_seconds"] for r in runs.values()),
            "report": relatorio,
            "classification_report": b,
            "reference_classification_report": a,
            "confusion_matrix_labels": rotulos,
            "confusion_matrix_counts": confusion_matrix(y, pred_lp40, labels=rotulos),
            "confusion_matrix_normalized_true": matrizes["TC1 + LP40"],
            "comparison_vs_TC1_global": globais.rename_axis("metric").reset_index().to_dict("records"),
            "comparison_vs_TC1_per_stage": por_estagio.to_dict("records"),
            "comparison_vs_TC1_subject_kappa": por_sujeito.reset_index().to_dict("records"),
        }

        # Compare percentages by reference stage on one colour scale.
        fig, axes = plt.subplots(1, 2, figsize=(12, 5), layout="constrained")
        for ax, (titulo, cm) in zip(axes, matrizes.items()):
            disp = ConfusionMatrixDisplay(cm, display_labels=rotulos)
            disp.plot(ax=ax, cmap="Blues", colorbar=False, values_format=".1%")
            disp.im_.set_clim(0, 1)
            ax.set(title=titulo, xlabel="Predicted stage", ylabel="Reference stage")
        fig.suptitle("68 subjects — fixed folds A–E — row-normalised confusion matrices")

        buffer = io.BytesIO()
        fig.savefig(buffer, format="png", dpi=220, bbox_inches="tight")
        destino = pasta / f"{nome}.json"
        imagem = pasta / f"{nome}_confusion_vs_TC1.png"
        if not _drive_is_file(destino):
            _atomic_json(destino, exportacao)
        if not _drive_is_file(imagem):
            _atomic_bytes(imagem, buffer.getvalue())

        _display_en(globais.round(3))
        _display_en(por_estagio.set_index("Estágio").round(3))
        plt.show()
        plt.close(fig)
        print("Report saved:", destino)
        print("Image saved:", imagem)


### 3.4 — SEL40: selective LP40 with 16 F2 features

**RUN_SEL40_68** uses LP40 on the EEG when power at 48–50 Hz accounts for **at least 10%** of power at 0.5–50 Hz. The same signal rule applies to training and test, without stage labels. The cell restores the spectral ratios and the required LP40 feature caches; the other epochs use the original F2 features.


In [ ]:
from adapter import cached
from scipy.signal import welch

QC_HIGHFREQ_VERSION_68 = 1
QC_HIGHFREQ_SETTINGS_68 = {
    "fs_hz": 100.0,
    "epoch_seconds": 30.0,
    "window": "hann",
    "nperseg": 800,
    "noverlap": 400,
    "detrend": "constant",
    "scaling": "density",
    "total_band": [0.5, 50.0],
    "high_band": [48.0, 50.0],
}

SELECTIVE_LP40_CONFIG_68 = {
    "version": 1,
    "ratio_version": QC_HIGHFREQ_VERSION_68,
    "ratio_settings": dict(QC_HIGHFREQ_SETTINGS_68),
    "threshold_percent": 10.0,
    "threshold_operator": ">=",
    "filter": {
        "cutoff_hz": 40.0,
        "order_per_pass": 4,
        "application": "sosfiltfilt per epoch",
        "padtype": "odd",
    },
    "selection": "signal only; same rule for training and test epochs",
}


def carregar_racios_alta_68(record):
    assert SESSAO_PRONTA, "Run section 1 first."
    assert record[:5] in DEVELOPMENT_GROUPS_68

    caminhos = [
        Path(p) for p in ficheiros_b0_68
        if Path(p).name == f"{record}-PSG.edf"
    ]
    assert len(caminhos) == 1
    psg = caminhos[0]

    indices = np.flatnonzero(record_ids_b0_68 == record)
    tempos, duracao = tempos_epocas_68[record]
    tempos = np.asarray(tempos)
    np.testing.assert_array_equal(
        tempos, onsets_epocas_68[indices]
    )

    chave = {
        "version": QC_HIGHFREQ_VERSION_68,
        "course_revision": COURSE_REVISION,
        "record": record,
        "file": _file_signature(psg),
        "onsets_sha256": assinatura_arrays_68(tempos),
        "settings": QC_HIGHFREQ_SETTINGS_68,
        "runtime": {
            k: runtime_versions[k]
            for k in ["numpy", "scipy", "mne"]
        },
    }

    def calcular():
        local = _local_copy(psg, "inspection_psg")
        raw = mne.io.read_raw_edf(
            local, preload=False, verbose="ERROR"
        )
        try:
            fs = float(raw.info["sfreq"])
            assert np.isclose(
                fs, QC_HIGHFREQ_SETTINGS_68["fs_hz"]
            )
            assert raw.first_samp == 0
            assert np.isclose(raw.n_times / fs, duracao)

            n = int(
                QC_HIGHFREQ_SETTINGS_68["epoch_seconds"] * fs
            )
            inicios = np.rint(tempos * fs).astype(int)
            assert np.all(inicios >= 0)
            assert np.all(inicios + n <= raw.n_times)

            sinal = raw.get_data(picks="EEG Fpz-Cz")[0]
        finally:
            raw.close()

        eeg = np.stack([sinal[i:i + n] for i in inicios])
        del sinal
        assert np.isfinite(eeg).all()

        cfg = QC_HIGHFREQ_SETTINGS_68
        f, p = welch(
            eeg,
            fs=fs,
            window=cfg["window"],
            nperseg=cfg["nperseg"],
            noverlap=cfg["noverlap"],
            detrend=cfg["detrend"],
            scaling=cfg["scaling"],
            axis=-1,
        )

        def potencia(banda):
            mascara = (f >= banda[0]) & (f <= banda[1])
            return np.trapezoid(
                p[:, mascara], f[mascara], axis=1
            )

        total = potencia(cfg["total_band"])
        assert np.all(total > 0), (
            f"An epoch has no measurable power; check {record}."
        )
        percentagem = (
            100 * potencia(cfg["high_band"]) / total
        )

        return {
            "record": record,
            "onsets_seconds": tempos.copy(),
            "percent_48_50": percentagem,
        }

    dados = cached(
        "viggy_highfreq_ratio", chave, calcular, CACHE_VIGGY
    )

    assert dados["record"] == record
    np.testing.assert_array_equal(
        dados["onsets_seconds"], tempos
    )

    percentagem = np.asarray(
        dados["percent_48_50"], dtype=float
    )
    assert percentagem.shape == (len(indices),)
    assert np.isfinite(percentagem).all()
    assert np.all(
        (percentagem >= 0) & (percentagem <= 100 + 1e-9)
    )
    return percentagem


if RUN_SEL40_68 or RUN_DIAGNOSTICO_SEL40_68:
    assert SESSAO_PRONTA, "Run section 1 first."
    assert SELECTIVE_LP40_CONFIG_68["threshold_percent"] == 10.0

    NOME_SEL40_68 = "SEL40_68_fixed_AE"

    FRACAO_ALTA_68 = np.full(len(y_b0_68), np.nan)
    _cobertura_sel40_68 = np.zeros(
        len(y_b0_68), dtype=np.uint8
    )
    _linhas_sel40_gravacoes = []
    _inicio_sel40_qc = time.perf_counter()

    assert len(b0_gravacoes_68) == 133
    assert len({
        str(d["record"]) for d in b0_gravacoes_68
    }) == 133

    for numero, dados in enumerate(b0_gravacoes_68, 1):
        record = str(dados["record"])
        indices = np.flatnonzero(
            record_ids_b0_68 == record
        )
        percentagem = carregar_racios_alta_68(record)

        assert len(indices) == len(dados["y"]) == len(percentagem)
        np.testing.assert_array_equal(
            y_b0_68[indices], dados["y"]
        )
        assert np.all(_cobertura_sel40_68[indices] == 0)

        FRACAO_ALTA_68[indices] = percentagem
        _cobertura_sel40_68[indices] += 1

        selecionadas = percentagem >= SELECTIVE_LP40_CONFIG_68["threshold_percent"]

        _linhas_sel40_gravacoes.append({
            "Gravação": record,
            "Participante": record[:5],
            "Épocas": len(indices),
            "Com LP40": int(selecionadas.sum()),
            "Com LP40 (%)": float(
                100 * selecionadas.mean()
            ),
            "Mediana 48–50 (%)": float(
                np.median(percentagem)
            ),
            "P95 48–50 (%)": float(
                np.percentile(percentagem, 95)
            ),
        })

        if numero % 10 == 0 or numero == len(b0_gravacoes_68):
            print(
                f"Ratios prepared: {numero}/133 | "
                f"tempo: "
                f"{(time.perf_counter() - _inicio_sel40_qc) / 60:.1f} min",
                flush=True,
            )

    # Check that every epoch has exactly one ratio.
    assert np.all(_cobertura_sel40_68 == 1)
    assert np.isfinite(FRACAO_ALTA_68).all()
    assert np.all(
        (FRACAO_ALTA_68 >= 0)
        & (FRACAO_ALTA_68 <= 100 + 1e-9)
    )

    SEL40_MASK_68 = FRACAO_ALTA_68 >= SELECTIVE_LP40_CONFIG_68["threshold_percent"]

    resumo_sel40_gravacoes_68 = pd.DataFrame(
        _linhas_sel40_gravacoes
    ).set_index("Gravação")

    resumo_sel40_estagios_68 = pd.DataFrame([
        {
            "Estágio": e,
            "Épocas": int(np.sum(y_b0_68 == e)),
            "Com LP40": int(
                SEL40_MASK_68[y_b0_68 == e].sum()
            ),
            "Com LP40 (%)": float(
                100 * SEL40_MASK_68[y_b0_68 == e].mean()
            ),
        }
        for e in ESTAGIOS_68
        if np.any(y_b0_68 == e)
    ]).set_index("Estágio")

    assert len(resumo_sel40_gravacoes_68) == 133
    assert (
        resumo_sel40_gravacoes_68["Participante"].nunique()
        == 68
    )
    assert (
        resumo_sel40_gravacoes_68["Épocas"].sum()
        == len(y_b0_68)
    )
    assert (
        resumo_sel40_estagios_68["Épocas"].sum()
        == len(y_b0_68)
    )

    print(f"Total epochs: {len(y_b0_68)}")
    print(
        f"Receberiam LP40: {SEL40_MASK_68.sum()} "
        f"({100 * SEL40_MASK_68.mean():.2f}%)"
    )
    print(
        f"Would retain the original signal: "
        f"{(~SEL40_MASK_68).sum()}"
    )

    print("\nDISTRIBUTION BY STAGE")
    _display_en(resumo_sel40_estagios_68.round(3))

    print("\nTEN RECORDINGS WITH THE HIGHEST SELECTED PERCENTAGE")
    _display_en(
        resumo_sel40_gravacoes_68
        .sort_values(
            ["Com LP40 (%)", "Gravação"],
            ascending=[False, True],
        )
        .head(10)
        .round(3)
    )

    # Save a compact summary; ratios are already cached.
    destino_sel40_qc_68 = (
        EXPERIMENTS_DIR / "SEL40_68_quality_summary_v1.json"
    )

    if not _drive_is_file(destino_sel40_qc_68):
        _atomic_json(destino_sel40_qc_68, {
            "experiment": NOME_SEL40_68,
            "stage": "signal-only quality measurement; no classifier fitted",
            "course_revision": COURSE_REVISION,
            "runtime": runtime_versions,
            "config": SELECTIVE_LP40_CONFIG_68,
            "n_subjects": 68,
            "n_recordings": 133,
            "n_epochs": len(y_b0_68),
            "n_selected": int(SEL40_MASK_68.sum()),
            "percent_selected": float(
                100 * SEL40_MASK_68.mean()
            ),
            "data_sha256": assinatura_arrays_68(
                FRACAO_ALTA_68,
                groups_b0_68,
                record_ids_b0_68,
            ),
            "per_record": (
                resumo_sel40_gravacoes_68.reset_index()
                .to_dict(orient="records")
            ),
            "per_stage": (
                resumo_sel40_estagios_68.reset_index()
                .to_dict(orient="records")
            ),
        })

        print("Summary saved:", destino_sel40_qc_68)

    print("Selection mask prepared and aligned. No model has been fitted yet.")


In [ ]:
if RUN_SEL40_68:
    assert SESSAO_PRONTA, "Run section 1 first."
    assert SELECTIVE_LP40_CONFIG_68["threshold_percent"] == 10.0
    assert SEL40_MASK_68.shape == y_b0_68.shape
    np.testing.assert_array_equal(
        SEL40_MASK_68, FRACAO_ALTA_68 >= 10.0
    )

    _ref_lp40_sel = _read_json(
        EXPERIMENTS_DIR / "LP40_68_fixed_AE_fold_A.json"
    )
    _meta_lp40_sel = _ref_lp40_sel.get(
        "metadata", _ref_lp40_sel
    )

    # Check the original LP40 configuration.
    assert _meta_lp40_sel["course_revision"] == COURSE_REVISION

    for campo in ["recordings", "fold_assignment", "feature_names"]:
        assert (
            _meta_lp40_sel[campo]
            == config_f2_guardada_68[campo]
        )

    assert _meta_lp40_sel["classifier_params"] == (
        modelo_base_f2_68.named_steps["clf"].get_params()
    )
    assert _meta_lp40_sel["scaler_params"] == (
        modelo_base_f2_68.named_steps["scale"].get_params()
    )

    _pp_lp40_sel = _meta_lp40_sel["details"]["preprocessing"]

    assert _pp_lp40_sel["version"] == 1
    assert _pp_lp40_sel["channels"] == ["eeg"]
    assert _pp_lp40_sel["application"] == (
        "sosfiltfilt, per epoch, axis=-1"
    )
    assert _pp_lp40_sel["settings"] == {
        "cutoff_hz": 40.0,
        "order_per_pass": 4,
        "padtype": "odd",
    }
    assert _meta_lp40_sel["feature_version"] == 1
    assert (
        _meta_lp40_sel["details"]["base_feature_version"]
        == B0_FEATURE_VERSION
    )
    assert (
        _meta_lp40_sel["details"]["relative_feature_version"]
        == F2_FEATURE_VERSION
    )



    _registos_sel40 = set(record_ids_b0_68[SEL40_MASK_68])
    X_sel40_68, _fontes_sel40 = repor_lp40_68(_registos_sel40)
    X_sel40_68[~SEL40_MASK_68] = X_f2_68[~SEL40_MASK_68]
    assert X_sel40_68.shape == X_f2_68.shape and np.isfinite(X_sel40_68).all()
    print(f"SEL40: {X_sel40_68.shape} | {int(SEL40_MASK_68.sum())} epochs with LP40")

    SEL40_RECIPE_68 = {
        "version": 1,
        "base_features": (
            "F2 v1, 11 original plus five relative EEG powers"
        ),
        "preprocessing": copy.deepcopy(SELECTIVE_LP40_CONFIG_68),
        "filtered_feature_source": "LP40_68_fixed_AE",
        "filtered_feature_version": _meta_lp40_sel["feature_version"],
        "filtered_feature_runtime": copy.deepcopy(
            _meta_lp40_sel["runtime"]
        ),
        "selection_sha256": assinatura_arrays_68(
            SEL40_MASK_68, record_ids_b0_68
        ),
        "selected_epochs": int(SEL40_MASK_68.sum()),
        "training_selection": (
            "all original retained epochs; no Wake crop"
        ),
    }


    np.testing.assert_array_equal(
        SEL40_MASK_68, FRACAO_ALTA_68 >= 10.0
    )
    assert SELECTIVE_LP40_CONFIG_68["threshold_percent"] == 10.0
    assert (
        SEL40_RECIPE_68["preprocessing"]
        == SELECTIVE_LP40_CONFIG_68
    )
    assert (
        SEL40_RECIPE_68["selected_epochs"]
        == int(SEL40_MASK_68.sum())
    )
    assert (
        SEL40_RECIPE_68["selection_sha256"]
        == assinatura_arrays_68(
            SEL40_MASK_68, record_ids_b0_68
        )
    )

    # Restore the LP40 reference without fitting it again.
    resultados_lp40_68 = {
        f: _read_json(
            EXPERIMENTS_DIR
            / f"LP40_68_fixed_AE_fold_{f}.json"
        )
        for f in "ABCDE"
    }

    for f, r in resultados_lp40_68.items():
        meta = r.get("metadata", r)

        assert r["fold"] == f
        assert meta["course_revision"] == COURSE_REVISION
        assert meta["fold_assignment"] == FOLDS_68
        assert meta["feature_names"] == list(
            nomes_features_f2_68
        )
        assert meta["classifier_params"] == (
            modelo_base_f2_68.named_steps["clf"].get_params()
        )
        assert meta["scaler_params"] == (
            modelo_base_f2_68.named_steps["scale"].get_params()
        )

    previsoes_lp40_68, relatorio_lp40_68 = (
        juntar_previsoes_68(resultados_lp40_68)
    )

    # Fit one model per fold.
    # Keep all training epochs.
    (
        resultados_sel40_68,
        tabela_sel40_68,
        previsoes_sel40_68,
        relatorio_sel40_68,
    ) = avaliar_nova_experiencia_68(
        nome=NOME_SEL40_68,
        X=X_sel40_68,
        nomes_features=nomes_features_f2_68,
        receita=SEL40_RECIPE_68,
    )

    # Show results and the confusion matrix, then export.
    (
        comparacao_global_f2_sel40_68,
        comparacao_participantes_f2_sel40_68,
    ) = apresentar_experiencia_68(
        NOME_SEL40_68,
        resultados_sel40_68,
        tabela_sel40_68,
        previsoes_sel40_68,
        relatorio_sel40_68,
        guardar=True,
    )

    comparacao_folds_sel40_68 = pd.DataFrame({
        "F2": [
            resultados_f2_68[f]["report"]["cohens_kappa"]
            for f in "ABCDE"
        ],
        "LP40": [
            resultados_lp40_68[f]["report"]["cohens_kappa"]
            for f in "ABCDE"
        ],
        "SEL40": (
            tabela_sel40_68["cohens_kappa"].to_numpy()
        ),
    }, index=tabela_sel40_68.index)

    comparacao_folds_sel40_68["SEL40 - F2"] = (
        comparacao_folds_sel40_68["SEL40"]
        - comparacao_folds_sel40_68["F2"]
    )
    comparacao_folds_sel40_68["SEL40 - LP40"] = (
        comparacao_folds_sel40_68["SEL40"]
        - comparacao_folds_sel40_68["LP40"]
    )

    print("\nKAPPA ON MATCHED FOLDS")
    _display_en(comparacao_folds_sel40_68.round(3))

    comparacao_tres_globais_sel40_68 = pd.DataFrame({
        "F2": {
            m: relatorio_f2_68[m]
            for m in METRICAS_68
        },
        "LP40": {
            m: relatorio_lp40_68[m]
            for m in METRICAS_68
        },
        "SEL40": {
            m: relatorio_sel40_68[m]
            for m in METRICAS_68
        },
    })

    print("\nPOOLED METRICS — THREE STRATEGIES")
    _display_en(comparacao_tres_globais_sel40_68.round(3))


## 4 — Optional diagnostics

- **RUN_DIAGNOSTICO_SINAL_68** compares signals before and after LP40 in three recordings, checks slow variation and inspects a transient. It does not fit a model.
- **RUN_DIAGNOSTICO_SEL40_68** compares saved F2, LP40 and SEL40 predictions on the selected and remaining epochs. The original SEL40 report already includes these results.

The slow-variation check found changes below 0.30 Hz in selected windows. It did not establish their source or justify a high-pass model. No high-pass model was trained at this stage.


In [ ]:
if RUN_DIAGNOSTICO_SINAL_68:
    from scipy.signal import butter, sosfiltfilt, welch


    def inspecionar_lp40_68(record):
        assert SESSAO_PRONTA, "Run section 1 first."
        assert record[:5] in DEVELOPMENT_GROUPS_68

        caminhos = [
            Path(p) for p in ficheiros_b0_68
            if Path(p).name == f"{record}-PSG.edf"
        ]
        assert len(caminhos) == 1
        local = _local_copy(caminhos[0], "inspection_psg")

        indices = np.flatnonzero(record_ids_b0_68 == record)
        labels = y_b0_68[indices]
        tempos, duracao = tempos_epocas_68[record]
        tempos = np.asarray(tempos)

        np.testing.assert_array_equal(
            tempos, onsets_epocas_68[indices]
        )

        raw = mne.io.read_raw_edf(
            local, preload=False, verbose="ERROR"
        )
        try:
            fs = float(raw.info["sfreq"])
            assert np.isclose(fs, 100.0) and raw.first_samp == 0
            assert np.isclose(raw.n_times / fs, duracao)

            n = int(30 * fs)
            inicios = np.rint(tempos * fs).astype(int)

            assert len(inicios) == len(labels) and len(labels) > 0
            assert np.all(inicios >= 0)
            assert np.all(inicios + n <= raw.n_times)

            sinal = raw.get_data(picks="EEG Fpz-Cz")[0]
        finally:
            raw.close()

        original = np.stack([sinal[i:i + n] for i in inicios])
        del sinal

        assert original.shape == (len(labels), n)
        assert np.isfinite(original).all()

        # Match the LP40 experiment: order 4 per pass and per epoch.
        sos = butter(
            4, 40.0, btype="lowpass", fs=fs, output="sos"
        )
        filtrado = sosfiltfilt(
            sos, original, axis=-1, padtype="odd"
        )

        # Use eight-second Welch segments for this inspection.
        # This does not change feature extraction.
        psd_cfg = dict(
            fs=fs,
            window="hann",
            nperseg=int(8 * fs),
            noverlap=int(4 * fs),
            detrend="constant",
            scaling="density",
            axis=-1,
        )
        f, p0 = welch(original, **psd_cfg)
        _, p1 = welch(filtrado, **psd_cfg)

        p0 *= 1e12  # V²/Hz -> µV²/Hz
        p1 *= 1e12

        def potencia(psd, inferior, superior):
            banda = (f >= inferior) & (f <= superior)
            return np.trapezoid(
                psd[:, banda], f[banda], axis=1
            )

        total0 = potencia(p0, 0.5, 50)
        total1 = potencia(p1, 0.5, 50)
        baixa0 = potencia(p0, 0.5, 30)
        baixa1 = potencia(p1, 0.5, 30)

        assert np.all(total0 > 0) and np.all(total1 > 0)
        assert np.all(baixa0 > 0)

        alta0 = 100 * potencia(p0, 48, 50) / total0
        alta1 = 100 * potencia(p1, 48, 50) / total1
        retida = 100 * baixa1 / baixa0

        tabela = pd.DataFrame([
            {
                "Estágio": e,
                "Épocas": int(np.sum(labels == e)),
                "48–50 antes (%)": float(
                    np.median(alta0[labels == e])
                ),
                "48–50 depois (%)": float(
                    np.median(alta1[labels == e])
                ),
                "0.5–30 mantida (%)": float(
                    np.median(retida[labels == e])
                ),
            }
            for e in ESTAGIOS_68 if np.any(labels == e)
        ]).set_index("Estágio")

        print(record, "— medians by stage")
        _display_en(tabela.round(3))

        # Choose an N1 epoch near the median
        # of the 48–50 Hz power ratio.
        candidatos = np.flatnonzero(labels == "N1")
        assert len(candidatos) > 0

        i = candidatos[np.argmin(np.abs(
            alta0[candidatos] - np.median(alta0[candidatos])
        ))]

        print(
            f"N1 example: retained epoch {i}, "
            f"starts {tempos[i] / 60:.1f} min "
            "after the recording started."
        )

        t = np.arange(n) / fs
        zoom = (t >= 10) & (t < 11)

        # Median PSD across all N1 epochs.
        med0 = np.median(p0[candidatos], axis=0)
        med1 = np.median(p1[candidatos], axis=0)

        fig, axes = plt.subplots(2, 2, figsize=(13, 7))

        for ax, mascara, titulo in [
            (
                axes[0, 0],
                np.ones(n, dtype=bool),
                "N1 example — full 30-second epoch",
            ),
            (
                axes[0, 1],
                zoom,
                "Same N1 epoch — one-second zoom",
            ),
        ]:
            ax.plot(
                t[mascara], original[i, mascara] * 1e6,
                label="Original", alpha=0.65, linewidth=0.8,
            )
            ax.plot(
                t[mascara], filtrado[i, mascara] * 1e6,
                label="LP40", linewidth=1,
            )
            ax.set(
                title=titulo,
                xlabel="Time within epoch (s)",
                ylabel="EEG (µV)",
            )

        for ax, limites, titulo in [
            (
                axes[1, 0],
                (0.5, 50),
                "Median N1 PSD — all N1 epochs",
            ),
            (
                axes[1, 1],
                (45, 50),
                "Same median PSD — zoom near 50 Hz",
            ),
        ]:
            visivel = f >= 0.5
            ax.semilogy(
                f[visivel],
                np.maximum(med0[visivel], 1e-20),
                label="Original",
            )
            ax.semilogy(
                f[visivel],
                np.maximum(med1[visivel], 1e-20),
                label="LP40",
            )
            ax.axvspan(48, 50, color="grey", alpha=0.15)
            ax.set(
                title=titulo,
                xlabel="Frequency (Hz)",
                ylabel="PSD (µV²/Hz)",
                xlim=limites,
            )

        axes[1, 0].axvline(
            40, color="grey", linestyle="--"
        )

        for ax in axes.flat:
            ax.legend()
            ax.grid(True, alpha=0.25)

        fig.suptitle(
            f"{record} — EEG Fpz-Cz — LP40 inspection"
        )
        fig.tight_layout(rect=(0, 0, 1, .95))
        plt.show()
        plt.close(fig)

        return tabela



    comparacao_qc_lp40_68 = pd.concat(
        {record: inspecionar_lp40_68(record)
         for record in ("SC4611E0", "SC4531E0", "SC4141E0")},
        names=["Gravação", "Estágio"],
    ).reset_index()
    _display_en(comparacao_qc_lp40_68.round(3))


In [ ]:
if RUN_DIAGNOSTICO_SINAL_68:
    from scipy.signal import welch

    # Slow variation in two-minute windows.
    PASTA_TREINO = TRAIN_DIR
    PESSOAS = ["SC435", "SC461", "SC414"]
    ETAPAS = {f"Sleep stage {s}": i for i, s in enumerate("W1234R", 1)}
    assert PASTA_TREINO.is_dir(), "Mount Google Drive first."

    def medir_deriva(eeg_uv, onsets, fs):
        epoca = round(30 * fs)
        blocos = np.split(onsets, np.flatnonzero(np.diff(onsets) != epoca) + 1)

        # Use four consecutive valid epochs per window.
        inicios = np.concatenate([b[:4 * (len(b) // 4):4] for b in blocos])
        inicios = inicios[(inicios >= 0) & (inicios + 4 * epoca <= len(eeg_uv))]
        assert len(inicios), "No valid two-minute windows were found."

        janelas = np.stack([eeg_uv[i:i + 4 * epoca] for i in inicios])
        assert np.isfinite(janelas).all(), "Inspect the non-finite samples."

        f, psd = welch(janelas, fs=fs, nperseg=round(60 * fs),
                       noverlap=round(30 * fs), window="hann", detrend="constant")
        lenta = (f >= 0.05) & (f <= 0.30)
        total = (f >= 0.05) & (f <= 4.0)
        p_lenta = np.trapezoid(psd[:, lenta], f[lenta], axis=1)
        p_total = np.trapezoid(psd[:, total], f[total], axis=1)
        validas = np.flatnonzero(p_total > 0)
        assert len(validas), "No measurable power in this band."

        pct = 100 * p_lenta[validas] / p_total[validas]
        exemplo = validas[np.argmin(abs(pct - np.percentile(pct, 95)))]
        return f, psd, janelas, inicios, pct, exemplo

    fig, eixos = plt.subplots(3, 2, figsize=(12, 8), layout="constrained")

    for i, pessoa in enumerate(PESSOAS):
        candidatos = sorted(PASTA_TREINO.rglob(f"{pessoa}*-PSG.edf"))
        assert candidatos, f"Missing PSG for {pessoa} em train."
        psg = candidatos[0]
        hip = list(psg.parent.glob(f"{psg.name[:6]}*-Hypnogram.edf"))
        assert len(hip) == 1, f"Check the annotations for {psg.name}."

        raw = mne.io.read_raw_edf(
            psg, include=["EEG Fpz-Cz"], preload=True, verbose="ERROR"
        )
        raw.set_annotations(mne.read_annotations(_local_copy(hip[0], "annotations")), emit_warning=False)
        eventos, _ = mne.events_from_annotations(
            raw, event_id=ETAPAS, chunk_duration=30.0, verbose="ERROR"
        )
        fs = raw.info["sfreq"]
        onsets = eventos[:, 0] - raw.first_samp
        eeg_uv = raw.get_data()[0] * 1e6
        raw.close()

        f, psd, janelas, inicios, pct, j = medir_deriva(eeg_uv, onsets, fs)
        x = janelas[j]
        passo = round(2 * fs)
        tendencia = np.median(x[:len(x) // passo * passo].reshape(-1, passo), axis=1)

        ax, sp = eixos[i]
        ax.plot(np.arange(len(x)) / fs, x, lw=0.5, alpha=0.55)
        ax.plot((np.arange(len(tendencia)) + 0.5) * passo / fs, tendencia,
                color="tab:orange", lw=2, label="Two-second median")
        ax.set(title=f"{psg.name[:8]} — window near P95",
               xlabel="Time within window (s)", ylabel="EEG (µV)")
        ax.legend(fontsize=8)

        b = (f >= 0.05) & (f <= 4)
        sp.semilogy(f[b], np.median(psd[:, b], axis=0), label="Recording median")
        sp.semilogy(f[b], psd[j, b], color="tab:orange", label="Selected window")
        sp.axvspan(0.05, 0.30, color="tab:orange", alpha=0.15)
        sp.set(xlabel="Frequency (Hz)", ylabel="PSD (µV²/Hz)")
        sp.legend(fontsize=8)

        print(f"{psg.name[:8]} | {len(pct)} windows | power 0.05–0.30 Hz / 0.05–4 Hz: "
              f"median {np.median(pct):.2f}% | P95 {np.percentile(pct, 95):.2f}% | "
              f"example starts at {inicios[j] / fs / 60:.1f} min")

    plt.show()

    # Inspect the same transient in both EEG channels and EOG.
    psg = next(PASTA_TREINO.rglob("SC4351F0-PSG.edf"))
    hip = next(psg.parent.glob("SC4351*-Hypnogram.edf"))

    canais = ["EEG Fpz-Cz", "EEG Pz-Oz", "EOG horizontal"]
    raw = mne.io.read_raw_edf(
        psg, include=canais, preload=False, verbose="ERROR"
    )
    raw.set_annotations(mne.read_annotations(_local_copy(hip, "annotations")), emit_warning=False)

    fs = raw.info["sfreq"]
    centro = 84 * 60 + 67
    inicio = round((centro - 8) * fs)
    fim = round((centro + 8) * fs)

    canais = [c for c in canais if c in raw.ch_names]
    sinais = raw.get_data(picks=canais, start=inicio, stop=fim) * 1e6
    pico = np.argmax(sinais[0])
    tempo = (np.arange(sinais.shape[1]) - pico) / fs

    eventos, ids = mne.events_from_annotations(
        raw, event_id=ETAPAS, chunk_duration=30.0, verbose="ERROR"
    )
    amostra_pico = raw.first_samp + inicio + pico
    no_pico = eventos[
        (eventos[:, 0] <= amostra_pico)
        & (amostra_pico < eventos[:, 0] + round(30 * fs))
    ]
    nomes_etapas = {v: k for k, v in ids.items()}
    etapa = nomes_etapas[no_pico[0, 2]] if len(no_pico) else "no valid stage"
    raw.close()

    fig, eixos = plt.subplots(
        len(canais), 2, figsize=(11, 2.4 * len(canais)),
        sharex="col", squeeze=False, layout="constrained"
    )
    for i, canal in enumerate(canais):
        for ax in eixos[i]:
            ax.plot(tempo, sinais[i], lw=0.8)
            ax.axvline(0, color="tab:orange", ls="--", lw=1)
            ax.set_ylabel(f"{canal}\n(µV)")
        eixos[i, 1].set_xlim(-0.8, 0.8)

    eixos[0, 0].set_title("Window around the peak")
    eixos[0, 1].set_title("Peak close-up")
    for ax in eixos[-1]:
        ax.set_xlabel("Time relative to peak (s)")

    print(f"SC4351F0 | peak at {(inicio + pico) / fs / 60:.3f} min | {etapa}")
    plt.show()


In [ ]:
if RUN_DIAGNOSTICO_SEL40_68:
    from sklearn.metrics import (
        accuracy_score, balanced_accuracy_score,
        cohen_kappa_score, f1_score,
    )

    # Restore predictions and verify alignment.
    _relatorios_diag_sel40 = {}
    _previsoes_diag_sel40 = {}

    for modelo, nome in {
        "F2": "F2_68_fixed_AE",
        "LP40": "LP40_68_fixed_AE",
        "SEL40": "SEL40_68_fixed_AE",
    }.items():
        guardado = _read_json(EXPERIMENTS_DIR / f"{nome}.json")

        assert guardado["experiment"] == nome
        assert guardado["fold_assignment"] == FOLDS_68

        np.testing.assert_array_equal(
            guardado["report"]["y_true"], y_b0_68
        )
        np.testing.assert_array_equal(
            guardado["report"]["y_group"], groups_b0_68
        )

        pred = np.asarray(guardado["report"]["y_pred"])
        assert pred.shape == y_b0_68.shape

        _previsoes_diag_sel40[modelo] = pred
        _relatorios_diag_sel40[modelo] = guardado

    np.testing.assert_array_equal(
        SEL40_MASK_68, FRACAO_ALTA_68 >= 10.0
    )

    # Compare the three models on the same epochs.
    _linhas_subconjuntos_sel40 = []

    for grupo, mascara in [
        ("Cumpre critério >=10%", SEL40_MASK_68),
        ("Restantes épocas", ~SEL40_MASK_68),
    ]:
        yt = y_b0_68[mascara]
        assert len(yt) > 0

        for modelo, pred in _previsoes_diag_sel40.items():
            yp = pred[mascara]

            _linhas_subconjuntos_sel40.append({
                "Grupo": grupo,
                "Modelo": modelo,
                "Épocas": len(yt),
                "Kappa": (
                    cohen_kappa_score(yt, yp)
                    if len(np.unique(yt)) > 1 else np.nan
                ),
                "Macro-F1": f1_score(
                    yt, yp, labels=ESTAGIOS_68,
                    average="macro", zero_division=0,
                ),
                "Accuracy": accuracy_score(yt, yp),
                "Balanced accuracy": balanced_accuracy_score(yt, yp),
            })

    diagnostico_subconjuntos_sel40_68 = pd.DataFrame(
        _linhas_subconjuntos_sel40
    ).set_index(["Grupo", "Modelo"])

    print("COMPARE MODELS WITHIN EACH GROUP")
    _display_en(diagnostico_subconjuntos_sel40_68.round(3))

    # Review the ten recordings with the highest selected percentage.
    _top_registos_sel40 = (
        resumo_sel40_gravacoes_68
        .sort_values(
            ["Com LP40 (%)", "Gravação"],
            ascending=[False, True],
        )
        .head(10).index
    )

    _linhas_registos_sel40 = []

    for record in _top_registos_sel40:
        mascara = record_ids_b0_68 == record
        yt = y_b0_68[mascara]

        linha = {
            "Gravação": record,
            "Com LP40 (%)": float(
                resumo_sel40_gravacoes_68.loc[record, "Com LP40 (%)"]
            ),
        }

        for modelo, pred in _previsoes_diag_sel40.items():
            linha[modelo] = (
                cohen_kappa_score(yt, pred[mascara])
                if len(np.unique(yt)) > 1 else np.nan
            )

        linha["SEL40 - F2"] = linha["SEL40"] - linha["F2"]
        linha["SEL40 - LP40"] = linha["SEL40"] - linha["LP40"]

        _linhas_registos_sel40.append(linha)

    diagnostico_registos_sel40_68 = pd.DataFrame(
        _linhas_registos_sel40
    ).set_index("Gravação")

    print("\nTEN RECORDINGS WITH THE HIGHEST SELECTED PERCENTAGE")
    _display_en(diagnostico_registos_sel40_68.round(3))


## 5 — Earlier experiments: recipes and source

**Start with the results in section 2.** The archive below explains how earlier experiments were prepared and gives their original code in numbered text blocks. It does not run during setup. The archived snippets may rely on variables from older notebook versions; use sections 1–4 for the current workflow.

| Section | What it contains |
|---|---|
| **5.1** | Eight-participant LOSO: B0, F1, F2, C1 and P1 |
| **5.2** | B0 on the 68-participant folds |
| **5.3** | F2 relative powers |
| **5.4** | EEG time and spectrum inspection |
| **5.5** | LP40 on all EEG epochs |
| **5.6** | C1 training-only Wake crop |
| **5.7** | LR1 logistic regression |


### 5.1 — Eight-participant exploration

The eight subjects are SC400–SC407, with 16 recordings and 44,303 test epochs. Each participant is tested once in leave-one-subject-out validation. F2 became the exploratory reference. The Wake crop improved N1 recall at the cost of lower kappa, and the 35 Hz LPF did not improve the overall result. The complete score table is in 2.1.


**Recorded pilot confusion matrices** — the values below are from the eight-participant LOSO runs and are rounded to whole percentages.

**F2 · eight participants**

| Reference stage | Pred. W | Pred. N1 | Pred. N2 | Pred. N3 | Pred. REM |
|---|---:|---:|---:|---:|---:|
| W | **98%** | 0% | 1% | 0% | 0% |
| N1 | 40% | **27%** | 15% | 0% | 17% |
| N2 | 3% | 2% | **88%** | 2% | 4% |
| N3 | 10% | 0% | 14% | **76%** | 0% |
| REM | 5% | 5% | 8% | 0% | **82%** |

**P1 · eight participants**

| Reference stage | Pred. W | Pred. N1 | Pred. N2 | Pred. N3 | Pred. REM |
|---|---:|---:|---:|---:|---:|
| W | **98%** | 0% | 1% | 0% | 0% |
| N1 | 42% | **25%** | 16% | 0% | 17% |
| N2 | 4% | 2% | **88%** | 2% | 5% |
| N3 | 10% | 0% | 13% | **76%** | 0% |
| REM | 5% | 5% | 8% | 0% | **81%** |


**Original cell 10**

```python
if RUN_8_EXPERIMENTS:
    DEVELOPMENT_SUBJECTS = list(range(8))
```

**Original cell 11**

```python
if RUN_8_EXPERIMENTS:
    import tempfile

    track = SleepEDFTrack(cfg=BASELINE_CONFIG)
    requested_groups = {f"SC4{int(s):02d}" for s in DEVELOPMENT_SUBJECTS}

    track.download(str(TRAIN_DIR), subset=DEVELOPMENT_SUBJECTS)

    if CACHE_8_FILE.is_file():
        # This cache contains only SC400–SC407 even if there are more EDF files in the folder.
        recs = track.load_cached(str(TRAIN_DIR), cache_file=str(CACHE_8_FILE))
    else:
        # The temporary folder contains links to the original files.
        # The temporary folder is deleted after reading. Files on Drive stay in place.
        selected_files = sorted(
            p for p in TRAIN_DIR.rglob("*.edf")
            if p.name[:5] in requested_groups
        )
        with tempfile.TemporaryDirectory(prefix="viggy_8_") as tmp:
            for p in selected_files:
                link = Path(tmp) / p.name
                if link.exists():
                    raise ValueError(f"Nome de EDF repetido: {p.name}")
                link.symlink_to(p.resolve())
            recs = track.load(tmp)

    track.assert_no_holdout(recs)
    actual_groups = {r.group for r in recs}
    if actual_groups != requested_groups or len(recs) != 16:
        raise ValueError("The pilot must contain only SC400–SC407, with 16 recordings.")

    print("recordings:", len(recs), "| subjects:", sorted(actual_groups))
```

**Original cell 13**

```python
if RUN_8_EXPERIMENTS:
    X8, y8, groups8 = track.build_dataset(recs)
    rep8 = track.evaluate(X8, y8, groups8)

    print(f"Épocas: {len(y8)} | Features: {X8.shape[1]}")
    print(f"Split: {rep8['split_unit']} | Participantes: {rep8['n_groups']}")
    print("Variation across participants:", rep8["summary"])

    print(f"\nKappa global: {rep8['cohens_kappa']:.3f}")
    print(f"Macro-F1: {rep8['macro_f1']:.3f}")
    print(f"Accuracy: {rep8['accuracy']:.3f}")
```

**Original cell 15**

```python
if RUN_8_EXPERIMENTS:
    import numpy as np

    # Find the five EEG power columns.
    nomes = track.feature_names()
    bandas = ["delta", "theta", "alpha", "sigma", "beta"]
    idx = [nomes.index(f"eeg_{b}") for b in bandas]

    potencias = X8[:, idx]
    if not np.isfinite(potencias).all() or (potencias < 0).any():
        raise ValueError("Check invalid powers before continuing.")

    # Calculate the proportions separately for each epoch.
    total = potencias.sum(axis=1, keepdims=True)
    X8_rel = X8.copy()
    X8_rel[:, idx] = np.divide(
        potencias, total,
        out=np.zeros_like(potencias),
        where=total > 0
    )

    # Evaluate again on the same eight participants.
    rep8_rel = track.evaluate(X8_rel, y8, groups8)

    print("Baseline:", rep8["summary"])
    print("Relativas:", rep8_rel["summary"])
    print(f"\nKappa global: {rep8_rel['cohens_kappa']:.3f}")
    print(f"Macro-F1: {rep8_rel['macro_f1']:.3f}")
    print(f"Accuracy: {rep8_rel['accuracy']:.3f}")
```

**Original cell 16**

```python
if RUN_8_EXPERIMENTS:
    import pandas as pd
    from IPython.display import display
    from sklearn.metrics import classification_report

    # Compare each participant's kappa.
    b0 = pd.DataFrame(rep8["per_group"]).set_index("group")
    f1 = pd.DataFrame(rep8_rel["per_group"]).set_index("group")

    comparacao = pd.DataFrame({
        "Baseline": b0["cohens_kappa"],
        "Relativas": f1["cohens_kappa"]
    }).sort_index()

    comparacao["Diferença"] = (
        comparacao["Relativas"] - comparacao["Baseline"]
    )
    display(comparacao.round(3))

    # Precision, recall, F1 and epoch count by stage.
    for nome, resultado in [("BASELINE", rep8), ("RELATIVAS", rep8_rel)]:
        print(f"\n{nome}")
        print(classification_report(
            resultado["y_true"],
            resultado["y_pred"],
            labels=["W", "N1", "N2", "N3", "REM"],
            digits=3,
            zero_division=0
        ))
```

**Original cell 18**

```python
if RUN_8_EXPERIMENTS:
    # Append only the five relative powers.
    X8_comb = np.column_stack([X8, X8_rel[:, idx]])

    nomes_comb = track.feature_names() + [
        f"eeg_{b}_relative" for b in bandas
    ]

    print("Features:", X8_comb.shape[1])

    # Use the same model, participants and evaluation.
    rep8_comb = track.evaluate(X8_comb, y8, groups8)

    print("\nBaseline:", rep8["summary"])
    print("Relativas:", rep8_rel["summary"])
    print("Combinadas:", rep8_comb["summary"])

    print(f"\nKappa global: {rep8_comb['cohens_kappa']:.3f}")
    print(f"Macro-F1: {rep8_comb['macro_f1']:.3f}")
    print(f"Accuracy: {rep8_comb['accuracy']:.3f}")
```

**Original cell 19**

```python
if RUN_8_EXPERIMENTS:
    from sklearn.metrics import classification_report, ConfusionMatrixDisplay

    # Kappa by participant.
    k_rel = {r["group"]: r["cohens_kappa"] for r in rep8_rel["per_group"]}
    k_comb = {r["group"]: r["cohens_kappa"] for r in rep8_comb["per_group"]}

    tabela = pd.DataFrame({
        "Relativas": k_rel,
        "Combinadas": k_comb
    }).sort_index()

    tabela["Diferença"] = tabela["Combinadas"] - tabela["Relativas"]
    display(tabela.round(3))

    # Results by stage.
    ordem = ["W", "N1", "N2", "N3", "REM"]

    print(classification_report(
        rep8_comb["y_true"], rep8_comb["y_pred"],
        labels=ordem, digits=3, zero_division=0
    ))

    # Confusion matrix: each row sums to one.
    fig, ax = plt.subplots(figsize=(6, 5))
    ConfusionMatrixDisplay.from_predictions(
        rep8_comb["y_true"], rep8_comb["y_pred"],
        labels=ordem, normalize="true",
        values_format=".2f", cmap="Blues", colorbar=False, ax=ax
    )
    ax.set(title="Combinadas", xlabel="Previsto", ylabel="Real")
    plt.tight_layout()
    plt.show()
```

**Original cell 21**

```python
if RUN_8_EXPERIMENTS and SAVE_8_REPORTS:
    import json
    from pathlib import Path

    arquivo = Path(DATA) / "experiments" / "features_B0_F1_F2.json"
    arquivo.parent.mkdir(parents=True, exist_ok=True)

    conteudo = {
        "course_revision": COURSE_REVISION,
        "runtime": runtime_versions,
        "baseline_config": BASELINE_CONFIG,
        "recordings": [
            {
                "record": r.meta["record"],
                "subject": r.group,
                "epochs": len(r.labels)
            }
            for r in recs
        ],
        "reports": {
            "B0_absolute": rep8,
            "F1_relative": rep8_rel,
            "F2_combined": rep8_comb
        }
    }

    arquivo.write_text(
        json.dumps(conteudo, default=converter_numpy, indent=2),
        encoding="utf-8"
    )
    print("Saved:", arquivo)
```

**Original cell 23**

```python
if RUN_8_EXPERIMENTS:
    import pandas as pd
    from IPython.display import display

    ordem = ["W", "N1", "N2", "N3", "REM"]

    contagens = pd.crosstab(
        pd.Series(groups8, name="Participante"),
        pd.Series(y8, name="Estágio")
    ).reindex(columns=ordem, fill_value=0)

    percentagens = (
        contagens.div(contagens.sum(axis=1), axis=0) * 100
    )

    print("Stage percentage by participant:")
    display(percentagens.round(1))
```


**Original cell 24**

```python
if RUN_8_EXPERIMENTS:
    linhas = []

    for rec in recs:
        etapas = np.asarray(rec.labels)
        sono = np.flatnonzero(
            np.isin(etapas, ["N1", "N2", "N3", "REM"])
        )

        if sono.size == 0:
            raise ValueError(f"{rec.meta['record']}: no sleep epochs.")

        primeira, ultima = sono[0], sono[-1]
        wake = etapas == "W"

        # Epoch duration in minutes.
        minutos = rec.epochs["eeg"].shape[1] / rec.fs / 60

        linhas.append({
            "Gravação": rec.meta["record"],
            "W total (min)": wake.sum() * minutos,
            "W início (min)": wake[:primeira].sum() * minutos,
            "W fim (min)": wake[ultima + 1:].sum() * minutos,
            "W entre sono (min)": wake[primeira:ultima + 1].sum() * minutos,
        })

    display(pd.DataFrame(linhas).round(1))
```

**Original cell 25**

```python
if RUN_8_EXPERIMENTS:
    from sklearn.model_selection import LeaveOneGroupOut
    from sklearn.metrics import classification_report

    MARGEM_MIN = 30

    # Find the feature rows for each recording.
    limites = np.cumsum([0] + [len(r.labels) for r in recs])

    # Check that recordings, features and labels are aligned.
    assert X8_comb.shape[0] == len(y8) == limites[-1]
    np.testing.assert_array_equal(
        y8, np.concatenate([r.labels for r in recs])
    )
    np.testing.assert_array_equal(
        groups8,
        np.repeat([r.group for r in recs], [len(r.labels) for r in recs])
    )

    previsoes = np.empty_like(y8)
    numero_fold = np.empty(len(y8), dtype=int)

    for fold, (treino, teste) in enumerate(
        LeaveOneGroupOut().split(X8_comb, y8, groups8)
    ):
        participantes_treino = set(groups8[treino])
        manter = np.zeros(len(y8), dtype=bool)

        for rec, offset in zip(recs, limites[:-1]):
            if rec.group not in participantes_treino:
                continue

            # Use training participants' labels only.
            sono = np.flatnonzero(
                np.isin(rec.labels, ["N1", "N2", "N3", "REM"])
            )
            minutos_por_epoca = rec.epochs["eeg"].shape[1] / rec.fs / 60
            margem = int(round(MARGEM_MIN / minutos_por_epoca))

            inicio = max(0, int(sono[0]) - margem)
            fim = min(len(rec.labels), int(sono[-1]) + margem + 1)
            manter[offset + inicio:offset + fim] = True

        fit = treino[manter[treino]]

        # Fit a fresh scaler and Random Forest on this training set only.
        modelo = track.baseline()
        modelo.fit(X8_comb[treino], y8[treino])

        # Keep all epochs for the test participant.
        previsoes[teste] = modelo.predict(X8_comb[teste])
        numero_fold[teste] = fold

        print(f"Fold {fold + 1}/8 finished: {len(treino)} training epochs")

    rep8_crop30 = track._make_report(
        y8, previsoes, groups=groups8, folds=numero_fold
    )

    print("\nF2 — full training:", rep8_comb["summary"])
    print("C1 — cropped training:", rep8_crop30["summary"])

    for metrica in ["cohens_kappa", "macro_f1", "accuracy", "balanced_accuracy"]:
        print(f"{metrica}: {rep8_crop30[metrica]:.3f}")

    print(classification_report(
        rep8_crop30["y_true"], rep8_crop30["y_pred"],
        labels=["W", "N1", "N2", "N3", "REM"],
        digits=3, zero_division=0
    ))
```

**Original cell 27**

```python
if RUN_8_EXPERIMENTS and SAVE_8_REPORTS:
    import json
    from pathlib import Path

    arquivo_c1 = Path(DATA) / "experiments" / "C1_crop30.json"
    arquivo_c1.parent.mkdir(parents=True, exist_ok=True)

    arquivo_c1.write_text(
        json.dumps({
            "course_revision": COURSE_REVISION,
            "runtime": runtime_versions,
            "baseline_config": BASELINE_CONFIG,
            "feature_names": nomes_comb,
            "margin_minutes": MARGEM_MIN,
            "crop_scope": "training_only",
            "evaluation_scope": "all_original_epochs",
            "report": rep8_crop30,
        }, default=converter_numpy, indent=2),
        encoding="utf-8"
    )
    print("Saved:", arquivo_c1)
```

**Original cell 28**

```python
if RUN_8_EXPERIMENTS:
    k_f2 = pd.DataFrame(rep8_comb["per_group"]).set_index("group")
    k_c1 = pd.DataFrame(rep8_crop30["per_group"]).set_index("group")

    comparacao_c1 = pd.DataFrame({
        "F2": k_f2["cohens_kappa"],
        "C1": k_c1["cohens_kappa"],
    }).sort_index()

    comparacao_c1["Diferença"] = comparacao_c1["C1"] - comparacao_c1["F2"]
    display(comparacao_c1.round(3))
```

**Original cell 30**

```python
if RUN_8_EXPERIMENTS:
    qc_linhas = []
    qc_amplitudes = {}

    for rec in recs:
        eeg = np.asarray(rec.epochs["eeg"])
        validos = np.isfinite(eeg).all(axis=1)

        # Peak-to-peak amplitude per epoch, in microvolts.
        amplitudes = np.full(len(eeg), np.nan)
        amplitudes[validos] = np.ptp(eeg[validos], axis=1) * 1e6

        registo = rec.meta["record"]
        qc_amplitudes[registo] = amplitudes

        if validos.any():
            p50, p95 = np.percentile(amplitudes[validos], [50, 95])
            maximo = amplitudes[validos].max()
        else:
            p50 = p95 = maximo = np.nan

        qc_linhas.append({
            "Gravação": registo,
            "Épocas": len(eeg),
            "NaN/Inf": int((~validos).sum()),
            "Constantes": int((amplitudes == 0).sum()),
            "P50 (µV)": p50,
            "P95 (µV)": p95,
            "Máx (µV)": maximo,
        })

    qc_eeg = pd.DataFrame(qc_linhas)
    display(qc_eeg.round(1))
```

**Original cell 31**

```python
if RUN_8_EXPERIMENTS:
    from scipy.signal import welch

    alvo = "SC4061E0"
    rec = next(r for r in recs if r.meta["record"] == alvo)

    amp = qc_amplitudes[alvo]
    indices_validos = np.flatnonzero(np.isfinite(amp))

    i_med = indices_validos[
        np.argmin(np.abs(amp[indices_validos] - np.median(amp[indices_validos])))
    ]
    i_max = indices_validos[np.argmax(amp[indices_validos])]

    exemplos = [("Mediana", i_med), ("Máxima", i_max)]
    fig, axes = plt.subplots(2, 3, figsize=(15, 7))

    for linha, (nome, i) in enumerate(exemplos):
        x = np.asarray(rec.epochs["eeg"][i]) * 1e6
        fs = float(rec.fs)
        t = np.arange(len(x)) / fs

        # Full epoch.
        axes[linha, 0].plot(t, x, lw=0.7)
        axes[linha, 0].set(
            title=f"{nome}: epoch {i} ({rec.labels[i]})\n"
                  f"Pico a pico = {amp[i]:.1f} µV",
            xlabel="Tempo (s)", ylabel="EEG (µV)"
        )

        # Two-second close-up around the largest excursion.
        pico = np.argmax(np.abs(x - np.median(x)))
        n_zoom = int(round(2 * fs))
        inicio = int(np.clip(pico - n_zoom // 2, 0, len(x) - n_zoom))
        fim = inicio + n_zoom

        axes[linha, 1].plot(t[inicio:fim], x[inicio:fim], ".-", ms=2, lw=0.7)
        axes[linha, 1].set(
            title="Detalhe de 2 segundos",
            xlabel="Tempo (s)", ylabel="EEG (µV)"
        )
        axes[linha, 1].set_ylim(axes[linha, 0].get_ylim())

        # Diagnostic spectrum: eight-second window.
        nperseg = min(len(x), int(round(8 * fs)))
        f, psd = welch(
            x, fs=fs, window="hann",
            nperseg=nperseg, noverlap=nperseg // 2,
            detrend="constant", scaling="density"
        )
        mostrar = (f > 0) & (psd > 0)
        axes[linha, 2].semilogy(f[mostrar], psd[mostrar])
        axes[linha, 2].set(
            title="PSD Welch — janela de 8 s",
            xlabel="Frequency (Hz)", ylabel="PSD (µV²/Hz)",
            xlim=(0, fs / 2)
        )

        for ax in axes[linha]:
            ax.grid(alpha=0.2)

        print(
            f"{nome}: epoch {i}, stage {rec.labels[i]}, "
            f"mínimo {x.min():.3f} µV, maximum {x.max():.3f} µV"
        )

    fig.suptitle(f"{alvo} — EEG Fpz-Cz, sem filtragem adicional")
    fig.tight_layout(rect=(0, 0, 1, 0.94))
    plt.show()
```


**Original cell 32**

```python
if RUN_8_EXPERIMENTS:
    from pathlib import Path

    alvo = "SC4061E0"
    rec = next(r for r in recs if r.meta["record"] == alvo)
    ficheiro = (
        Path(DATA) / "train" / "physionet-sleep-data"
        / f"{alvo}-PSG.edf"
    )

    # Read only the EDF header.
    with ficheiro.open("rb") as f_edf:
        f_edf.seek(252)
        n_canais = int(f_edf.read(4))

        def campo(largura):
            bloco = f_edf.read(n_canais * largura).decode("ascii")
            return [
                bloco[i * largura:(i + 1) * largura].strip()
                for i in range(n_canais)
            ]

        canais = campo(16)
        campo(80)  # Transducer information.
        unidades = campo(8)
        fis_min = np.array(campo(8), dtype=float)
        fis_max = np.array(campo(8), dtype=float)
        dig_min = np.array(campo(8), dtype=float)
        dig_max = np.array(campo(8), dtype=float)
        pre_filtros = campo(80)

    j = canais.index("EEG Fpz-Cz")
    unidade = unidades[j].replace("µ", "u").replace("μ", "u").lower()
    fator = {"uv": 1e-6, "mv": 1e-3, "v": 1.0}[unidade]

    # Convert the limits to volts to match the loaded data.
    lim_inf = fis_min[j] * fator
    lim_sup = fis_max[j] * fator
    passo = (lim_sup - lim_inf) / (dig_max[j] - dig_min[j])

    eeg = np.asarray(rec.epochs["eeg"])
    nos_limites = (
        np.isclose(eeg, lim_inf, rtol=0, atol=passo / 2)
        | np.isclose(eeg, lim_sup, rtol=0, atol=passo / 2)
    )
    epocas_limite = nos_limites.any(axis=1)

    print("Limites EDF:", fis_min[j], "a", fis_max[j], unidades[j])
    print("Prefilters reported in the EDF header:", pre_filtros[j])

    resumo_limites = []
    for etapa in ["W", "N1", "N2", "N3", "REM"]:
        mascara = np.asarray(rec.labels) == etapa
        if not mascara.any():
            continue

        resumo_limites.append({
            "Estágio": etapa,
            "Épocas": int(mascara.sum()),
            "Épocas com algum limite (%)": 100 * epocas_limite[mascara].mean(),
            "Amostras nos limites (%)": 100 * nos_limites[mascara].mean(),
        })

    display(pd.DataFrame(resumo_limites).round(3))
```

**Original cell 34**

```python
if RUN_8_EXPERIMENTS:
    from scipy.signal import butter, sosfiltfilt, welch

    PREPROCESS_VERSION = 1
    PREPROCESS_SETTINGS = {"cutoff_hz": 35.0, "order": 4}

    def eeg_lowpass(eeg, fs, cutoff_hz=35.0, order=4):
        sos = butter(
            order, cutoff_hz, btype="lowpass", fs=fs, output="sos"
        )
        return sosfiltfilt(sos, eeg, axis=-1, padtype="odd")

    rec = next(r for r in recs if r.meta["record"] == "SC4061E0")
    indices_n3 = np.flatnonzero(np.asarray(rec.labels) == "N3")
    exemplos_filtro = [4, int(indices_n3[len(indices_n3) // 2])]

    fig, axes = plt.subplots(2, 2, figsize=(12, 7))

    for linha, i in enumerate(exemplos_filtro):
        original = np.asarray(rec.epochs["eeg"][i])
        filtrado = eeg_lowpass(original, rec.fs, **PREPROCESS_SETTINGS)

        fs = float(rec.fs)
        t = np.arange(len(original)) / fs

        # Select a four-second detail from the original signal.
        pico = np.argmax(np.abs(original - np.median(original)))
        n_zoom = int(round(4 * fs))
        inicio = int(np.clip(pico - n_zoom // 2, 0, len(original) - n_zoom))
        fim = inicio + n_zoom

        for sinal, nome in [(original, "Original"), (filtrado, "Passa-baixo")]:
            x = sinal * 1e6

            axes[linha, 0].plot(
                t[inicio:fim], x[inicio:fim],
                label=nome, lw=0.8, alpha=0.8
            )

            nperseg = min(len(x), int(round(8 * fs)))
            f, psd = welch(
                x, fs=fs, window="hann",
                nperseg=nperseg, noverlap=nperseg // 2,
                detrend="constant", scaling="density"
            )
            mostrar = (f > 0) & (psd > 0)
            axes[linha, 1].semilogy(f[mostrar], psd[mostrar], label=nome)

        axes[linha, 0].set(
            title=f"{rec.labels[i]} — epoch {i}, detalhe de 4 s",
            xlabel="Tempo (s)", ylabel="EEG (µV)"
        )
        axes[linha, 1].set(
            title="PSD before and after",
            xlabel="Frequency (Hz)", ylabel="PSD (µV²/Hz)",
            xlim=(0, fs / 2)
        )
        axes[linha, 1].axvline(35, color="grey", ls="--", lw=0.8)

        for ax in axes[linha]:
            ax.legend()
            ax.grid(alpha=0.2)

    fig.suptitle("SC4061E0 — efeito do passa-baixo")
    fig.tight_layout(rect=(0, 0, 1, 0.94))
    plt.show()
```

**Original cell 35**

```python
if RUN_8_EXPERIMENTS:
    from adapter import cached
    import hashlib
    from pathlib import Path

    FEATURE_VERSION = 1
    CACHE_VIGGY = str(Path(DATA) / "cache" / "viggy")

    nomes_base = track.feature_names()
    bandas_viggy = ["delta", "theta", "alpha", "sigma", "beta"]
    idx_viggy = [nomes_base.index(f"eeg_{b}") for b in bandas_viggy]
    nomes_viggy = nomes_base + [f"eeg_{b}_relative" for b in bandas_viggy]

    def extrair_features_viggy(rec):
        X11, _, _ = track.extract_features(rec, cfg=BASELINE_CONFIG)
        potencias = X11[:, idx_viggy]

        if not np.isfinite(potencias).all() or (potencias < 0).any():
            raise ValueError("Invalid band powers.")

        total = potencias.sum(axis=1, keepdims=True)
        relativas = np.divide(
            potencias, total,
            out=np.zeros_like(potencias),
            where=total > 0
        )
        return np.column_stack([X11, relativas])

    # Check the F2 feature construction on the first recording.
    n0 = len(recs[0].labels)
    np.testing.assert_allclose(
        extrair_features_viggy(recs[0]),
        X8_comb[:n0],
        rtol=1e-12, atol=0
    )
    assert nomes_viggy == nomes_comb

    def features_p1_com_cache(rec):
        # Identify the input data without using stage labels.
        h = hashlib.sha256()
        for canal in ["eeg", "eog", "emg"]:
            a = np.ascontiguousarray(rec.epochs[canal])
            h.update(f"{canal}:{a.shape}:{a.dtype}".encode())
            h.update(memoryview(a).cast("B"))

        origem = {
            "record": rec.meta["record"],
            "fs": float(rec.fs),
            "sha256": h.hexdigest(),
        }
        chave_pre = {
            "version": PREPROCESS_VERSION,
            "settings": dict(PREPROCESS_SETTINGS),
            "input": origem,
            "runtime": {k: runtime_versions[k] for k in ["numpy", "scipy"]},
        }

        cfg = track._cfg(BASELINE_CONFIG)
        chave_feat = {
            "version": FEATURE_VERSION,
            "input": chave_pre,
            "course_revision": COURSE_REVISION,
            "settings": {
                k: cfg[k] for k in track.feature_cfg_keys() if k in cfg
            },
            "names": nomes_viggy,
        }

        def calcular_features():
            eeg_filtrado = cached(
                "viggy_preprocess",
                chave_pre,
                lambda: eeg_lowpass(
                    rec.epochs["eeg"], rec.fs, **PREPROCESS_SETTINGS
                ),
                CACHE_VIGGY
            )
            rec_filtrado = rec.replace_epochs(
                dict(rec.epochs, eeg=eeg_filtrado)
            )
            return extrair_features_viggy(rec_filtrado)

        X16 = cached(
            "viggy_features", chave_feat, calcular_features, CACHE_VIGGY
        )
        assert X16.shape == (len(rec.labels), len(nomes_viggy))
        return X16

    X8_p1 = np.vstack([features_p1_com_cache(r) for r in recs])

    # Check that examples, labels, groups and auxiliary signals match.
    np.testing.assert_array_equal(
        np.concatenate([r.labels for r in recs]), y8
    )
    np.testing.assert_array_equal(
        np.repeat([r.group for r in recs], [len(r.labels) for r in recs]),
        groups8
    )
    assert X8_p1.shape == X8_comb.shape

    aux = [nomes_base.index("eog_movement"), nomes_base.index("emg_rms")]
    np.testing.assert_allclose(
        X8_p1[:, aux], X8_comb[:, aux], rtol=1e-12, atol=0
    )

    print("F2 feature extraction checked on the first recording.")
    print(
        f"P1: {len(y8)} epochs | {X8_p1.shape[1]} features | "
        f"{len(np.unique(groups8))} participantes"
    )
    print("Verified labels, groups, EOG and EMG.")
```


**Original cell 36**

```python
if RUN_8_EXPERIMENTS:
    from sklearn.metrics import classification_report

    assert np.isfinite(X8_p1).all()

    rep8_p1 = track.evaluate(
        X8_p1, y8, groups8, cfg=BASELINE_CONFIG
    )

    print("F2 — sem LPF adicional:", rep8_comb["summary"])
    print("P1 — LPF de 35 Hz:", rep8_p1["summary"])

    print("\nPooled P1 metrics:")
    for metrica in [
        "cohens_kappa", "macro_f1", "accuracy", "balanced_accuracy"
    ]:
        print(f"{metrica}: {rep8_p1[metrica]:.3f}")

    print("\nResults by stage:")
    print(classification_report(
        rep8_p1["y_true"],
        rep8_p1["y_pred"],
        labels=["W", "N1", "N2", "N3", "REM"],
        digits=3,
        zero_division=0
    ))
```

**Original cell 38**

```python
if RUN_8_EXPERIMENTS and SAVE_8_REPORTS:
    import json
    import pandas as pd
    from pathlib import Path

    registo_p1 = {
        "experiment": "P1_lowpass35",
        "course_revision": COURSE_REVISION,
        "runtime": runtime_versions,
        "baseline_config": BASELINE_CONFIG,
        "preprocessing": {
            "version": PREPROCESS_VERSION,
            "settings": dict(PREPROCESS_SETTINGS),
            "method": "Butterworth SOS; EEG only; forward-backward; per epoch; odd padding"
        },
        "feature_version": FEATURE_VERSION,
        "feature_names": nomes_viggy,
        "evaluation_scope": "all_original_epochs",
        "training_crop": False,
        "records": [
            {"record": r.meta["record"],
             "subject": r.group,
             "epochs": len(r.labels)}
            for r in recs
        ],
        "report": rep8_p1
    }

    destino = Path(DATA) / "experiments" / "P1_lowpass35.json"
    destino.parent.mkdir(parents=True, exist_ok=True)

    with destino.open("w", encoding="utf-8") as f:
        json.dump(registo_p1, f, default=converter_numpy,
                  ensure_ascii=False, indent=2)

    print("Saved:", destino)
```

**Original cell 39**

```python
if RUN_8_EXPERIMENTS:
    tabela = pd.DataFrame({
        nome: {r["group"]: r["cohens_kappa"] for r in rep["per_group"]}
        for nome, rep in {"F2": rep8_comb, "P1": rep8_p1}.items()
    })
    tabela["Diferença"] = tabela["P1"] - tabela["F2"]
    display(tabela.round(3))
```

**Original cell 40**

```python
if RUN_8_EXPERIMENTS:
    import matplotlib.pyplot as plt
    from sklearn.metrics import ConfusionMatrixDisplay

    estagios = ["W", "N1", "N2", "N3", "REM"]
    comparacoes = [
        ("F2 — sem LPF adicional", rep8_comb),
        ("P1 — LPF de 35 Hz", rep8_p1)
    ]

    fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))

    for ax, (nome, rep) in zip(axes, comparacoes):
        ConfusionMatrixDisplay.from_predictions(
            rep["y_true"], rep["y_pred"],
            labels=estagios,
            normalize="true",
            values_format=".2f",
            cmap="Blues",
            colorbar=False,
            im_kw={"vmin": 0, "vmax": 1},
            ax=ax
        )
        ax.set_title(nome)
        ax.set_xlabel("Predicted stage")
        ax.set_ylabel("Reference stage")

    plt.tight_layout()
    plt.show()
```


### 5.2 — B0 on the fixed A–E folds

B0 uses the 11 original per-epoch features and a Random Forest. It was the first common comparison on all 68 development participants and full test recordings.


**Original cell 57**

```python
from pathlib import Path
from tempfile import TemporaryDirectory
from time import perf_counter
from adapter import cached

# Use the original eight-participant baseline.
track_b0_68 = SleepEDFTrack(cfg={})
B0_FEATURE_VERSION = 1

def preparar_b0_gravacao(psg):
    psg = Path(psg).resolve()

    assert psg.name[:5] in DEVELOPMENT_GROUPS_68

    marcacoes = sorted(
        psg.parent.glob(f"{psg.name[:6]}*-Hypnogram.edf")
    )
    assert len(marcacoes) == 1, f"Check annotations for {psg.name}"
    hyp = marcacoes[0]

    chave = {
        "version": B0_FEATURE_VERSION,
        "course_revision": COURSE_REVISION,
        "load": {"fs_target": 100.0, "epoch_s": 30.0},
        "preprocessing": "none",
        "features": "original_11_welch",
        "runtime": {
            nome: runtime_versions[nome]
            for nome in ("python", "numpy", "scipy", "mne")
        },
        "files": [
            (p.name, p.stat().st_size, p.stat().st_mtime_ns)
            for p in (psg, hyp)
        ],
    }

    def calcular():
        # Use temporary links to the existing EDF files.
        with TemporaryDirectory() as temporaria:
            for origem in (psg, hyp):
                (Path(temporaria) / origem.name).symlink_to(
                    origem.resolve()
                )

            gravacoes = track_b0_68.load(
                temporaria, fs_target=100.0, epoch_s=30.0
            )
            assert len(gravacoes) == 1

            X, y, grupos = track_b0_68.build_dataset(gravacoes)

            assert X.shape == (len(y), 11)
            assert np.isfinite(X).all()

            return {
                "record": gravacoes[0].meta["record"],
                "X": X,
                "y": y,
                "groups": grupos,
                "feature_names": track_b0_68.feature_names(),
            }

    return cached(
        "viggy_b0_features",
        chave,
        calcular,
        CACHE_VIGGY,
    )
```

**Original cell 58**

```python
from time import perf_counter

# Select recordings from the 68 development participants.
ficheiros_b0_68 = sorted(
    p for p in TRAIN_DIR.rglob("*-PSG.edf")
    if p.name[:5] in DEVELOPMENT_GROUPS_68
)

assert len(ficheiros_b0_68) == 133
assert len({p.name for p in ficheiros_b0_68}) == 133
assert {p.name[:5] for p in ficheiros_b0_68} == set(
    DEVELOPMENT_GROUPS_68
)

nomes_features_b0 = track_b0_68.feature_names()

# This list stores only features, labels and identifiers.
b0_gravacoes_68 = []

inicio_preparacao_68 = perf_counter()

for numero, psg in enumerate(ficheiros_b0_68, start=1):
    inicio_gravacao = perf_counter()

    dados = preparar_b0_gravacao(psg)

    # Check that each result belongs to the expected recording.
    assert dados["record"] == psg.name.removesuffix("-PSG.edf")
    assert set(dados["groups"]) == {psg.name[:5]}
    assert dados["feature_names"] == nomes_features_b0
    assert dados["X"].shape == (len(dados["y"]), 11)
    assert len(dados["groups"]) == len(dados["y"])

    b0_gravacoes_68.append(dados)

    segundos = perf_counter() - inicio_gravacao
    minutos_total = (
        perf_counter() - inicio_preparacao_68
    ) / 60

    print(
        f"[{numero:03d}/133] {dados['record']} | "
        f"{len(dados['y'])} epochs | "
        f"{segundos:.1f} s | "
        f"total: {minutos_total:.1f} min",
        flush=True,
    )

total_epocas_b0_68 = sum(
    len(dados["y"]) for dados in b0_gravacoes_68
)

print("\nPREPARATION COMPLETE")
print("Participantes:", len(DEVELOPMENT_GROUPS_68))
print("Recordings:", len(b0_gravacoes_68))
print("Total epochs:", total_epocas_b0_68)
print("Features per epoch:", len(nomes_features_b0))
```

**Original cell 59**

```python
import numpy as np
import pandas as pd

# Combine only features and identifiers.
X_b0_68 = np.concatenate([
    dados["X"] for dados in b0_gravacoes_68
])

y_b0_68 = np.concatenate([
    dados["y"] for dados in b0_gravacoes_68
])

groups_b0_68 = np.concatenate([
    dados["groups"] for dados in b0_gravacoes_68
])

# Record the source recording for each epoch.
record_ids_b0_68 = np.concatenate([
    np.repeat(dados["record"], len(dados["y"]))
    for dados in b0_gravacoes_68
])

assert X_b0_68.shape == (360621, 11)
assert len(y_b0_68) == len(groups_b0_68) == len(record_ids_b0_68)
assert set(groups_b0_68) == set(DEVELOPMENT_GROUPS_68)
assert not (set(groups_b0_68) & HOLDOUT_GROUPS)
assert np.isfinite(X_b0_68).all()

# Save the train and test indices defined by lists A–E.
SPLITS_68 = {}
resumo_folds = []
vezes_em_teste = np.zeros(len(y_b0_68), dtype=np.uint8)

for fold, participantes_teste in FOLDS_68.items():
    mascara_teste = np.isin(groups_b0_68, participantes_teste)

    fit = np.flatnonzero(~mascara_teste)
    teste = np.flatnonzero(mascara_teste)

    sujeitos_treino = set(groups_b0_68[treino])
    sujeitos_teste = set(groups_b0_68[teste])

    assert sujeitos_teste == set(participantes_teste)
    assert sujeitos_treino.isdisjoint(sujeitos_teste)

    SPLITS_68[fold] = (treino, teste)
    vezes_em_teste[teste] += 1

    resumo_folds.append({
        "Fold": fold,
        "Participantes treino": len(sujeitos_treino),
        "Participantes teste": len(sujeitos_teste),
        "Épocas treino": len(treino),
        "Épocas teste": len(teste),
    })

# Each epoch must appear in the test set exactly once.
assert np.all(vezes_em_teste == 1)

print("Baseline data:", X_b0_68.shape)
display(pd.DataFrame(resumo_folds).set_index("Fold"))
print("Folds A–E prepared and verified.")
```


**Original cell 61**

```python
from sklearn.base import clone

# Clone the reference model for every fold.
modelo_base_b0 = track_b0_68.baseline()
modelo_base_b0.set_params(clf__n_jobs=-1)

# Metadata shared across the five folds.
meta_b0_68 = {
    "experiment": "B0_68_fixed_AE",
    "course_revision": COURSE_REVISION,
    "runtime": runtime_versions,
    "feature_version": B0_FEATURE_VERSION,
    "feature_names": nomes_features_b0,
    "fold_assignment": FOLDS_68,
    "recordings": [
        {"record": d["record"], "epochs": len(d["y"])}
        for d in b0_gravacoes_68
    ],
    "classifier_params": modelo_base_b0.named_steps["clf"].get_params(),
    "scaler_params": modelo_base_b0.named_steps["scale"].get_params(),
}

resultados_b0_68 = {}

for fold in "ABCDE":
    treino, teste = SPLITS_68[fold]
    destino = EXPERIMENTS_DIR / f"B0_68_fixed_AE_fold_{fold}.json"

    if destino.exists():
        resultado = json.loads(destino.read_text(encoding="utf-8"))

        # Reuse results only when data and configuration match.
        assert resultado["fold"] == fold
        for campo, valor in meta_b0_68.items():
            assert resultado[campo] == valor, (
                f"Fold {fold}: different configuration in {campo}."
            )

        np.testing.assert_array_equal(
            resultado["test_indices"], teste
        )
        np.testing.assert_array_equal(
            resultado["report"]["y_true"], y_b0_68[teste]
        )
        np.testing.assert_array_equal(
            resultado["report"]["y_group"], groups_b0_68[teste]
        )

        print(f"Fold {fold}: reused the saved result.",
              flush=True)

    else:
        print(f"\nFold {fold}: fitting...", flush=True)

        modelo = clone(modelo_base_b0)

        inicio = perf_counter()
        modelo.fit(X_b0_68[treino], y_b0_68[treino])
        tempo_treino = perf_counter() - inicio

        inicio = perf_counter()
        previsoes = modelo.predict(X_b0_68[teste])
        tempo_previsao = perf_counter() - inicio

        relatorio = track_b0_68._make_report(
            y_b0_68[teste],
            previsoes,
            groups=groups_b0_68[teste],
        )

        resultado = {
            **meta_b0_68,
            "fold": fold,
            "training_subjects": sorted(set(groups_b0_68[treino])),
            "test_subjects": sorted(set(groups_b0_68[teste])),
            "test_indices": teste,
            "fit_seconds": tempo_treino,
            "predict_seconds": tempo_previsao,
            "report": relatorio,
        }

        # Write a temporary file first.
        temporario = destino.with_suffix(".tmp")
        temporario.write_text(
            json.dumps(
                resultado, default=converter_numpy, indent=2
            ),
            encoding="utf-8",
        )
        temporario.replace(destino)

        print(
            f"Fold {fold} finished: "
            f"kappa = {relatorio['cohens_kappa']:.3f} | "
            f"treino = {tempo_treino / 60:.2f} min",
            flush=True,
        )

    resultados_b0_68[fold] = resultado

# Summarise the five folds.
tabela_b0_68 = pd.DataFrame([
    {
        "Fold": fold,
        **{
            metrica: resultado["report"][metrica]
            for metrica in [
                "cohens_kappa", "macro_f1",
                "accuracy", "balanced_accuracy"
            ]
        },
        "Treino (min)": resultado["fit_seconds"] / 60,
    }
    for fold, resultado in resultados_b0_68.items()
]).set_index("Fold")

print("\nBASELINE — FIVE FOLDS COMPLETE")
display(tabela_b0_68.round(3))

kappas = tabela_b0_68["cohens_kappa"]
print(
    f"Mean kappa ± SD across five folds: "
    f"{kappas.mean():.3f} ± {kappas.std(ddof=1):.3f}"
)
```

**Original cell 62**

```python
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
)

estagios = ["W", "N1", "N2", "N3", "REM"]

# Put each prediction back at its original epoch index.
previsoes_b0_68 = np.empty_like(y_b0_68)
fold_por_epoca_b0_68 = np.empty(len(y_b0_68), dtype="<U1")
cobertura_b0_68 = np.zeros(len(y_b0_68), dtype=np.uint8)

for fold, resultado in resultados_b0_68.items():
    indices = np.asarray(resultado["test_indices"], dtype=int)
    relatorio = resultado["report"]

    np.testing.assert_array_equal(
        indices, SPLITS_68[fold][1]
    )
    np.testing.assert_array_equal(
        relatorio["y_true"], y_b0_68[indices]
    )
    np.testing.assert_array_equal(
        relatorio["y_group"], groups_b0_68[indices]
    )

    assert len(relatorio["y_pred"]) == len(indices)
    assert np.all(cobertura_b0_68[indices] == 0)

    previsoes_b0_68[indices] = relatorio["y_pred"]
    fold_por_epoca_b0_68[indices] = fold
    cobertura_b0_68[indices] += 1

assert np.all(cobertura_b0_68 == 1)

# Pooled report with fold and participant results.
relatorio_b0_68 = track_b0_68._make_report(
    y_b0_68,
    previsoes_b0_68,
    groups=groups_b0_68,
    folds=fold_por_epoca_b0_68,
)

print("BASELINE — POOLED METRICS")
for metrica in [
    "cohens_kappa", "macro_f1", "accuracy", "balanced_accuracy"
]:
    print(f"{metrica}: {relatorio_b0_68[metrica]:.3f}")

print("\nRESULTS BY STAGE")
print(classification_report(
    y_b0_68,
    previsoes_b0_68,
    labels=estagios,
    digits=3,
    zero_division=0,
))

# Pooled matrix: each row sums to one.
cm_b0_68 = confusion_matrix(
    y_b0_68, previsoes_b0_68, labels=estagios
)
cm_b0_68_normalizada = (
    cm_b0_68 / cm_b0_68.sum(axis=1, keepdims=True)
)

fig, ax = plt.subplots(figsize=(6, 5))
ConfusionMatrixDisplay(
    cm_b0_68_normalizada,
    display_labels=estagios,
).plot(
    ax=ax,
    cmap="Blues",
    values_format=".3f",
    colorbar=False,
    im_kw={"vmin": 0, "vmax": 1},
)

ax.set_title("B0 — 68 subjects — fixed A–E folds")
ax.set_xlabel("Predicted stage")
ax.set_ylabel("True stage")
plt.tight_layout()
plt.show()

# Values for the team spreadsheet.
tabela_cm_b0_68 = pd.DataFrame(
    cm_b0_68_normalizada,
    index=estagios,
    columns=estagios,
)
tabela_cm_b0_68.index.name = "True / Predicted"

print("\nMATRIZ PARA COPIAR PARA O GOOGLE SHEETS")
print(tabela_cm_b0_68.to_csv(
    sep="\t", float_format="%.4f"
))
```


**Original cell 63**

```python
metricas = [
    "cohens_kappa", "macro_f1", "accuracy", "balanced_accuracy"
]

resumo_folds_b0_68 = {
    metrica: {
        "mean": tabela_b0_68[metrica].mean(),
        "std": tabela_b0_68[metrica].std(ddof=1),
        "n": 5,
        "ddof": 1,
    }
    for metrica in metricas
}

consolidado_b0_68 = {
    **meta_b0_68,
    "cv_method": "fixed_subject_5fold_AE",
    "preprocessing": "no additional filtering",
    "spectral_method": "welch",
    "training_crop": False,
    "evaluation_scope": "all original valid epochs, including Wake",
    "fold_metric_summary": resumo_folds_b0_68,
    "fit_seconds_total": sum(
        r["fit_seconds"] for r in resultados_b0_68.values()
    ),
    "report": relatorio_b0_68,
}

destino = EXPERIMENTS_DIR / "B0_68_fixed_AE.json"
temporario = destino.with_suffix(".tmp")

temporario.write_text(
    json.dumps(
        consolidado_b0_68,
        default=converter_numpy,
        indent=2,
    ),
    encoding="utf-8",
)
temporario.replace(destino)

imagem = EXPERIMENTS_DIR / "B0_68_fixed_AE_confusion.png"
fig.savefig(imagem, dpi=200, bbox_inches="tight")

print("Report saved:", destino)
print("Image saved:", imagem)
```


### 5.3 — F2 absolute and relative EEG powers

F2 adds five relative EEG powers to the 11 B0 features, giving 16 features per epoch. It is the reference for the 16-feature filtering and training-selection comparisons.


**Original cell 64**

```python
F2_FEATURE_VERSION = 1

bandas_f2_68 = ["delta", "theta", "alpha", "sigma", "beta"]

indices_bandas_f2_68 = [
    nomes_features_b0.index(f"eeg_{banda}")
    for banda in bandas_f2_68
]

potencias = X_b0_68[:, indices_bandas_f2_68]

assert np.isfinite(potencias).all()
assert (potencias >= 0).all()

# Proportion of power in each of the five bands.
total = potencias.sum(axis=1, keepdims=True)

relativas = np.divide(
    potencias,
    total,
    out=np.zeros_like(potencias, dtype=float),
    where=total > 0,
)

# Keep the 11 features and append five relative powers.
X_f2_68 = np.column_stack([X_b0_68, relativas])

nomes_features_f2_68 = nomes_features_b0 + [
    f"eeg_{banda}_relative"
    for banda in bandas_f2_68
]

assert X_f2_68.shape == (len(y_b0_68), 16)
assert np.isfinite(X_f2_68).all()

print("Baseline:", X_b0_68.shape)
print("F2:", X_f2_68.shape)
print("F2 prepared with the same labels and folds A–E.")
```

**Original cell 65**

```python
from sklearn.base import clone

def avaliar_experiencia_68(
    nome, X, nomes_features, versao_features, detalhes=None
):
    assert X.shape == (len(y_b0_68), len(nomes_features))
    assert np.isfinite(X).all()

    modelo_base = clone(modelo_base_b0)

    meta = {
        **meta_b0_68,
        "experiment": nome,
        "feature_version": versao_features,
        "feature_names": list(nomes_features),
        "classifier_params": modelo_base.named_steps["clf"].get_params(),
        "scaler_params": modelo_base.named_steps["scale"].get_params(),
    }
    if detalhes is not None:
        meta["details"] = detalhes

    resultados = {}

    for fold in "ABCDE":
        treino, teste = SPLITS_68[fold]

        # Check that the test set matches B0.
        np.testing.assert_array_equal(
            teste,
            resultados_b0_68[fold]["test_indices"],
        )
        assert set(groups_b0_68[teste]) == set(FOLDS_68[fold])
        assert set(groups_b0_68[treino]).isdisjoint(
            set(groups_b0_68[teste])
        )

        destino = EXPERIMENTS_DIR / f"{nome}_fold_{fold}.json"

        if destino.exists():
            resultado = json.loads(
                destino.read_text(encoding="utf-8")
            )

            assert resultado["fold"] == fold
            for campo, valor in meta.items():
                assert resultado[campo] == valor, (
                    f"Fold {fold}: different configuration in {campo}."
                )

            np.testing.assert_array_equal(
                resultado["test_indices"], teste
            )
            np.testing.assert_array_equal(
                resultado["report"]["y_true"], y_b0_68[teste]
            )
            np.testing.assert_array_equal(
                resultado["report"]["y_group"], groups_b0_68[teste]
            )

            print(f"{nome} — fold {fold}: reused.", flush=True)

        else:
            print(
                f"\n{nome} — fold {fold}: fitting...",
                flush=True,
            )

            # Fit a fresh scaler and classifier in each fold.
            modelo = clone(modelo_base)

            inicio = perf_counter()
            modelo.fit(X[treino], y_b0_68[treino])
            tempo_treino = perf_counter() - inicio

            inicio = perf_counter()
            previsoes = modelo.predict(X[teste])
            tempo_previsao = perf_counter() - inicio

            relatorio = track_b0_68._make_report(
                y_b0_68[teste],
                previsoes,
                groups=groups_b0_68[teste],
            )

            resultado = {
                **meta,
                "fold": fold,
                "training_subjects": sorted(set(groups_b0_68[treino])),
                "test_subjects": sorted(set(groups_b0_68[teste])),
                "test_indices": teste,
                "fit_seconds": tempo_treino,
                "predict_seconds": tempo_previsao,
                "report": relatorio,
            }

            temporario = destino.with_suffix(".tmp")
            temporario.write_text(
                json.dumps(
                    resultado,
                    default=converter_numpy,
                    indent=2,
                ),
                encoding="utf-8",
            )
            temporario.replace(destino)

            print(
                f"Fold {fold} finished: "
                f"kappa = {relatorio['cohens_kappa']:.3f} | "
                f"treino = {tempo_treino / 60:.2f} min",
                flush=True,
            )

        resultados[fold] = resultado

    tabela = pd.DataFrame([
        {
            "Fold": fold,
            **{
                metrica: resultado["report"][metrica]
                for metrica in [
                    "cohens_kappa", "macro_f1",
                    "accuracy", "balanced_accuracy",
                ]
            },
            "Treino (min)": resultado["fit_seconds"] / 60,
        }
        for fold, resultado in resultados.items()
    ]).set_index("Fold")

    return resultados, tabela


# Evaluate F2.
resultados_f2_68, tabela_f2_68 = avaliar_experiencia_68(
    nome="F2_68_fixed_AE",
    X=X_f2_68,
    nomes_features=nomes_features_f2_68,
    versao_features=F2_FEATURE_VERSION,
    detalhes={
        "base_feature_version": B0_FEATURE_VERSION,
        "relative_bands": bandas_f2_68,
        "denominator": "sum of the five EEG band powers",
        "normalization_scope": "each epoch independently",
    },
)

print("\nF2 — FIVE FOLDS COMPLETE")
display(tabela_f2_68.round(3))

kappas_f2 = tabela_f2_68["cohens_kappa"]
print(
    f"Mean kappa ± SD across five folds: "
    f"{kappas_f2.mean():.3f} ± {kappas_f2.std(ddof=1):.3f}"
)

comparacao_b0_f2_68 = pd.DataFrame({
    "B0": tabela_b0_68["cohens_kappa"],
    "F2": tabela_f2_68["cohens_kappa"],
})
comparacao_b0_f2_68["Diferença"] = (
    comparacao_b0_f2_68["F2"] - comparacao_b0_f2_68["B0"]
)

print("\nKAPPA COMPARISON ON MATCHED FOLDS")
display(comparacao_b0_f2_68.round(3))
```


**Original cell 66**

```python
from sklearn.metrics import (
    classification_report,
    ConfusionMatrixDisplay,
)

# Helper to assemble predictions from all five folds.
def juntar_previsoes_68(resultados):
    assert set(resultados) == set("ABCDE")

    previsoes = np.empty_like(y_b0_68)
    folds = np.empty(len(y_b0_68), dtype="<U1")
    cobertura = np.zeros(len(y_b0_68), dtype=np.uint8)

    for fold, resultado in resultados.items():
        indices = np.asarray(resultado["test_indices"], dtype=int)
        relatorio = resultado["report"]

        np.testing.assert_array_equal(
            indices, SPLITS_68[fold][1]
        )
        np.testing.assert_array_equal(
            relatorio["y_true"], y_b0_68[indices]
        )
        np.testing.assert_array_equal(
            relatorio["y_group"], groups_b0_68[indices]
        )

        assert len(relatorio["y_pred"]) == len(indices)
        assert np.all(cobertura[indices] == 0)

        previsoes[indices] = relatorio["y_pred"]
        folds[indices] = fold
        cobertura[indices] += 1

    assert np.all(cobertura == 1)

    relatorio = track_b0_68._make_report(
        y_b0_68,
        previsoes,
        groups=groups_b0_68,
        folds=folds,
    )

    return previsoes, relatorio


previsoes_f2_68, relatorio_f2_68 = juntar_previsoes_68(
    resultados_f2_68
)

estagios = ["W", "N1", "N2", "N3", "REM"]

print("F2 — POOLED METRICS")
for metrica in [
    "cohens_kappa", "macro_f1", "accuracy", "balanced_accuracy"
]:
    print(f"{metrica}: {relatorio_f2_68[metrica]:.3f}")

print("\nF2 — RESULTS BY STAGE")
print(classification_report(
    y_b0_68,
    previsoes_f2_68,
    labels=estagios,
    digits=3,
    zero_division=0,
))

# Compare directly with the baseline.
classes_b0 = classification_report(
    y_b0_68, previsoes_b0_68,
    labels=estagios, output_dict=True, zero_division=0,
)
classes_f2 = classification_report(
    y_b0_68, previsoes_f2_68,
    labels=estagios, output_dict=True, zero_division=0,
)

comparacao_estagios_b0_f2_68 = pd.DataFrame([
    {
        "Estágio": estagio,
        "Recall B0": classes_b0[estagio]["recall"],
        "Recall F2": classes_f2[estagio]["recall"],
        "F1 B0": classes_b0[estagio]["f1-score"],
        "F1 F2": classes_f2[estagio]["f1-score"],
        "Diferença F1": (
            classes_f2[estagio]["f1-score"]
            - classes_b0[estagio]["f1-score"]
        ),
    }
    for estagio in estagios
]).set_index("Estágio")

print("\nCOMPARISON BY STAGE")
display(comparacao_estagios_b0_f2_68.round(3))

# Use the same colour scale as the baseline confusion matrix.
fig_f2_68, ax_f2_68 = plt.subplots(figsize=(6, 5))

ConfusionMatrixDisplay.from_predictions(
    y_b0_68,
    previsoes_f2_68,
    labels=estagios,
    normalize="true",
    values_format=".3f",
    cmap="Blues",
    colorbar=False,
    im_kw={"vmin": 0, "vmax": 1},
    ax=ax_f2_68,
)

ax_f2_68.set_title("F2 — 68 subjects — fixed A–E folds")
ax_f2_68.set_xlabel("Predicted stage")
ax_f2_68.set_ylabel("True stage")
plt.tight_layout()
plt.show()
```

**Original cell 67**

```python
# Separate shared metadata from fold A fields.
campos_do_fold = {
    "fold", "training_subjects", "test_subjects",
    "test_indices", "fit_seconds", "predict_seconds", "report",
}

meta_f2_68 = {
    campo: valor
    for campo, valor in resultados_f2_68["A"].items()
    if campo not in campos_do_fold
}

consolidado_f2_68 = {
    **meta_f2_68,
    "cv_method": "fixed_subject_5fold_AE",
    "preprocessing": "no additional filtering",
    "spectral_method": "welch",
    "training_crop": False,
    "evaluation_scope": "all original valid epochs, including Wake",
    "fold_metric_summary": {
        metrica: {
            "mean": tabela_f2_68[metrica].mean(),
            "std": tabela_f2_68[metrica].std(ddof=1),
            "n": 5,
            "ddof": 1,
        }
        for metrica in [
            "cohens_kappa", "macro_f1",
            "accuracy", "balanced_accuracy",
        ]
    },
    "fit_seconds_total": sum(
        r["fit_seconds"] for r in resultados_f2_68.values()
    ),
    "report": relatorio_f2_68,
}

destino_f2 = EXPERIMENTS_DIR / "F2_68_fixed_AE.json"
temporario_f2 = destino_f2.with_suffix(".tmp")

temporario_f2.write_text(
    json.dumps(
        consolidado_f2_68,
        default=converter_numpy,
        indent=2,
    ),
    encoding="utf-8",
)
temporario_f2.replace(destino_f2)

imagem_f2 = EXPERIMENTS_DIR / "F2_68_fixed_AE_confusion.png"
fig_f2_68.savefig(imagem_f2, dpi=200, bbox_inches="tight")

print("Report saved:", destino_f2)
print("Image saved:", imagem_f2)
```


### 5.4 — Signal and spectrum checks

The historical inspection compares retained EEG epochs and near-50 Hz power across development recordings. The current, optional diagnostic cells are in section 4.


**Original cell 68**

```python
# Results for each of the 68 participants.
scores_inspecao = pd.DataFrame(
    relatorio_f2_68["per_group"]
)[["group", "cohens_kappa"]]

scores_inspecao = (
    scores_inspecao
    .dropna(subset=["cohens_kappa"])
    .sort_values(["cohens_kappa", "group"])
    .reset_index(drop=True)
)

assert len(scores_inspecao) >= 3

selecoes = [
    ("Kappa mais baixo", 0),
    ("Middle kappa", len(scores_inspecao) // 2),
    ("Kappa mais alto", len(scores_inspecao) - 1),
]

plano_inspecao_68 = []

for perfil, posicao in selecoes:
    linha = scores_inspecao.iloc[posicao]
    participante = str(linha["group"])

    # Choose the first available recording for this participant.
    gravacoes = sorted(
        [
            dados for dados in b0_gravacoes_68
            if dados["record"][:5] == participante
        ],
        key=lambda dados: dados["record"],
    )

    assert gravacoes
    dados = gravacoes[0]

    plano_inspecao_68.append({
        "Perfil": perfil,
        "Participante": participante,
        "Kappa F2": float(linha["cohens_kappa"]),
        "Gravação": dados["record"],
        "Épocas": len(dados["y"]),
        "Épocas N1": int(np.count_nonzero(dados["y"] == "N1")),
    })

display(pd.DataFrame(plano_inspecao_68).round(3))
```

**Original cell 69**

```python
from pathlib import Path
from tempfile import TemporaryDirectory
from scipy.signal import welch

def carregar_gravacao_inspecao_68(record_id):
    assert record_id[:5] in DEVELOPMENT_GROUPS_68

    ficheiros = list(TRAIN_DIR.rglob(f"{record_id}-PSG.edf"))
    assert len(ficheiros) == 1
    psg = ficheiros[0]

    marcacoes = list(
        psg.parent.glob(f"{record_id[:6]}*-Hypnogram.edf")
    )
    assert len(marcacoes) == 1

    # Read only this file pair through local links.
    with TemporaryDirectory() as pasta:
        for origem in (psg, marcacoes[0]):
            (Path(pasta) / origem.name).symlink_to(origem.resolve())

        gravacoes = track_b0_68.load(
            pasta, fs_target=100.0, epoch_s=30.0
        )

    assert len(gravacoes) == 1
    return gravacoes[0]


record_id_inspecao = plano_inspecao_68[0]["Gravação"]

# Reuse the recording in memory if this cell is run again.
if (
    "rec_inspecao_68" not in globals()
    or rec_inspecao_68.meta["record"] != record_id_inspecao
):
    rec_inspecao_68 = carregar_gravacao_inspecao_68(
        record_id_inspecao
    )

rec = rec_inspecao_68
fs = float(rec.fs)
eeg_uv = np.asarray(rec.epochs["eeg"]) * 1e6

# Check alignment with the saved F2 predictions.
indices_gravacao = np.flatnonzero(
    record_ids_b0_68 == record_id_inspecao
)
np.testing.assert_array_equal(
    rec.labels, y_b0_68[indices_gravacao]
)
previsoes_gravacao = previsoes_f2_68[indices_gravacao]

finitas = np.isfinite(eeg_uv).all(axis=1)
print(f"Gravação: {record_id_inspecao} | Epochs: {len(rec.labels)}")
print("Epochs with NaN/Inf:", int((~finitas).sum()))
assert finitas.all(), "Check invalid values before plotting."

amplitudes = np.ptp(eeg_uv, axis=1)
print("Completely flat epochs:", int((amplitudes == 0).sum()))

fig_inspecao_68, axes = plt.subplots(2, 2, figsize=(13, 7))
exemplos_inspecao_68 = []

for linha, estagio in enumerate(["W", "N1"]):
    candidatos = np.flatnonzero(rec.labels == estagio)
    assert len(candidatos) > 0

    # Epoch closest to the median peak-to-peak amplitude in this stage.
    mediana = np.median(amplitudes[candidatos])
    i = candidatos[
        np.argmin(np.abs(amplitudes[candidatos] - mediana))
    ]

    sinal = eeg_uv[i]
    tempo = np.arange(len(sinal)) / fs

    axes[linha, 0].plot(tempo, sinal, linewidth=0.7)
    axes[linha, 0].set(
        title=f"True: {estagio} | F2: {previsoes_gravacao[i]} | Epoch: {i}",
        xlabel="Time within epoch (s)",
        ylabel="EEG (µV)",
        xlim=(0, len(sinal) / fs),
    )

    # These settings are only for the inspection plot.
    nperseg = min(int(8 * fs), len(sinal))
    frequencias, psd = welch(
        sinal, fs=fs, window="hann",
        nperseg=nperseg, noverlap=nperseg // 2,
        detrend="constant", scaling="density",
    )

    positivos = psd > 0
    axes[linha, 1].semilogy(
        frequencias[positivos], psd[positivos]
    )
    axes[linha, 1].set(
        title=f"{estagio} — Welch PSD",
        xlabel="Frequency (Hz)",
        ylabel="PSD (µV²/Hz)",
        xlim=(0, fs / 2),
    )

    for ax in axes[linha]:
        ax.grid(True, alpha=0.25)

    exemplos_inspecao_68.append({
        "Reference stage": estagio,
        "Época retida": int(i),
        "Previsão F2": str(previsoes_gravacao[i]),
        "Pico-a-pico (µV)": float(amplitudes[i]),
    })

fig_inspecao_68.suptitle(f"{record_id_inspecao} — EEG Fpz-Cz")
fig_inspecao_68.tight_layout()
plt.show()

display(pd.DataFrame(exemplos_inspecao_68).round(3))
```


**Original cell 70**

```python
def resumir_espectros_68(rec):
    fs = float(rec.fs)
    assert np.isclose(fs, 100.0)

    eeg = np.asarray(rec.epochs["eeg"]) * 1e6
    assert np.isfinite(eeg).all()

    nperseg = min(int(8 * fs), eeg.shape[1])

    # Calculate one PSD per epoch without joining non-consecutive epochs.
    frequencias, psds = welch(
        eeg,
        fs=fs,
        window="hann",
        nperseg=nperseg,
        noverlap=nperseg // 2,
        detrend="constant",
        scaling="density",
        axis=-1,
    )

    banda_total = (frequencias >= 0.5) & (frequencias <= 50)
    banda_alta = (frequencias >= 48) & (frequencias <= 50)

    potencia_total = np.trapezoid(
        psds[:, banda_total],
        frequencias[banda_total],
        axis=1,
    )
    potencia_alta = np.trapezoid(
        psds[:, banda_alta],
        frequencias[banda_alta],
        axis=1,
    )

    percentagem_alta = 100 * np.divide(
        potencia_alta,
        potencia_total,
        out=np.full_like(potencia_total, np.nan),
        where=potencia_total > 0,
    )

    linhas = []
    fig, axes = plt.subplots(1, 2, figsize=(13, 4.5), sharey=True)

    for estagio in ["W", "N1", "N2", "N3", "REM"]:
        mascara = rec.labels == estagio
        valores = percentagem_alta[mascara]
        valores = valores[np.isfinite(valores)]

        linhas.append({
            "Estágio": estagio,
            "Épocas": int(mascara.sum()),
            "Mediana 48–50 Hz (%)": (
                float(np.median(valores)) if len(valores) else np.nan
            ),
            "P95 48–50 Hz (%)": (
                float(np.percentile(valores, 95))
                if len(valores) else np.nan
            ),
        })

        if not mascara.any():
            continue

        # Median PSD at each frequency for this stage.
        mediana_psd = np.median(psds[mascara], axis=0)
        visiveis = banda_total & (mediana_psd > 0)

        for ax in axes:
            ax.semilogy(
                frequencias[visiveis],
                mediana_psd[visiveis],
                label=estagio,
            )

    axes[0].set(title="Median PSD by stage", xlim=(0.5, 50))
    axes[1].set(title="Zoom near 50 Hz", xlim=(45, 50))

    for ax in axes:
        ax.set_xlabel("Frequency (Hz)")
        ax.set_ylabel("PSD (µV²/Hz)")
        ax.axvspan(48, 50, color="grey", alpha=0.12)
        ax.grid(True, alpha=0.25)
        ax.legend()

    fig.suptitle(rec.meta["record"])
    fig.tight_layout()

    return pd.DataFrame(linhas).set_index("Estágio"), fig


tabela_espectral_68, fig_espectral_68 = resumir_espectros_68(
    rec_inspecao_68
)

print("Percentage of 0.5–50 Hz power in 48–50 Hz:")
display(tabela_espectral_68.round(2))
plt.show()
```

**Original cell 71**

```python
# Keep the SC4611E0 diagnostic.
primeira_gravacao = plano_inspecao_68[0]["Gravação"]

tabelas_espectrais_68 = {
    primeira_gravacao: tabela_espectral_68.copy()
}

figuras_espectrais_68 = {
    primeira_gravacao: fig_espectral_68
}

# Inspect the participants with middle and high kappa.
for selecao in plano_inspecao_68[1:]:
    record_id = selecao["Gravação"]

    print(
        f"\n{selecao['Perfil']} | "
        f"{selecao['Participante']} | "
        f"Gravação: {record_id}",
        flush=True,
    )

    rec_diagnostico = carregar_gravacao_inspecao_68(record_id)

    # Check that the evaluation uses the same epochs.
    indices = np.flatnonzero(record_ids_b0_68 == record_id)
    np.testing.assert_array_equal(
        rec_diagnostico.labels,
        y_b0_68[indices],
    )

    tabela_atual, figura_atual = resumir_espectros_68(
        rec_diagnostico
    )

    tabelas_espectrais_68[record_id] = tabela_atual.copy()
    figuras_espectrais_68[record_id] = figura_atual

    display(tabela_atual.round(2))
    plt.show()

    del rec_diagnostico

# Compare the three recordings.
comparacao_espectral_68 = pd.concat(
    tabelas_espectrais_68,
    names=["Gravação", "Estágio"],
).reset_index()

print("\nCOMPARISON OF THREE RECORDINGS")
display(comparacao_espectral_68.round(2))
```


### 5.5 — LP40 applied to all EEG epochs

The 40 Hz Butterworth low-pass changes EEG only. The caches retain the same labels, epoch order and EOG/EMG features as F2. Full LP40 produced little change in overall kappa.


**Original cell 72**

```python
from scipy.signal import butter, sosfiltfilt, welch

LP40_PREPROCESS_VERSION = 1
LP40_SETTINGS = {
    "cutoff_hz": 40.0,
    "order_per_pass": 4,
    "padtype": "odd",
}

def filtrar_eeg_lp40(eeg, fs):
    sos = butter(
        LP40_SETTINGS["order_per_pass"],
        LP40_SETTINGS["cutoff_hz"],
        btype="lowpass",
        fs=fs,
        output="sos",
    )
    return sosfiltfilt(
        sos,
        eeg,
        axis=-1,
        padtype=LP40_SETTINGS["padtype"],
    )


rec_preview = rec_inspecao_68
assert rec_preview.meta["record"] == "SC4611E0"

epoca_preview = 1798
assert rec_preview.labels[epoca_preview] == "N1"

fs = float(rec_preview.fs)

# Use a copy in µV for the plots.
original = rec_preview.epochs["eeg"][epoca_preview] * 1e6
filtrado = filtrar_eeg_lp40(original, fs)

assert np.isfinite(filtrado).all()

tempo = np.arange(len(original)) / fs
zoom = (tempo >= 10) & (tempo < 11)

parametros_psd = {
    "fs": fs,
    "window": "hann",
    "nperseg": int(8 * fs),
    "noverlap": int(4 * fs),
    "detrend": "constant",
    "scaling": "density",
}

frequencias, psd_original = welch(original, **parametros_psd)
_, psd_filtrado = welch(filtrado, **parametros_psd)

fig_lp40_preview_68, axes = plt.subplots(
    1, 2, figsize=(13, 4.5)
)

axes[0].plot(
    tempo[zoom], original[zoom],
    label="Original", alpha=0.7,
)
axes[0].plot(
    tempo[zoom], filtrado[zoom],
    label="LP40", linewidth=1.5,
)
axes[0].set(
    title="N1 — epoch 1798 — one-second zoom",
    xlabel="Time within epoch (s)",
    ylabel="EEG (µV)",
)

# Use the full 30 seconds for the PSD.
axes[1].semilogy(
    frequencias, psd_original, label="Original"
)
axes[1].semilogy(
    frequencias, psd_filtrado, label="LP40"
)
axes[1].axvline(40, color="grey", linestyle="--")
axes[1].set(
    title="PSD before and after filtering",
    xlabel="Frequency (Hz)",
    ylabel="PSD (µV²/Hz)",
    xlim=(0.5, 50),
)

for ax in axes:
    ax.grid(True, alpha=0.25)
    ax.legend()

fig_lp40_preview_68.suptitle("SC4611E0 — EEG-only LP40 preview")
fig_lp40_preview_68.tight_layout()
plt.show()
```

**Original cell 73**

```python
from adapter import cached
from time import perf_counter

LP40_FEATURE_VERSION = 1

def preparar_lp40_gravacao_68(psg):
    psg = Path(psg).resolve()
    record_id = psg.name.removesuffix("-PSG.edf")

    assert record_id[:5] in DEVELOPMENT_GROUPS_68

    marcacoes = list(
        psg.parent.glob(f"{psg.name[:6]}*-Hypnogram.edf")
    )
    assert len(marcacoes) == 1

    referencia = next(
        dados for dados in b0_gravacoes_68
        if dados["record"] == record_id
    )

    chave = {
        "course_revision": COURSE_REVISION,
        "files": [
            (p.name, p.stat().st_size, p.stat().st_mtime_ns)
            for p in (psg, marcacoes[0])
        ],
        "load": {"fs_target": 100.0, "epoch_s": 30.0},
        "runtime": {
            nome: runtime_versions[nome]
            for nome in ("python", "numpy", "scipy", "mne")
        },
        "preprocessing": {
            "version": LP40_PREPROCESS_VERSION,
            "settings": dict(LP40_SETTINGS),
            "channels": ["eeg"],
            "application": "sosfiltfilt, per epoch, axis=-1",
        },
        "features": {
            "version": LP40_FEATURE_VERSION,
            "base_version": B0_FEATURE_VERSION,
            "relative_version": F2_FEATURE_VERSION,
            "spectral_method": "welch",
            "names": nomes_features_f2_68,
        },
    }

    def calcular():
        rec = carregar_gravacao_inspecao_68(record_id)

        # Filter EEG in volts and keep the loaded EOG and EMG values.
        rec_filtrado = type(rec)(
            group=rec.group,
            fs=rec.fs,
            epochs={
                **rec.epochs,
                "eeg": filtrar_eeg_lp40(rec.epochs["eeg"], rec.fs),
            },
            labels=rec.labels,
            meta=dict(rec.meta),
        )

        X11, labels, participante = track_b0_68.extract_features(
            rec_filtrado,
            cfg={"spectral_method": "welch"},
        )

        potencias = X11[:, indices_bandas_f2_68]
        assert np.isfinite(potencias).all()
        assert (potencias >= 0).all()

        total = potencias.sum(axis=1, keepdims=True)
        relativas = np.divide(
            potencias,
            total,
            out=np.zeros_like(potencias, dtype=float),
            where=total > 0,
        )

        return {
            "record": record_id,
            "X": np.column_stack([X11, relativas]),
            "y": labels,
            "groups": np.repeat(participante, len(labels)),
            "feature_names": list(nomes_features_f2_68),
        }

    dados = cached(
        "viggy_lp40_features",
        chave,
        calcular,
        CACHE_VIGGY,
    )

    # Check that the comparison uses the same epochs and participants.
    assert dados["record"] == record_id
    assert dados["feature_names"] == nomes_features_f2_68
    assert dados["X"].shape == (len(referencia["y"]), 16)
    assert np.isfinite(dados["X"]).all()

    np.testing.assert_array_equal(dados["y"], referencia["y"])
    np.testing.assert_array_equal(dados["groups"], referencia["groups"])

    # Check that EOG and EMG features are unchanged.
    indices_auxiliares = [
        nomes_features_b0.index(nome)
        for nome in ("eog_movement", "emg_rms")
    ]
    np.testing.assert_allclose(
        dados["X"][:, indices_auxiliares],
        referencia["X"][:, indices_auxiliares],
        rtol=1e-12,
        atol=0,
    )

    return dados


psg_lp40_teste = next(TRAIN_DIR.rglob("SC4611E0-PSG.edf"))

inicio = perf_counter()
lp40_teste = preparar_lp40_gravacao_68(psg_lp40_teste)

print("\nGravação:", lp40_teste["record"])
print("Épocas:", len(lp40_teste["y"]))
print("Features:", lp40_teste["X"].shape[1])
print("Verified labels, participants and EOG/EMG.")
print(f"Preparation time: {perf_counter() - inicio:.1f} segundos")
```


**Original cell 74**

```python
# Prepare LP40 for the same 68 participants.
assert len(ficheiros_b0_68) == 133
assert {
    p.name[:5] for p in ficheiros_b0_68
} == set(DEVELOPMENT_GROUPS_68)

lp40_gravacoes_68 = []
inicio_lp40_68 = perf_counter()

for numero, psg in enumerate(ficheiros_b0_68, start=1):
    inicio_gravacao = perf_counter()

    dados = preparar_lp40_gravacao_68(psg)
    lp40_gravacoes_68.append(dados)

    segundos = perf_counter() - inicio_gravacao
    minutos_total = (perf_counter() - inicio_lp40_68) / 60

    print(
        f"[{numero:03d}/133] {dados['record']} | "
        f"{len(dados['y'])} epochs | "
        f"{segundos:.1f} s | "
        f"total: {minutos_total:.1f} min",
        flush=True,
    )

# Combine only the feature and identifier tables.
X_lp40_68 = np.concatenate([
    dados["X"] for dados in lp40_gravacoes_68
])
y_lp40_68 = np.concatenate([
    dados["y"] for dados in lp40_gravacoes_68
])
groups_lp40_68 = np.concatenate([
    dados["groups"] for dados in lp40_gravacoes_68
])
record_ids_lp40_68 = np.concatenate([
    np.repeat(dados["record"], len(dados["y"]))
    for dados in lp40_gravacoes_68
])

# Check that the epochs and their order exactly match F2.
assert X_lp40_68.shape == X_f2_68.shape

np.testing.assert_array_equal(y_lp40_68, y_b0_68)
np.testing.assert_array_equal(groups_lp40_68, groups_b0_68)
np.testing.assert_array_equal(
    record_ids_lp40_68, record_ids_b0_68
)

print("\nLP40 — PREPARATION COMPLETE")
print("Participantes:", len(set(groups_lp40_68)))
print("Recordings:", len(lp40_gravacoes_68))
print("Shape:", X_lp40_68.shape)
print("Verified labels, participants and epoch order.")
```

**Original cell 75**

```python
# Evaluate LP40 on the same five folds as B0 and F2.

resultados_lp40_68, tabela_lp40_68 = avaliar_experiencia_68(
    nome="LP40_68_fixed_AE",
    X=X_lp40_68,
    nomes_features=nomes_features_f2_68,
    versao_features=LP40_FEATURE_VERSION,
    detalhes={
        "base_feature_version": B0_FEATURE_VERSION,
        "relative_feature_version": F2_FEATURE_VERSION,
        "relative_bands": list(bandas_f2_68),
        "denominator": "sum of the five EEG band powers",
        "normalization_scope": "each epoch independently",
        "spectral_method": "welch",
        "preprocessing": {
            "version": LP40_PREPROCESS_VERSION,
            "method": "Butterworth low-pass",
            "settings": dict(LP40_SETTINGS),
            "channels": ["eeg"],
            "application": "sosfiltfilt, per epoch, axis=-1",
        },
        "training_crop": False,
    },
)

print("\nLP40 — FIVE FOLDS COMPLETE")
display(tabela_lp40_68.round(3))

kappas_lp40 = tabela_lp40_68["cohens_kappa"]

print(
    "\nMean kappa ± SD across five folds: "
    f"{kappas_lp40.mean():.3f} ± "
    f"{kappas_lp40.std(ddof=1):.3f}"
)

print(
    "Total fit time: "
    f"{tabela_lp40_68['Treino (min)'].sum():.1f} min"
)

# Use the same training and test participants in each fold.
comparacao_f2_lp40_68 = pd.DataFrame({
    "F2": tabela_f2_68["cohens_kappa"],
    "LP40": tabela_lp40_68["cohens_kappa"],
})

comparacao_f2_lp40_68["Diferença"] = (
    comparacao_f2_lp40_68["LP40"]
    - comparacao_f2_lp40_68["F2"]
)

print("\nKAPPA COMPARISON ON MATCHED FOLDS")
display(comparacao_f2_lp40_68.round(3))
```

**Original cell 76**

```python
from sklearn.metrics import classification_report

# Assemble test predictions from all five folds.
previsoes_lp40_68, relatorio_lp40_68 = juntar_previsoes_68(
    resultados_lp40_68
)

print("LP40 — POOLED METRICS")
for metrica in [
    "cohens_kappa",
    "macro_f1",
    "accuracy",
    "balanced_accuracy",
]:
    print(f"{metrica}: {relatorio_lp40_68[metrica]:.3f}")

estagios = ["W", "N1", "N2", "N3", "REM"]

print("\nLP40 — RESULTS BY STAGE")
print(classification_report(
    y_b0_68,
    previsoes_lp40_68,
    labels=estagios,
    digits=3,
    zero_division=0,
))

# Compare F2 and LP40 on the same epochs.
classes_f2 = classification_report(
    y_b0_68,
    previsoes_f2_68,
    labels=estagios,
    output_dict=True,
    zero_division=0,
)

classes_lp40 = classification_report(
    y_b0_68,
    previsoes_lp40_68,
    labels=estagios,
    output_dict=True,
    zero_division=0,
)

comparacao_estagios_f2_lp40_68 = pd.DataFrame([
    {
        "Estágio": estagio,
        "Recall F2": classes_f2[estagio]["recall"],
        "Recall LP40": classes_lp40[estagio]["recall"],
        "F1 F2": classes_f2[estagio]["f1-score"],
        "F1 LP40": classes_lp40[estagio]["f1-score"],
        "Diferença F1": (
            classes_lp40[estagio]["f1-score"]
            - classes_f2[estagio]["f1-score"]
        ),
    }
    for estagio in estagios
]).set_index("Estágio")

print("\nCOMPARISON BY STAGE")
display(comparacao_estagios_f2_lp40_68.round(3))
```

**Original cell 77**

```python
from sklearn.metrics import cohen_kappa_score

linhas = []

for participante in sorted(set(groups_b0_68)):
    mascara = groups_b0_68 == participante

    kappa_f2 = cohen_kappa_score(
        y_b0_68[mascara],
        previsoes_f2_68[mascara],
    )

    kappa_lp40 = cohen_kappa_score(
        y_b0_68[mascara],
        previsoes_lp40_68[mascara],
    )

    linhas.append({
        "Participante": participante,
        "F2": kappa_f2,
        "LP40": kappa_lp40,
        "Diferença": kappa_lp40 - kappa_f2,
    })

comparacao_participantes_f2_lp40_68 = (
    pd.DataFrame(linhas).set_index("Participante")
)

print("THREE PARTICIPANTS SELECTED FOR INSPECTION")
display(
    comparacao_participantes_f2_lp40_68.loc[
        ["SC461", "SC453", "SC414"]
    ].round(3)
)

diferencas = comparacao_participantes_f2_lp40_68["Diferença"]

print("\nACROSS 68 PARTICIPANTS — BEFORE ROUNDING:")
print("Kappa aumentou:", int((diferencas > 0).sum()))
print("Kappa diminuiu:", int((diferencas < 0).sum()))
print("Kappa igual:", int((diferencas == 0).sum()))
```


**Original cell 78**

```python
from sklearn.metrics import ConfusionMatrixDisplay

EXPERIMENTS_DIR.mkdir(parents=True, exist_ok=True)

# Confusion matrix: each row sums to 100%.
fig_lp40_68, ax = plt.subplots(figsize=(7, 6))

ConfusionMatrixDisplay.from_predictions(
    y_b0_68,
    previsoes_lp40_68,
    labels=["W", "N1", "N2", "N3", "REM"],
    normalize="true",
    values_format=".3f",
    cmap="Blues",
    colorbar=False,
    ax=ax,
)

ax.set_title("LP40 — 68 subjects — fixed folds A–E")
fig_lp40_68.tight_layout()

destino_imagem = (
    EXPERIMENTS_DIR / "LP40_68_fixed_AE_confusion.png"
)
fig_lp40_68.savefig(
    destino_imagem, dpi=200, bbox_inches="tight"
)
plt.show()

# Restore the configuration shared by all five folds.
campos_especificos_fold = {
    "fold", "training_subjects", "test_subjects",
    "test_indices", "fit_seconds", "predict_seconds", "report",
}

metadados_lp40 = {
    chave: valor
    for chave, valor in resultados_lp40_68["A"].items()
    if chave not in campos_especificos_fold
}

metricas = [
    "cohens_kappa", "macro_f1",
    "accuracy", "balanced_accuracy",
]

exportacao_lp40 = {
    **metadados_lp40,
    "split_unit": "subject",
    "cv_method": "five fixed folds A–E",
    "n_subjects": len(set(groups_b0_68)),
    "n_epochs": len(y_b0_68),
    "cv_mean_sd_across_folds": {
        metrica: {
            "mean": float(tabela_lp40_68[metrica].mean()),
            "sd": float(tabela_lp40_68[metrica].std(ddof=1)),
        }
        for metrica in metricas
    },
    "sd_ddof": 1,
    "total_fit_seconds": sum(
        resultado["fit_seconds"]
        for resultado in resultados_lp40_68.values()
    ),
    "fold_metrics": (
        tabela_lp40_68.reset_index().to_dict(orient="records")
    ),
    "report": relatorio_lp40_68,
    "kappa_comparison_vs_F2_by_subject": (
        comparacao_participantes_f2_lp40_68
        .reset_index()
        .to_dict(orient="records")
    ),
    "working_decision": (
        "Keep F2 as reference. LP40 produced mixed results, "
        "with a small overall gain and lower N1 performance."
    ),
}

destino_json = EXPERIMENTS_DIR / "LP40_68_fixed_AE.json"
temporario = destino_json.with_suffix(".tmp")

temporario.write_text(
    json.dumps(
        exportacao_lp40,
        default=converter_numpy,
        indent=2,
        ensure_ascii=False,
    ),
    encoding="utf-8",
)
temporario.replace(destino_json)

print("Report saved:", destino_json)
print("Image saved:", destino_imagem)
```


### 5.6 — C1: crop Wake only during training

C1 keeps the first-to-last sleep interval and 30 minutes on each side during training. All valid test epochs remain. It increased N1 recall but reduced kappa with F2.


**Original cell 82**

```python
C1_CONFIG_68 = {
    "version": 1,
    "margin_minutes": 30.0,
    "epoch_seconds": 30.0,
    "time_basis": "original PSG event times in seconds",
    "selection": "complete epochs inside the sleep window plus margins",
    "scope": "training only",
    "no_sleep_policy": "keep all epochs",
}

# The earlier setup restored the epoch times once in section 1.5.
# Locate each recording in the full feature matrix.
limites = np.r_[
    0,
    np.cumsum([len(d["y"]) for d in b0_gravacoes_68])
]

SPLITS_C1_68 = {}
C1_JANELAS_68 = {}
linhas_resumo = []
linhas_distribuicao = []

estagios = ["W", "N1", "N2", "N3", "REM"]
margem = C1_CONFIG_68["margin_minutes"] * 60
epoca_s = C1_CONFIG_68["epoch_seconds"]


# 2 — Crop only the training recordings in each fold.
for fold in "ABCDE":
    treino, teste = SPLITS_68[fold]
    sujeitos_treino = set(groups_b0_68[treino])

    partes_treino = []
    C1_JANELAS_68[fold] = {}

    for dados, inicio in zip(b0_gravacoes_68, limites[:-1]):
        record = dados["record"]

        if record[:5] not in sujeitos_treino:
            continue

        indices = np.arange(inicio, inicio + len(dados["y"]))
        labels = y_b0_68[indices]
        tempos, duracao = tempos_c1_68[record]

        sono = np.flatnonzero(
            np.isin(labels, ["N1", "N2", "N3", "REM"])
        )

        if len(sono):
            esquerda = max(0.0, tempos[sono[0]] - margem)
            direita = min(
                duracao, tempos[sono[-1]] + epoca_s + margem
            )
        else:
            esquerda, direita = 0.0, duracao

        manter = (
            (tempos >= esquerda - 1e-6)
            & (tempos + epoca_s <= direita + 1e-6)
        )

        # No sleep epoch may be removed.
        assert np.all(labels[~manter] == "W")

        partes_treino.append(indices[manter])
        C1_JANELAS_68[fold][record] = {
            "start_seconds": float(esquerda),
            "end_seconds": float(direita),
            "epochs_before": len(indices),
            "epochs_after": int(manter.sum()),
            "has_sleep": bool(len(sono)),
        }

    treino_c1 = np.concatenate(partes_treino)
    retiradas = np.setdiff1d(treino, treino_c1)

    assert np.all(np.isin(treino_c1, treino))
    assert np.intersect1d(treino_c1, teste).size == 0
    assert set(groups_b0_68[treino_c1]) == sujeitos_treino
    assert np.all(y_b0_68[retiradas] == "W")

    # Keep the original test indices.
    SPLITS_C1_68[fold] = (treino_c1, teste.copy())

    linhas_resumo.append({
        "Fold": fold,
        "Treino antes": len(treino),
        "Treino depois": len(treino_c1),
        "Wake retiradas": len(retiradas),
        "Teste completo": len(teste),
    })

    for estagio in estagios:
        linhas_distribuicao.append({
            "Fold": fold,
            "Estágio": estagio,
            "Antes (%)": 100 * np.mean(
                y_b0_68[treino] == estagio
            ),
            "Depois (%)": 100 * np.mean(
                y_b0_68[treino_c1] == estagio
            ),
        })


resumo_c1_68 = pd.DataFrame(linhas_resumo).set_index("Fold")
distribuicao_c1_68 = pd.DataFrame(
    linhas_distribuicao
).set_index(["Fold", "Estágio"])

print("\nC1_68 — EPOCH COUNTS")
display(resumo_c1_68)

print("\nSTAGE DISTRIBUTION IN TRAINING")
display(distribuicao_c1_68.round(2))

print("\nOnly Wake was removed from the training sets.")
print("Os cinco conjuntos de teste continuam completos.")
```


**Original cell 83**

```python
import json
import platform
import scipy
import sklearn
from time import perf_counter
from sklearn.base import clone
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier

if "SleepEDFTrack" not in globals():
    from sleep_edf import SleepEDFTrack

# Check versions before fitting.
runtime_c1_68 = {
    "python": platform.python_version(),
    "numpy": np.__version__,
    "scipy": scipy.__version__,
    "scikit-learn": sklearn.__version__,
    "mne": mne.__version__,
}

diferencas = {
    pacote: {
        "F2": config_f2_guardada_68["runtime"][pacote],
        "current": runtime_c1_68[pacote],
    }
    for pacote in ["numpy", "scipy", "scikit-learn"]
    if runtime_c1_68[pacote] !=
       config_f2_guardada_68["runtime"][pacote]
}

if diferencas:
    raise RuntimeError(
        "Match the F2 package versions before fitting: "
        + str(diferencas)
    )

track_c1_68 = SleepEDFTrack(cfg={})

# Rebuild the model using the saved F2 parameters.
modelo_base_c1_68 = Pipeline([
    ("scale", StandardScaler(
        **config_f2_guardada_68["scaler_params"]
    )),
    ("clf", RandomForestClassifier(
        **config_f2_guardada_68["classifier_params"]
    )),
])

NOME_C1_68 = "C1_68_crop30_fixed_AE"

campos_fold = {
    "fold", "training_subjects", "test_subjects",
    "test_indices", "fit_seconds", "predict_seconds", "report",
}

meta_c1_68 = {
    chave: valor
    for chave, valor in config_f2_guardada_68.items()
    if chave not in campos_fold
}

meta_c1_68.update({
    "experiment": NOME_C1_68,
    "runtime": runtime_c1_68,
    "reference_experiment": "F2_68_fixed_AE",
    "feature_cache_runtime": config_f2_guardada_68["runtime"],
    "training_selection": dict(C1_CONFIG_68),
})


def converter_json_c1_68(obj):
    if isinstance(obj, np.ndarray):
        return obj.tolist()
    if isinstance(obj, np.generic):
        return obj.item()
    raise TypeError(f"Unsupported type: {type(obj).__name__}")


if "resultados_c1_68" not in globals():
    resultados_c1_68 = {}


def avaliar_fold_c1_68(fold):
    treino, teste = SPLITS_C1_68[fold]

    np.testing.assert_array_equal(
        teste, resultados_f2_68[fold]["test_indices"]
    )
    assert np.intersect1d(treino, teste).size == 0

    destino = EXPERIMENTS_DIR / f"{NOME_C1_68}_fold_{fold}.json"
    guardado_na_drive = destino.is_file()

    if guardado_na_drive:
        resultado = json.loads(
            destino.read_text(encoding="utf-8")
        )
        print(
            f"C1 — fold {fold}: saved result found.",
            flush=True,
        )

    elif fold in resultados_c1_68:
        resultado = resultados_c1_68[fold]
        print(
            f"C1 — fold {fold}: result restored from memory.",
            flush=True,
        )

    else:
        print(
            f"\nC1 — fold {fold}: fitting with "
            f"{len(treino)} epochs...",
            flush=True,
        )

        modelo = clone(modelo_base_c1_68)

        inicio = perf_counter()
        modelo.fit(X_f2_68[treino], y_b0_68[treino])
        tempo_treino = perf_counter() - inicio

        inicio = perf_counter()
        previsoes = modelo.predict(X_f2_68[teste])
        tempo_previsao = perf_counter() - inicio

        relatorio = track_c1_68._make_report(
            y_b0_68[teste],
            previsoes,
            groups=groups_b0_68[teste],
        )

        resultado = {
            **meta_c1_68,
            "fold": fold,
            "training_subjects": sorted(set(groups_b0_68[treino])),
            "test_subjects": sorted(set(groups_b0_68[teste])),
            "train_indices": treino,
            "test_indices": teste,
            "training_windows": C1_JANELAS_68[fold],
            "fit_seconds": tempo_treino,
            "predict_seconds": tempo_previsao,
            "report": relatorio,
        }

    # Check configuration and alignment before reuse or export.
    assert resultado["fold"] == fold

    for campo, valor in meta_c1_68.items():
        assert resultado[campo] == valor, (
            f"Configuração diferente no fold {fold}: {campo}"
        )

    np.testing.assert_array_equal(
        resultado["train_indices"], treino
    )
    np.testing.assert_array_equal(
        resultado["test_indices"], teste
    )
    np.testing.assert_array_equal(
        resultado["report"]["y_true"], y_b0_68[teste]
    )
    np.testing.assert_array_equal(
        resultado["report"]["y_group"], groups_b0_68[teste]
    )
    assert resultado["training_windows"] == C1_JANELAS_68[fold]

    # Keep the result in memory before writing to Drive.
    resultados_c1_68[fold] = resultado

    if not guardado_na_drive:
        temporario = destino.with_suffix(".tmp")
        temporario.write_text(
            json.dumps(
                resultado,
                default=converter_json_c1_68,
                indent=2,
            ),
            encoding="utf-8",
        )
        temporario.replace(destino)
        print("Saved:", destino, flush=True)

    print(
        f"Fold {fold} finished: "
        f"kappa = {resultado['report']['cohens_kappa']:.3f} | "
        f"treino = {resultado['fit_seconds'] / 60:.2f} min",
        flush=True,
    )

    return resultado


# Fit one model per fold on cropped training epochs only.
for fold in "ABCDE":
    avaliar_fold_c1_68(fold)

metricas = [
    "cohens_kappa", "macro_f1", "accuracy", "balanced_accuracy"
]

tabela_c1_68 = pd.DataFrame([
    {
        "Fold": fold,
        **{
            m: resultados_c1_68[fold]["report"][m]
            for m in metricas
        },
        "Treino (min)": (
            resultados_c1_68[fold]["fit_seconds"] / 60
        ),
    }
    for fold in "ABCDE"
]).set_index("Fold")

print("\nC1 — FIVE FOLDS COMPLETE")
display(tabela_c1_68.round(3))

kappas_c1 = tabela_c1_68["cohens_kappa"]

print(
    "\nMean kappa ± SD across five folds: "
    f"{kappas_c1.mean():.3f} ± {kappas_c1.std(ddof=1):.3f}"
)
print(
    "Total fit time: "
    f"{tabela_c1_68['Treino (min)'].sum():.1f} min"
)

comparacao_f2_c1_68 = pd.DataFrame({
    "F2": [
        resultados_f2_68[f]["report"]["cohens_kappa"]
        for f in "ABCDE"
    ],
    "C1": kappas_c1.to_numpy(),
}, index=pd.Index(list("ABCDE"), name="Fold"))

comparacao_f2_c1_68["Diferença"] = (
    comparacao_f2_c1_68["C1"] - comparacao_f2_c1_68["F2"]
)

print("\nCOMPARISON ON MATCHED FOLDS")
display(comparacao_f2_c1_68.round(3))
```


**Original cell 84**

```python
import matplotlib.pyplot as plt
from sklearn.metrics import (
    classification_report,
    ConfusionMatrixDisplay,
)


def reconstruir_oof_68(resultados):
    """Assemble test predictions in the original epoch order."""
    assert set(resultados) == set("ABCDE")

    previsoes = np.empty_like(y_b0_68)
    folds = np.empty(len(y_b0_68), dtype="<U1")
    cobertura = np.zeros(len(y_b0_68), dtype=np.uint8)

    for fold in "ABCDE":
        resultado = resultados[fold]
        indices = np.asarray(resultado["test_indices"], dtype=int)
        report = resultado["report"]

        np.testing.assert_array_equal(
            indices, SPLITS_68[fold][1]
        )
        np.testing.assert_array_equal(
            report["y_true"], y_b0_68[indices]
        )
        np.testing.assert_array_equal(
            report["y_group"], groups_b0_68[indices]
        )

        assert len(report["y_pred"]) == len(indices)
        assert np.all(cobertura[indices] == 0)

        previsoes[indices] = report["y_pred"]
        folds[indices] = fold
        cobertura[indices] += 1

    assert np.all(cobertura == 1)

    relatorio = track_c1_68._make_report(
        y_b0_68,
        previsoes,
        groups=groups_b0_68,
        folds=folds,
    )

    return previsoes, relatorio


previsoes_f2_68, relatorio_f2_68 = reconstruir_oof_68(
    resultados_f2_68
)
previsoes_c1_68, relatorio_c1_68 = reconstruir_oof_68(
    resultados_c1_68
)

# 1 — Pooled metrics on the same 360,621 test epochs.
metricas_globais = [
    "cohens_kappa",
    "macro_f1",
    "accuracy",
    "balanced_accuracy",
]

comparacao_global_f2_c1_68 = pd.DataFrame({
    "F2": {m: relatorio_f2_68[m] for m in metricas_globais},
    "C1": {m: relatorio_c1_68[m] for m in metricas_globais},
})
comparacao_global_f2_c1_68["Diferença"] = (
    comparacao_global_f2_c1_68["C1"]
    - comparacao_global_f2_c1_68["F2"]
)

print("COMPARISON — POOLED METRICS")
display(comparacao_global_f2_c1_68.round(3))

# 2 — Results by stage.
estagios = ["W", "N1", "N2", "N3", "REM"]

print("\nC1 — RESULTS BY STAGE")
print(classification_report(
    y_b0_68,
    previsoes_c1_68,
    labels=estagios,
    digits=3,
    zero_division=0,
))

classes_f2 = classification_report(
    y_b0_68,
    previsoes_f2_68,
    labels=estagios,
    output_dict=True,
    zero_division=0,
)
classes_c1 = classification_report(
    y_b0_68,
    previsoes_c1_68,
    labels=estagios,
    output_dict=True,
    zero_division=0,
)

comparacao_estagios_f2_c1_68 = pd.DataFrame([
    {
        "Estágio": estagio,
        "Recall F2": classes_f2[estagio]["recall"],
        "Recall C1": classes_c1[estagio]["recall"],
        "F1 F2": classes_f2[estagio]["f1-score"],
        "F1 C1": classes_c1[estagio]["f1-score"],
        "Diferença F1": (
            classes_c1[estagio]["f1-score"]
            - classes_f2[estagio]["f1-score"]
        ),
    }
    for estagio in estagios
]).set_index("Estágio")

print("\nCOMPARISON BY STAGE")
display(comparacao_estagios_f2_c1_68.round(3))

# 3 — Confusion matrices on the same scale.
fig_f2_c1_68, eixos = plt.subplots(1, 2, figsize=(13, 5.5))

for ax, titulo, previsoes in zip(
    eixos,
    ["F2 — full training", "C1 — cropped training"],
    [previsoes_f2_68, previsoes_c1_68],
):
    matriz = ConfusionMatrixDisplay.from_predictions(
        y_b0_68,
        previsoes,
        labels=estagios,
        normalize="true",
        values_format=".1%",
        cmap="Blues",
        colorbar=False,
        ax=ax,
    )

    matriz.im_.set_clim(0, 1)
    ax.set_title(titulo)
    ax.set_xlabel("Predicted stage")
    ax.set_ylabel("True stage")

fig_f2_c1_68.suptitle(
    "68 subjects — fixed folds A–E — complete test recordings"
)
fig_f2_c1_68.tight_layout()
plt.show()
```


**Original cell 85**

```python
# Compare kappa by participant.
kappa_f2_pessoa = (
    pd.DataFrame(relatorio_f2_68["per_group"])
    .set_index("group")["cohens_kappa"]
)
kappa_c1_pessoa = (
    pd.DataFrame(relatorio_c1_68["per_group"])
    .set_index("group")["cohens_kappa"]
)

comparacao_participantes_f2_c1_68 = pd.DataFrame({
    "F2": kappa_f2_pessoa,
    "C1": kappa_c1_pessoa,
})
assert len(comparacao_participantes_f2_c1_68) == 68
assert not comparacao_participantes_f2_c1_68.isna().any().any()

comparacao_participantes_f2_c1_68["Diferença"] = (
    comparacao_participantes_f2_c1_68["C1"]
    - comparacao_participantes_f2_c1_68["F2"]
)
diferencas = comparacao_participantes_f2_c1_68["Diferença"]

print("ACROSS 68 PARTICIPANTS — BEFORE ROUNDING")
print("Kappa aumentou:", int((diferencas > 0).sum()))
print("Kappa diminuiu:", int((diferencas < 0).sum()))
print("Kappa igual:", int((diferencas == 0).sum()))

# Consolidated report.
exportacao_c1_68 = {
    **meta_c1_68,
    "split_unit": "subject",
    "cv_method": "five fixed subject folds A–E",
    "n_subjects": len(set(groups_b0_68)),
    "n_recordings": len(b0_gravacoes_68),
    "n_epochs": len(y_b0_68),
    "cv_mean_sd_across_folds": {
        metrica: {
            "mean": float(tabela_c1_68[metrica].mean()),
            "sd": float(tabela_c1_68[metrica].std(ddof=1)),
        }
        for metrica in metricas_globais
    },
    "sd_ddof": 1,
    "total_fit_seconds": sum(
        r["fit_seconds"] for r in resultados_c1_68.values()
    ),
    "fold_metrics": (
        tabela_c1_68.reset_index().to_dict(orient="records")
    ),
    "training_epoch_counts": (
        resumo_c1_68.reset_index().to_dict(orient="records")
    ),
    "training_class_distribution": (
        distribuicao_c1_68.reset_index().to_dict(orient="records")
    ),
    "training_windows": C1_JANELAS_68,
    "report": relatorio_c1_68,
    "classification_report": classes_c1,
    "comparison_vs_F2_global": (
        comparacao_global_f2_c1_68.rename_axis("metric")
        .reset_index().to_dict(orient="records")
    ),
    "comparison_vs_F2_stage": (
        comparacao_estagios_f2_c1_68.reset_index()
        .to_dict(orient="records")
    ),
    "comparison_vs_F2_subject_kappa": (
        comparacao_participantes_f2_c1_68.reset_index()
        .to_dict(orient="records")
    ),
    "working_decision": (
        "Retain F2_68 as reference. C1 improves N1, macro-F1 "
        "and balanced accuracy, but lowers primary kappa and "
        "increases fold variability. F1 declines for the other stages."
    ),
}

destino_json = EXPERIMENTS_DIR / f"{NOME_C1_68}.json"
temporario = destino_json.with_suffix(".tmp")
temporario.write_text(
    json.dumps(
        exportacao_c1_68,
        default=converter_json_c1_68,
        indent=2,
        ensure_ascii=False,
    ),
    encoding="utf-8",
)
temporario.replace(destino_json)

# Single-model C1 matrix for the team spreadsheet.
fig_c1_68, ax = plt.subplots(figsize=(7, 6))

matriz_c1 = ConfusionMatrixDisplay.from_predictions(
    y_b0_68,
    previsoes_c1_68,
    labels=["W", "N1", "N2", "N3", "REM"],
    normalize="true",
    values_format=".3f",
    cmap="Blues",
    colorbar=False,
    ax=ax,
)
matriz_c1.im_.set_clim(0, 1)
ax.set_title("C1 — 68 subjects — fixed folds A–E")
fig_c1_68.tight_layout()

destino_imagem = (
    EXPERIMENTS_DIR / f"{NOME_C1_68}_confusion.png"
)
fig_c1_68.savefig(destino_imagem, dpi=200, bbox_inches="tight")
plt.close(fig_c1_68)

print("\nReport saved:", destino_json)
print("Image saved:", destino_imagem)
```


### 5.7 — LR1: logistic regression

LR1 keeps the 16 F2 features and the same A–E folds, and replaces Random Forest with logistic regression. Its increased N1 recall came with a much lower overall kappa.


**Original cell 86**

```python
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

necessarias = [
    "X_f2_68",
    "y_b0_68",
    "groups_b0_68",
    "SPLITS_68",
    "config_f2_guardada_68",
]

em_falta = [nome for nome in necessarias if nome not in globals()]

if em_falta:
    raise RuntimeError(
        "Restore these variables first: " + ", ".join(em_falta)
    )

assert X_f2_68.shape == (360621, 16)
assert len(y_b0_68) == len(groups_b0_68) == len(X_f2_68)
assert set(SPLITS_68) == set("ABCDE")

NOME_LR1_68 = "LR1_68_fixed_AE"

CONFIG_LR1_68 = {
    "C": 1.0,
    "penalty": "l2",
    "solver": "lbfgs",
    "class_weight": "balanced",
    "max_iter": 2000,
    "tol": 1e-4,
}

modelo_base_lr1_68 = Pipeline([
    (
        "scale",
        StandardScaler(
            **config_f2_guardada_68["scaler_params"]
        ),
    ),
    ("clf", LogisticRegression(**CONFIG_LR1_68)),
])

print("Experiment:", NOME_LR1_68)
print("Data folder:", X_f2_68.shape)
print("Folds: A, B, C, D, E")
print("Only change: Random Forest → logistic regression")
print("Ready. No model has been fitted yet.")
```


**Original cell 87**

```python
import copy
import hashlib
import json
import platform
import time
import warnings
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
import scipy
import sklearn
from IPython.display import display
from sklearn.base import clone
from sklearn.exceptions import ConvergenceWarning
SleepEDFTrack = type(track_c1_68)

# Full F2: no additional filter and no Wake crop.
assert X_f2_68.shape == (360621, 16)
assert np.isfinite(X_f2_68).all()
assert set(SPLITS_68) == set(resultados_f2_68) == set("ABCDE")
assert (
    len(y_b0_68)
    == len(groups_b0_68)
    == len(record_ids_b0_68)
    == len(X_f2_68)
)
assert len(np.unique(groups_b0_68)) == 68
assert len(np.unique(record_ids_b0_68)) == 133


def converter_json_lr1_68(obj):
    if isinstance(obj, np.ndarray):
        return obj.tolist()
    if isinstance(obj, np.generic):
        return obj.item()
    raise TypeError(type(obj).__name__)


def assinatura_lr1_68(*arrays):
    h = hashlib.sha256()

    for arr in arrays:
        a = np.asarray(arr)

        if a.dtype.kind in "OUS":
            a = a.astype(str)

        a = np.ascontiguousarray(a)
        h.update(str(a.shape).encode())
        h.update(a.dtype.str.encode())
        h.update(memoryview(a).cast("B"))

    return h.hexdigest()


runtime_lr1_68 = {
    "python": platform.python_version(),
    "numpy": np.__version__,
    "scipy": scipy.__version__,
    "scikit-learn": sklearn.__version__,
}

for pacote in ("numpy", "scipy", "scikit-learn"):
    anterior = config_f2_guardada_68.get("runtime", {}).get(pacote)

    if anterior is not None and anterior != runtime_lr1_68[pacote]:
        raise RuntimeError(
            f"Version differs from F2: {pacote}. Envia o aviso."
        )

# Check that test sets match F2,
# with disjoint subjects and complete training recordings.
cobertura_lr1_68 = np.zeros(len(y_b0_68), dtype=np.uint8)

for fold in "ABCDE":
    tr, te = (
        np.asarray(idx, dtype=int)
        for idx in SPLITS_68[fold]
    )

    assert np.all((tr >= 0) & (tr < len(y_b0_68)))
    assert np.all((te >= 0) & (te < len(y_b0_68)))
    assert len(tr) + len(te) == len(y_b0_68)

    assert (
        len(np.unique(np.concatenate([tr, te])))
        == len(y_b0_68)
    )

    assert set(groups_b0_68[tr]).isdisjoint(
        set(groups_b0_68[te])
    )

    ref = resultados_f2_68[fold]

    np.testing.assert_array_equal(te, ref["test_indices"])
    np.testing.assert_array_equal(
        y_b0_68[te], ref["report"]["y_true"]
    )
    np.testing.assert_array_equal(
        groups_b0_68[te], ref["report"]["y_group"]
    )

    cobertura_lr1_68[te] += 1

assert np.all(cobertura_lr1_68 == 1)

# Use the configuration and data identifier to check
# whether a saved result can be reused.
meta_lr1_68 = {
    "experiment": NOME_LR1_68,
    "reference": "F2_68_fixed_AE",
    "course_revision": "5949b723bbd34d172f072446dfcb802ef54b49f7",
    "split_unit": "subject",
    "cv_method": "five fixed subject folds A-E",
    "n_subjects": 68,
    "n_recordings": 133,
    "n_epochs": len(y_b0_68),
    "feature_names": list(nomes_features_f2_68),
    "feature_recipe": (
        "F2 version 1: original 11 plus five relative powers"
    ),
    "additional_filter": None,
    "training_selection": "all retained epochs",
    "data_sha256": assinatura_lr1_68(
        X_f2_68,
        y_b0_68,
        groups_b0_68,
        record_ids_b0_68,
    ),
    "runtime": runtime_lr1_68,
    "feature_cache_runtime": (
        config_f2_guardada_68.get("runtime", {})
    ),
    "classifier_params": (
        modelo_base_lr1_68.named_steps["clf"].get_params()
    ),
    "scaler_params": (
        modelo_base_lr1_68.named_steps["scale"].get_params()
    ),
}

meta_lr1_68 = json.loads(
    json.dumps(
        meta_lr1_68,
        default=converter_json_lr1_68,
    )
)

track_lr1_68 = SleepEDFTrack(cfg={})
resultados_lr1_68 = globals().get("resultados_lr1_68", {})

EXPERIMENTS_DIR = Path(EXPERIMENTS_DIR)
EXPERIMENTS_DIR.mkdir(parents=True, exist_ok=True)


def avaliar_fold_lr1_68(fold):
    tr, te = (
        np.asarray(idx, dtype=int)
        for idx in SPLITS_68[fold]
    )

    destino = (
        EXPERIMENTS_DIR
        / f"{NOME_LR1_68}_fold_{fold}.json"
    )

    guardar = False

    if destino.exists():
        r = json.loads(
            destino.read_text(encoding="utf-8")
        )

    elif fold in resultados_lr1_68:
        r = resultados_lr1_68[fold]
        guardar = True

    else:
        print(
            f"{NOME_LR1_68} — fold {fold}: fitting...",
            flush=True,
        )

        # Fit a fresh scaler and classifier in each fold.
        modelo = clone(modelo_base_lr1_68)

        inicio = time.perf_counter()

        with warnings.catch_warnings():
            warnings.simplefilter(
                "error", ConvergenceWarning
            )
            modelo.fit(
                X_f2_68[tr],
                y_b0_68[tr],
            )

        fit_seconds = time.perf_counter() - inicio
        n_iter = modelo.named_steps["clf"].n_iter_

        if np.any(
            n_iter >= modelo.named_steps["clf"].max_iter
        ):
            raise RuntimeError(
                "O ajuste atingiu max_iter. Envia este aviso."
            )

        inicio = time.perf_counter()
        pred = modelo.predict(X_f2_68[te])
        predict_seconds = time.perf_counter() - inicio

        r = {
            "metadata": copy.deepcopy(meta_lr1_68),
            "fold": fold,
            "created_at_utc": (
                datetime.now(timezone.utc).isoformat()
            ),
            "train_indices": tr,
            "test_indices": te,
            "fit_seconds": fit_seconds,
            "predict_seconds": predict_seconds,
            "n_iter": n_iter,
            "converged": True,
            "report": track_lr1_68._make_report(
                y_b0_68[te],
                pred,
                groups=groups_b0_68[te],
                folds=np.full(len(te), fold),
            ),
        }

        guardar = True

    # Check results loaded from Drive as well.
    assert r["metadata"] == meta_lr1_68, (
        "Different configuration. Do not reuse."
    )
    assert r["fold"] == fold and r["converged"]

    np.testing.assert_array_equal(
        r["train_indices"], tr
    )
    np.testing.assert_array_equal(
        r["test_indices"], te
    )
    np.testing.assert_array_equal(
        r["report"]["y_true"], y_b0_68[te]
    )
    np.testing.assert_array_equal(
        r["report"]["y_group"], groups_b0_68[te]
    )

    assert len(r["report"]["y_pred"]) == len(te)

    # Keep the result in memory before writing to Drive.
    resultados_lr1_68[fold] = r

    if guardar:
        temporario = destino.with_suffix(".tmp")

        temporario.write_text(
            json.dumps(
                r,
                default=converter_json_lr1_68,
                indent=2,
            ),
            encoding="utf-8",
        )

        temporario.replace(destino)
        print("Saved:", destino)

    else:
        print(
            f"Fold {fold}: reused the saved result."
        )

    print(
        f"Fold {fold}: "
        f"kappa = {r['report']['cohens_kappa']:.3f}"
        f" | fit = {r['fit_seconds']/60:.2f} min"
        f" | iterations = {int(np.max(r['n_iter']))}",
        flush=True,
    )


# Run the five folds.
for fold in "ABCDE":
    avaliar_fold_lr1_68(fold)


# Summary of fold metrics.
metricas_lr1_68 = [
    "cohens_kappa",
    "macro_f1",
    "accuracy",
    "balanced_accuracy",
]

tabela_lr1_68 = pd.DataFrame([
    {
        "Fold": fold,
        **{
            m: resultados_lr1_68[fold]["report"][m]
            for m in metricas_lr1_68
        },
        "Treino (min)": (
            resultados_lr1_68[fold]["fit_seconds"] / 60
        ),
    }
    for fold in "ABCDE"
]).set_index("Fold")

print("\nLR1 — FIVE FOLDS COMPLETE")
display(tabela_lr1_68.round(3))

print(
    f"Mean kappa ± SD: "
    f"{tabela_lr1_68['cohens_kappa'].mean():.3f}"
    f" ± {tabela_lr1_68['cohens_kappa'].std(ddof=1):.3f}"
)

print(
    f"Total fit time: "
    f"{tabela_lr1_68['Treino (min)'].sum():.1f} min"
)

comparacao_f2_lr1_68 = pd.DataFrame({
    "F2": {
        f: resultados_f2_68[f]["report"]["cohens_kappa"]
        for f in "ABCDE"
    },
    "LR1": tabela_lr1_68["cohens_kappa"],
})

comparacao_f2_lr1_68["Diferença"] = (
    comparacao_f2_lr1_68["LR1"]
    - comparacao_f2_lr1_68["F2"]
)

print("\nKAPPA COMPARISON ON MATCHED FOLDS")
display(comparacao_f2_lr1_68.round(3))
```


**Original cell 88**

```python
from sklearn.metrics import (
    classification_report,
    ConfusionMatrixDisplay,
)
import matplotlib.pyplot as plt

ESTAGIOS_LR1_68 = ["W", "N1", "N2", "N3", "REM"]


def juntar_oof_lr1_68(resultados):
    assert set(resultados) == set("ABCDE")

    previsoes = np.empty_like(y_b0_68)
    folds = np.empty(len(y_b0_68), dtype="<U1")
    cobertura = np.zeros(len(y_b0_68), dtype=np.uint8)

    for fold in "ABCDE":
        r = resultados[fold]
        idx = np.asarray(r["test_indices"], dtype=int)
        rep = r["report"]
        pred = np.asarray(rep["y_pred"])

        np.testing.assert_array_equal(
            idx, SPLITS_68[fold][1]
        )
        np.testing.assert_array_equal(
            rep["y_true"], y_b0_68[idx]
        )
        np.testing.assert_array_equal(
            rep["y_group"], groups_b0_68[idx]
        )

        assert len(np.unique(idx)) == len(idx)
        assert pred.shape == (len(idx),)
        assert np.isin(pred, ESTAGIOS_LR1_68).all()
        assert np.all(cobertura[idx] == 0)

        previsoes[idx] = pred
        folds[idx] = fold
        cobertura[idx] += 1

    assert np.all(cobertura == 1)

    relatorio = track_lr1_68._make_report(
        y_b0_68,
        previsoes,
        groups=groups_b0_68,
        folds=folds,
    )

    return previsoes, relatorio


assert all(
    r["converged"] and r["metadata"] == meta_lr1_68
    for r in resultados_lr1_68.values()
)

previsoes_lr1_68, relatorio_lr1_68 = juntar_oof_lr1_68(
    resultados_lr1_68
)

previsoes_f2_68, relatorio_f2_68 = juntar_oof_lr1_68(
    resultados_f2_68
)

# Calculate pooled metrics from all test predictions.
metricas_lr1_68 = [
    "cohens_kappa",
    "macro_f1",
    "accuracy",
    "balanced_accuracy",
]

comparacao_global_f2_lr1_68 = pd.DataFrame({
    "F2": {
        m: relatorio_f2_68[m]
        for m in metricas_lr1_68
    },
    "LR1": {
        m: relatorio_lr1_68[m]
        for m in metricas_lr1_68
    },
})

comparacao_global_f2_lr1_68["Diferença"] = (
    comparacao_global_f2_lr1_68["LR1"]
    - comparacao_global_f2_lr1_68["F2"]
)

print("COMPARISON — POOLED METRICS")
display(comparacao_global_f2_lr1_68.round(3))

print("\nLR1 — RESULTS BY STAGE")
print(classification_report(
    y_b0_68,
    previsoes_lr1_68,
    labels=ESTAGIOS_LR1_68,
    digits=3,
    zero_division=0,
))

classes_f2_lr1_68 = classification_report(
    y_b0_68,
    previsoes_f2_68,
    labels=ESTAGIOS_LR1_68,
    output_dict=True,
    zero_division=0,
)

classes_lr1_68 = classification_report(
    y_b0_68,
    previsoes_lr1_68,
    labels=ESTAGIOS_LR1_68,
    output_dict=True,
    zero_division=0,
)

comparacao_estagios_f2_lr1_68 = pd.DataFrame([
    {
        "Estágio": estagio,
        "Recall F2": classes_f2_lr1_68[estagio]["recall"],
        "Recall LR1": classes_lr1_68[estagio]["recall"],
        "F1 F2": classes_f2_lr1_68[estagio]["f1-score"],
        "F1 LR1": classes_lr1_68[estagio]["f1-score"],
        "Diferença F1": (
            classes_lr1_68[estagio]["f1-score"]
            - classes_f2_lr1_68[estagio]["f1-score"]
        ),
    }
    for estagio in ESTAGIOS_LR1_68
]).set_index("Estágio")

print("COMPARISON BY STAGE")
display(comparacao_estagios_f2_lr1_68.round(3))

# Use the same stage order and colour scale in both matrices.
fig_f2_lr1_68, axes = plt.subplots(
    1, 2, figsize=(13, 5.5)
)

for ax, pred, titulo in [
    (axes[0], previsoes_f2_68, "F2 — Random Forest"),
    (axes[1], previsoes_lr1_68, "LR1 — Logistic regression"),
]:
    matriz = ConfusionMatrixDisplay.from_predictions(
        y_b0_68,
        pred,
        labels=ESTAGIOS_LR1_68,
        normalize="true",
        values_format=".3f",
        cmap="Blues",
        colorbar=False,
        ax=ax,
    )

    matriz.im_.set_clim(0, 1)
    ax.set_title(titulo)
    ax.set_xlabel("Predicted stage")
    ax.set_ylabel("True stage")

fig_f2_lr1_68.suptitle(
    "68 subjects — fixed folds A–E — complete test recordings"
)
fig_f2_lr1_68.tight_layout(rect=(0, 0, 1, 0.94))
plt.show()
```


**Original cell 89**

```python
# Compare participants before rounding.
kappa_f2_lr1_68 = (
    pd.DataFrame(relatorio_f2_68["per_group"])
    .set_index("group")["cohens_kappa"]
)

kappa_lr1_68 = (
    pd.DataFrame(relatorio_lr1_68["per_group"])
    .set_index("group")["cohens_kappa"]
)

comparacao_participantes_f2_lr1_68 = pd.DataFrame({
    "F2": kappa_f2_lr1_68,
    "LR1": kappa_lr1_68,
}).sort_index()

assert len(comparacao_participantes_f2_lr1_68) == 68
assert comparacao_participantes_f2_lr1_68.index.is_unique
assert not comparacao_participantes_f2_lr1_68.isna().any().any()

comparacao_participantes_f2_lr1_68["Diferença"] = (
    comparacao_participantes_f2_lr1_68["LR1"]
    - comparacao_participantes_f2_lr1_68["F2"]
)

dif_lr1_68 = comparacao_participantes_f2_lr1_68["Diferença"]

print("ACROSS 68 PARTICIPANTS — BEFORE ROUNDING")
print("Kappa aumentou:", int((dif_lr1_68 > 0).sum()))
print("Kappa diminuiu:", int((dif_lr1_68 < 0).sum()))
print("Kappa igual:", int((dif_lr1_68 == 0).sum()))

# Consolidated report.
exportacao_lr1_68 = {
    **meta_lr1_68,

    "cv_mean_sd_across_folds": {
        m: {
            "mean": float(tabela_lr1_68[m].mean()),
            "sd": float(tabela_lr1_68[m].std(ddof=1)),
        }
        for m in metricas_lr1_68
    },
    "sd_ddof": 1,

    "fold_metrics": (
        tabela_lr1_68.reset_index()
        .to_dict(orient="records")
    ),

    "total_fit_seconds": sum(
        float(r["fit_seconds"])
        for r in resultados_lr1_68.values()
    ),

    "total_predict_seconds": sum(
        float(r["predict_seconds"])
        for r in resultados_lr1_68.values()
    ),

    "iterations_per_fold": {
        f: resultados_lr1_68[f]["n_iter"]
        for f in "ABCDE"
    },

    "all_folds_converged": all(
        r["converged"]
        for r in resultados_lr1_68.values()
    ),

    "report": relatorio_lr1_68,
    "classification_report": classes_lr1_68,

    "comparison_vs_F2_fold_kappa": (
        comparacao_f2_lr1_68.rename_axis("fold")
        .reset_index().to_dict(orient="records")
    ),

    "comparison_vs_F2_global": (
        comparacao_global_f2_lr1_68.rename_axis("metric")
        .reset_index().to_dict(orient="records")
    ),

    "comparison_vs_F2_stage": (
        comparacao_estagios_f2_lr1_68.reset_index()
        .to_dict(orient="records")
    ),

    "comparison_vs_F2_subject_kappa": (
        comparacao_participantes_f2_lr1_68.reset_index()
        .to_dict(orient="records")
    ),

    "working_decision": (
        "Retain F2_68 with Random Forest as the reference. "
        "LR1 is faster and improves balanced accuracy and N1 F1, "
        "but lowers kappa in all five folds, pooled kappa, "
        "macro-F1, accuracy and F1 for the other four stages."
    ),
}

destino_json_lr1_68 = (
    EXPERIMENTS_DIR / f"{NOME_LR1_68}.json"
)
temporario_lr1_68 = destino_json_lr1_68.with_suffix(".tmp")

temporario_lr1_68.write_text(
    json.dumps(
        exportacao_lr1_68,
        default=converter_json_lr1_68,
        indent=2,
        ensure_ascii=False,
    ),
    encoding="utf-8",
)

temporario_lr1_68.replace(destino_json_lr1_68)
print("\nReport saved:", destino_json_lr1_68)

# Single-model figure for the report and spreadsheet.
fig_lr1_68, ax_lr1_68 = plt.subplots(figsize=(7, 6))

matriz_lr1_68 = ConfusionMatrixDisplay.from_predictions(
    y_b0_68,
    previsoes_lr1_68,
    labels=ESTAGIOS_LR1_68,
    normalize="true",
    values_format=".3f",
    cmap="Blues",
    colorbar=False,
    ax=ax_lr1_68,
)

matriz_lr1_68.im_.set_clim(0, 1)
ax_lr1_68.set_title("LR1 — 68 subjects — fixed folds A–E")
ax_lr1_68.set_xlabel("Predicted stage")
ax_lr1_68.set_ylabel("True stage")
fig_lr1_68.tight_layout()

destino_imagem_lr1_68 = (
    EXPERIMENTS_DIR / f"{NOME_LR1_68}_confusion.png"
)

fig_lr1_68.savefig(
    destino_imagem_lr1_68,
    dpi=200,
    bbox_inches="tight",
)
plt.close(fig_lr1_68)

print("Image saved:", destino_imagem_lr1_68)
```
